In [ ]:
!python -m pip install -q pandas numpy scikit-learn scipy matplotlib




[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## SECTION 2: Transforming From 3 Classes to Binary

In [ ]:
%%writefile code.py
from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, Iterable, Literal, Optional, Sequence, Tuple, Union

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    roc_auc_score,
)

READMITTED_CLASSES_3: Tuple[str, str, str] = ("NO", "<30", ">30")


class TargetEngineeringError(ValueError):
    """Raised when the readmitted target contains unexpected/invalid values."""


def _normalize_readmitted_value(v: object) -> Optional[str]:
    """
    Normalize a single readmitted value:
    - None/NaN -> None
    - strings -> stripped, uppercased (except <30 and >30 remain as-is after upper)
    """
    if v is None:
        return None
    try:
        if pd.isna(v):
            return None
    except Exception:
        pass

    s = str(v).strip()
    if s == "":
        return None
    return s.upper()


def validate_readmitted_values(
    values: Union[pd.Series, Iterable[object]],
    allowed: Sequence[str] = READMITTED_CLASSES_3,
    *,
    allow_na: bool = False,
) -> None:
    """
    Validate that all non-missing readmitted values are inside 'allowed'.

    Raises:
        TargetEngineeringError if an unexpected value is found.
    """
    allowed_set = {a.upper() for a in allowed}
    if isinstance(values, pd.Series):
        raw_iter = values.tolist()
    else:
        raw_iter = list(values)

    unexpected = set()
    for v in raw_iter:
        nv = _normalize_readmitted_value(v)
        if nv is None:
            if not allow_na:
                unexpected.add(None)
            continue
        if nv not in allowed_set:
            unexpected.add(nv)

    if unexpected:
        raise TargetEngineeringError(
            f"Unexpected readmitted values found: {sorted([x for x in unexpected if x is not None])}"
            + (" (and missing values)" if None in unexpected else "")
            + f". Allowed values: {list(allowed)}."
        )


def binarize_readmitted(
    readmitted: pd.Series,
    *,
    positive_value: str = "<30",
    negative_values: Sequence[str] = ("NO", ">30"),
    output_name: str = "readmitted_30d",
    unknown_policy: Literal["error", "nan"] = "error",
) -> pd.Series:
    """
    Convert {NO, <30, >30} into binary {1 if <30, 0 otherwise}.

    Args:
        readmitted: pandas Series with original readmitted values.
        positive_value: value mapped to 1.
        negative_values: values mapped to 0.
        output_name: name for the output series.
        unknown_policy:
            - "error": raise if any unexpected/NA values appear
            - "nan": map unexpected/NA to <NA> (nullable integer)

    Returns:
        pd.Series of dtype int8 (or nullable Int8 if unknown_policy="nan").
    """
    pos = positive_value.upper()
    neg = tuple(v.upper() for v in negative_values)

    norm = readmitted.map(_normalize_readmitted_value)

    mapping: Dict[str, int] = {pos: 1, **{v: 0 for v in neg}}

    if unknown_policy == "error":
        validate_readmitted_values(norm, allowed=(pos, *neg), allow_na=False)
        out = norm.map(mapping).astype(np.int8)
        out.name = output_name
        return out

    if unknown_policy == "nan":
        out = norm.map(mapping)
        out = out.astype("Int8")  
        out.name = output_name
        return out

    raise ValueError(f"unknown_policy must be 'error' or 'nan', got: {unknown_policy}")


def keep_multiclass_readmitted(
    readmitted: pd.Series,
    *,
    output_name: str = "readmitted_3class",
    allowed: Sequence[str] = READMITTED_CLASSES_3,
    unknown_policy: Literal["error", "nan"] = "error",
) -> pd.Series:
    """
    Keep the original 3-class label, optionally validating values.

    Returns:
        pd.Series of dtype 'category' with categories in the allowed order,
        or with missing if unknown_policy="nan".
    """
    norm = readmitted.map(_normalize_readmitted_value)

    if unknown_policy == "error":
        validate_readmitted_values(norm, allowed=allowed, allow_na=False)
    elif unknown_policy == "nan":
        pass
    else:
        raise ValueError(f"unknown_policy must be 'error' or 'nan', got: {unknown_policy}")

    cat = pd.Categorical(norm, categories=[a.upper() for a in allowed], ordered=False)
    out = pd.Series(cat, index=readmitted.index, name=output_name)
    return out


def engineer_targets(
    df: pd.DataFrame,
    *,
    source_col: str = "readmitted",
    binary_col: str = "readmitted_30d",
    multiclass_col: str = "readmitted_3class",
    add_multiclass: bool = True,
    drop_source: bool = False,
    unknown_policy: Literal["error", "nan"] = "error",
) -> pd.DataFrame:
    """
    Add engineered target columns to a dataframe:
      - binary: 1 if <30 else 0
      - optional multiclass: categorical {NO, <30, >30}

    This keeps Section 2 self-contained and reproducible.

    Returns:
        A copy of df with new columns added (and optionally source_col dropped).
    """
    if source_col not in df.columns:
        raise KeyError(f"Column '{source_col}' not found in df. Available: {list(df.columns)}")

    out = df.copy()
    out[binary_col] = binarize_readmitted(
        out[source_col],
        output_name=binary_col,
        unknown_policy=unknown_policy,
    )

    if add_multiclass:
        out[multiclass_col] = keep_multiclass_readmitted(
            out[source_col],
            output_name=multiclass_col,
            unknown_policy=unknown_policy,
        )

    if drop_source:
        out.drop(columns=[source_col], inplace=True)

    return out


@dataclass(frozen=True)
class BinaryConfusionTerms:
    tp: int
    fp: int
    fn: int
    tn: int


def confusion_terms_binary(
    y_true: Union[pd.Series, np.ndarray, Sequence[int]],
    y_pred: Union[pd.Series, np.ndarray, Sequence[int]],
    *,
    positive_label: int = 1,
) -> BinaryConfusionTerms:
    """
    Return TP/FP/FN/TN for a binary task once the positive class is defined.

    This directly supports the Section 2 narrative about TP/FP/FN/TN meaning.
    """
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    # cm layout with labels [0,1] is:
    # [[TN, FP],
    #  [FN, TP]]
    tn, fp, fn, tp = cm.ravel()
    return BinaryConfusionTerms(tp=int(tp), fp=int(fp), fn=int(fn), tn=int(tn))


def metrics_binary(
    y_true: Union[pd.Series, np.ndarray, Sequence[int]],
    y_pred: Union[pd.Series, np.ndarray, Sequence[int]],
    y_score: Optional[Union[pd.Series, np.ndarray, Sequence[float]]] = None,
) -> Dict[str, Optional[float]]:
    """
    Convenience metric bundle for binary framing (Section 2 awareness):
      - accuracy
      - f1
      - balanced_accuracy
      - roc_auc (if y_score is provided)
    """
    res: Dict[str, Optional[float]] = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred, pos_label=1)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "roc_auc": None,
    }
    if y_score is not None:
        res["roc_auc"] = float(roc_auc_score(y_true, y_score))
    return res


def metrics_multiclass(
    y_true: Union[pd.Series, np.ndarray, Sequence[str]],
    y_pred: Union[pd.Series, np.ndarray, Sequence[str]],
    y_proba: Optional[np.ndarray] = None,
    *,
    labels: Sequence[str] = READMITTED_CLASSES_3,
) -> Dict[str, Optional[float]]:
    """
    Metric bundle for the advanced extension (3-class framing):
      - macro_f1
      - weighted_f1
      - balanced_accuracy (macro recall)
      - ovr_auc_macro (if y_proba is provided with shape [n_samples, n_classes])

    Notes:
      - For AUC, we use one-vs-rest multi-class AUC (macro).
      - Requires y_proba columns correspond to 'labels' in the same order.
    """
    labels_up = [l.upper() for l in labels]

    yt = pd.Series(y_true).map(_normalize_readmitted_value)
    yp = pd.Series(y_pred).map(_normalize_readmitted_value)

    res: Dict[str, Optional[float]] = {
        "macro_f1": float(f1_score(yt, yp, labels=labels_up, average="macro")),
        "weighted_f1": float(f1_score(yt, yp, labels=labels_up, average="weighted")),
        "balanced_accuracy": float(balanced_accuracy_score(yt, yp)),
        "ovr_auc_macro": None,
    }

    if y_proba is not None:
        y_proba = np.asarray(y_proba)
        if y_proba.ndim != 2 or y_proba.shape[1] != len(labels_up):
            raise ValueError(
                f"y_proba must have shape [n_samples, {len(labels_up)}] matching labels order {labels_up}."
            )
        res["ovr_auc_macro"] = float(
            roc_auc_score(yt, y_proba, multi_class="ovr", average="macro", labels=labels_up)
        )

    return res


Overwriting code.py


## CODE NECESSARY TO LOAD THE CSV FILE

In [12]:
import os, sys, importlib.util
import pandas as pd

def import_module_from_path(module_name: str, file_path: str):
    spec = importlib.util.spec_from_file_location(module_name, file_path)
    if spec is None or spec.loader is None:
        raise ImportError(f"Could not load spec for {module_name} from {file_path}")
    mod = importlib.util.module_from_spec(spec)
    sys.modules.pop(module_name, None)
    sys.modules[module_name] = mod
    spec.loader.exec_module(mod)
    return mod

PROJECT_DIR = os.getcwd()
project_code = import_module_from_path("project_code", os.path.join(PROJECT_DIR, "code.py"))

CSV_PATH = os.path.join(PROJECT_DIR, "database", "diabetic_data.csv")

df_raw = pd.read_csv(
    DATA_PATH,
    dtype="string",         
    keep_default_na=False,  # do NOT auto-convert 'NA', 'N/A', etc. into NaN
    na_values=[""],         # only empty strings become NaN (rare here, but safe)
    low_memory=False,
)

df = project_code.engineer_targets(
    df_raw,
    source_col="readmitted",
    binary_col="readmitted_30d",
    multiclass_col="readmitted_3class",
    add_multiclass=True,
    drop_source=False,
    unknown_policy="error",
)

df.shape, df["readmitted_30d"].value_counts(dropna=False)


((101766, 52),
 readmitted_30d
 0    90409
 1    11357
 Name: count, dtype: int64)

## CODE NECESSARY TO LOAD THE IDS MAPPING

In [13]:
from io import StringIO

def load_ids_mapping(path: str | Path) -> dict[str, pd.DataFrame]:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Mapping file not found at: {path.resolve()}")

    text = path.read_text(encoding="utf-8", errors="ignore").splitlines()

    blocks = []
    cur = []
    for line in text:
        s = line.strip()
        if s in {"", ","}:
            if cur:
                blocks.append("\n".join(cur))
                cur = []
            continue
        cur.append(line)
    if cur:
        blocks.append("\n".join(cur))

    out: dict[str, pd.DataFrame] = {}
    for block in blocks:
        dfm = pd.read_csv(StringIO(block), dtype="string")
        if dfm.shape[1] >= 2 and "description" in dfm.columns:
            key_col = dfm.columns[0]
            dfm = dfm.dropna(subset=[key_col]).copy()
            dfm[key_col] = dfm[key_col].astype("string").str.strip()
            dfm["description"] = dfm["description"].astype("string").str.strip()
            out[key_col] = dfm[[key_col, "description"]]
    return out

id_maps = load_ids_mapping(MAP_PATH)
print("Found mappings:", list(id_maps.keys()))
print({k: v.shape for k, v in id_maps.items()})


Found mappings: ['admission_type_id', 'discharge_disposition_id', 'admission_source_id']
{'admission_type_id': (8, 2), 'discharge_disposition_id': (30, 2), 'admission_source_id': (25, 2)}


## This code executes Section 2 using the dataset with the real data.

In [14]:
df = df_raw.copy()

def apply_mapping(df: pd.DataFrame, id_col: str, new_col: str) -> pd.DataFrame:
    if id_col not in df.columns or id_col not in id_maps:
        return df
    map_df = id_maps[id_col]
    mapper = map_df.set_index(id_col)["description"]
    mapper.index = mapper.index.astype("string")

    keys = df[id_col].astype("string").str.strip()
    df[new_col] = keys.map(mapper)  # unknown stays <NA>
    return df

df = apply_mapping(df, "admission_type_id", "admission_type")
df = apply_mapping(df, "discharge_disposition_id", "discharge_disposition")
df = apply_mapping(df, "admission_source_id", "admission_source")

df = df.drop(columns=[c for c in ["admission_type_id", "discharge_disposition_id", "admission_source_id"] if c in df.columns])

df = project_code.engineer_targets(
    df,
    source_col="readmitted",
    binary_col="readmitted_30d",
    multiclass_col="readmitted_3class",
    add_multiclass=True,
    drop_source=False,
    unknown_policy="error",  
)

df[["readmitted", "readmitted_30d", "readmitted_3class"]].head(), df["readmitted"].value_counts()


(  readmitted  readmitted_30d readmitted_3class
 0         NO               0                NO
 1        >30               0               >30
 2         NO               0                NO
 3         NO               0                NO
 4         NO               0                NO,
 readmitted
 NO     54864
 >30    35545
 <30    11357
 Name: count, dtype: Int64)

## SECTION 3: Important aspects of the dataset: Feature Types, Data Dictionary, and Quality Risks

In [ ]:
%%writefile section3_dataset_understanding.py
from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple, Union

import numpy as np
import pandas as pd


DEFAULT_MISSING_TOKENS: Tuple[str, ...] = (
    "", "?", "NA", "N/A", "NULL", "NAN", "UNKNOWN", "UNKNOWN/INVALID"
)

DEFAULT_ID_COLUMNS: Tuple[str, ...] = (
    "id",
    "url",
    "encounter_id",
    "patient_nbr",
    "patient_id",
    "paper_id",
)

DEFAULT_TEXT_COLUMNS: Tuple[str, ...] = (
    "title",
    "abstract",
    "text",
)

DEFAULT_INTERVAL_COLUMNS: Tuple[str, ...] = (
    "year",
)


def _is_missing_value(x: Any, *, treat_empty_as_missing: bool = True) -> bool:
    if x is None:
        return True
    try:
        if pd.isna(x):
            return True
    except Exception:
        pass
    if treat_empty_as_missing and isinstance(x, str) and x.strip() == "":
        return True
    return False


def _normalize_token(s: str) -> str:
    return s.strip().upper()


def _count_missing_like_tokens(
    series: pd.Series,
    *,
    missing_tokens: Sequence[str] = DEFAULT_MISSING_TOKENS,
) -> Dict[str, int]:
    """
    Count occurrences of missing/unknown encodings in an object/string-like column.
    Counting is case-insensitive and strips whitespace.
    """
    if not (pd.api.types.is_object_dtype(series.dtype) or pd.api.types.is_string_dtype(series.dtype)):
        return {}

    toks = {_normalize_token(t) for t in missing_tokens}
    counts: Dict[str, int] = {t: 0 for t in toks}

    for v in series.astype("object").tolist():
        if v is None:
            continue
        try:
            if pd.isna(v):
                continue
        except Exception:
            pass

        if isinstance(v, str):
            key = _normalize_token(v)
        else:
            key = _normalize_token(str(v))

        if key in counts:
            counts[key] += 1

    return {k: v for k, v in counts.items() if v > 0}


def _unique_ratio(series: pd.Series) -> float:
    n = len(series)
    if n == 0:
        return 0.0
    return float(series.nunique(dropna=True)) / float(n)


def _avg_text_length(series: pd.Series) -> float:
    vals = []
    for v in series.tolist():
        if _is_missing_value(v, treat_empty_as_missing=True):
            continue
        vals.append(len(str(v)))
    if not vals:
        return 0.0
    return float(np.mean(vals))


def infer_kind_and_scale(
    series: pd.Series,
    col: str,
    *,
    text_columns: Sequence[str] = DEFAULT_TEXT_COLUMNS,
    interval_columns: Sequence[str] = DEFAULT_INTERVAL_COLUMNS,
    ordinal_columns: Sequence[str] = (),
) -> Tuple[str, str]:
    """
    Returns (kind, scale) where:
      - kind  in {"numeric","categorical","text"}
      - scale in {"nominal","ordinal","interval","ratio","text"}
    """
    col_low = col.strip().lower()

    if col_low in {c.lower() for c in text_columns}:
        return "text", "text"

    if pd.api.types.is_numeric_dtype(series.dtype) and not pd.api.types.is_bool_dtype(series.dtype):
        if col_low in {c.lower() for c in interval_columns}:
            return "numeric", "interval"
        return "numeric", "ratio"

    if pd.api.types.is_bool_dtype(series.dtype):
        return "categorical", "nominal"

    avg_len = _avg_text_length(series)
    if avg_len >= 30.0:
        return "text", "text"

    if col_low in {c.lower() for c in ordinal_columns}:
        return "categorical", "ordinal"

    return "categorical", "nominal"


def infer_role(
    df: pd.DataFrame,
    col: str,
    kind: str,
    *,
    id_columns: Sequence[str] = DEFAULT_ID_COLUMNS,
    derived_text_column: str = "text",
    id_like_unique_ratio_threshold: float = 0.98,
) -> str:
    """
    Returns a role label to help the report/checklist:
      - "identifier"
      - "raw_text_feature"
      - "derived_feature"
      - "metadata_feature"
      - "numeric_feature"
      - "categorical_feature"
    """
    col_low = col.strip().lower()
    s = df[col]

    if col_low in {c.lower() for c in id_columns}:
        return "identifier"

    if col_low == derived_text_column.lower():
        return "derived_feature"

    if _unique_ratio(s) >= id_like_unique_ratio_threshold and kind != "numeric":
        return "identifier"

    if col_low in {"year", "venue"}:
        return "metadata_feature"

    if kind == "text":
        return "raw_text_feature"
    if kind == "numeric":
        return "numeric_feature"
    return "categorical_feature"


@dataclass(frozen=True)
class DatasetUnderstandingReport:
    data_dictionary: pd.DataFrame
    missing_summary: pd.DataFrame
    missing_tokens_found: pd.DataFrame
    duplicates_summary: pd.DataFrame
    leakage_risks: pd.DataFrame
    quality_risks: pd.DataFrame


def build_data_dictionary(
    df: pd.DataFrame,
    *,
    text_columns: Sequence[str] = DEFAULT_TEXT_COLUMNS,
    interval_columns: Sequence[str] = DEFAULT_INTERVAL_COLUMNS,
    ordinal_columns: Sequence[str] = (),
    id_columns: Sequence[str] = DEFAULT_ID_COLUMNS,
) -> pd.DataFrame:
    rows: List[Dict[str, Any]] = []
    n = len(df)

    for col in df.columns:
        s = df[col]
        kind, scale = infer_kind_and_scale(
            s,
            col,
            text_columns=text_columns,
            interval_columns=interval_columns,
            ordinal_columns=ordinal_columns,
        )
        role = infer_role(df, col, kind, id_columns=id_columns)

        miss = int(s.isna().sum())
        empty = 0
        if pd.api.types.is_object_dtype(s.dtype) or pd.api.types.is_string_dtype(s.dtype):
            empty = int((s.astype("object").map(lambda x: isinstance(x, str) and x.strip() == "")).sum())

        nun = int(s.nunique(dropna=True))
        ur = float(nun) / float(n) if n else 0.0

        examples = []
        for v in s.tolist():
            if _is_missing_value(v, treat_empty_as_missing=True):
                continue
            examples.append(v)
            if len(examples) >= 5:
                break

        rows.append(
            {
                "column": col,
                "pandas_dtype": str(s.dtype),
                "kind": kind,               # numeric / categorical / text
                "scale": scale,             # nominal / ordinal / interval / ratio / text
                "role": role,               # identifier / feature etc
                "n_rows": n,
                "n_unique": nun,
                "unique_ratio": ur,
                "missing_count": miss,
                "missing_rate": (miss / n) if n else 0.0,
                "empty_string_count": empty,
                "examples": examples,
            }
        )

    return pd.DataFrame(rows).sort_values(["role", "kind", "column"]).reset_index(drop=True)


def summarize_missingness(
    df: pd.DataFrame,
    *,
    treat_empty_as_missing: bool = True,
) -> pd.DataFrame:
    rows: List[Dict[str, Any]] = []
    n = len(df)

    for col in df.columns:
        s = df[col]
        na = int(s.isna().sum())
        empty = 0
        if treat_empty_as_missing and (pd.api.types.is_object_dtype(s.dtype) or pd.api.types.is_string_dtype(s.dtype)):
            empty = int((s.astype("object").map(lambda x: isinstance(x, str) and x.strip() == "")).sum())

        miss_total = na + empty
        rows.append(
            {
                "column": col,
                "na_count": na,
                "empty_string_count": empty,
                "missing_total": miss_total,
                "missing_rate": (miss_total / n) if n else 0.0,
            }
        )

    return pd.DataFrame(rows).sort_values("missing_rate", ascending=False).reset_index(drop=True)


def detect_missing_tokens(
    df: pd.DataFrame,
    *,
    missing_tokens: Sequence[str] = DEFAULT_MISSING_TOKENS,
) -> pd.DataFrame:
    """
    Detects tokens like '?', 'N/A', 'Unknown', etc., per column (string/object columns).
    Returns a long-form DataFrame: column, token, count
    """
    rows: List[Dict[str, Any]] = []
    for col in df.columns:
        series = df[col]
        counts = _count_missing_like_tokens(series, missing_tokens=missing_tokens)
        for token, cnt in sorted(counts.items()):
            rows.append({"column": col, "token": token, "count": int(cnt)})
    return pd.DataFrame(rows)


def duplicates_report(
    df: pd.DataFrame,
    *,
    key_columns: Sequence[str] = ("url",),
    content_columns: Sequence[str] = ("title", "abstract"),
) -> pd.DataFrame:
    """
    Returns a compact table of duplicate signals:
      - duplicates by key_columns (e.g., URL duplicates)
      - duplicates by content_columns (title+abstract duplicates)
    """
    rows: List[Dict[str, Any]] = []

    def _dup_stats(subset: Sequence[str], name: str) -> None:
        present = [c for c in subset if c in df.columns]
        if not present:
            rows.append(
                {
                    "scope": name,
                    "subset": list(subset),
                    "available_subset": [],
                    "duplicate_rows": 0,
                    "duplicate_groups": 0,
                }
            )
            return

        dup_mask = df.duplicated(subset=present, keep=False)
        duplicate_rows = int(dup_mask.sum())
        # number of duplicated keys/groups (excluding non-duplicated)
        duplicate_groups = int(df.loc[dup_mask, present].drop_duplicates().shape[0])

        rows.append(
            {
                "scope": name,
                "subset": list(subset),
                "available_subset": present,
                "duplicate_rows": duplicate_rows,
                "duplicate_groups": duplicate_groups,
            }
        )

    _dup_stats(key_columns, "key_duplicates")
    _dup_stats(content_columns, "content_duplicates")

    return pd.DataFrame(rows)


def leakage_risk_report(
    df: pd.DataFrame,
    *,
    id_columns: Sequence[str] = DEFAULT_ID_COLUMNS,
    id_like_unique_ratio_threshold: float = 0.98,
) -> pd.DataFrame:
    """
    Flags columns that are likely to cause leakage / trivial memorization in supervised setups
    (or trivial retrieval shortcuts in IR): identifier-like columns, near-unique columns, etc.
    """
    risks: List[Dict[str, Any]] = []
    n = len(df)

    for col in df.columns:
        s = df[col]
        col_low = col.strip().lower()
        ur = _unique_ratio(s)
        nun = int(s.nunique(dropna=True))

        # Name-based ID columns
        if col_low in {c.lower() for c in id_columns}:
            risks.append(
                {
                    "column": col,
                    "risk_type": "identifier_like",
                    "severity": "high",
                    "reason": "Column name matches a known identifier field (e.g., 'url'/'id').",
                    "unique_ratio": ur,
                    "n_unique": nun,
                    "n_rows": n,
                }
            )
            continue

        # ID-like by uniqueness ratio (very close to 1)
        if ur >= id_like_unique_ratio_threshold and not pd.api.types.is_numeric_dtype(s.dtype):
            risks.append(
                {
                    "column": col,
                    "risk_type": "identifier_like",
                    "severity": "medium",
                    "reason": f"Very high uniqueness ratio (>= {id_like_unique_ratio_threshold}).",
                    "unique_ratio": ur,
                    "n_unique": nun,
                    "n_rows": n,
                }
            )

    return pd.DataFrame(risks).sort_values(["severity", "unique_ratio"], ascending=[True, False]).reset_index(drop=True)


def quality_risk_report(
    df: pd.DataFrame,
    *,
    missing_rate_warn: float = 0.20,
    high_cardinality_warn: float = 0.50,
    treat_empty_as_missing: bool = True,
) -> pd.DataFrame:
    """
    Heuristic quality risks: high missingness, high cardinality, constant columns.
    """
    risks: List[Dict[str, Any]] = []
    n = len(df)

    for col in df.columns:
        s = df[col]

        # Missingness risk
        miss_rate = float(
            (_is_missing_value_count(s, treat_empty_as_missing=treat_empty_as_missing) / n) if n else 0.0
        )
        if miss_rate >= missing_rate_warn:
            risks.append(
                {
                    "column": col,
                    "risk_type": "high_missingness",
                    "severity": "medium",
                    "metric": "missing_rate",
                    "value": miss_rate,
                    "threshold": missing_rate_warn,
                    "note": "Consider explicit handling in preprocessing (drop/impute/tokenize missing).",
                }
            )

        # Cardinality risk (mostly for categorical/text columns)
        ur = float(s.nunique(dropna=True) / n) if n else 0.0
        if ur >= high_cardinality_warn and not pd.api.types.is_numeric_dtype(s.dtype):
            risks.append(
                {
                    "column": col,
                    "risk_type": "high_cardinality",
                    "severity": "low",
                    "metric": "unique_ratio",
                    "value": ur,
                    "threshold": high_cardinality_warn,
                    "note": "May lead to sparse features / overfitting if one-hot encoded.",
                }
            )

        # Constant column risk
        nun = int(s.nunique(dropna=True))
        if nun <= 1:
            risks.append(
                {
                    "column": col,
                    "risk_type": "constant_or_almost_constant",
                    "severity": "low",
                    "metric": "n_unique",
                    "value": nun,
                    "threshold": 1,
                    "note": "Usually safe to drop (no predictive signal).",
                }
            )

    return pd.DataFrame(risks).sort_values(["severity", "risk_type", "column"]).reset_index(drop=True)


def _is_missing_value_count(series: pd.Series, *, treat_empty_as_missing: bool) -> int:
    na = int(series.isna().sum())
    if not treat_empty_as_missing:
        return na
    if pd.api.types.is_object_dtype(series.dtype) or pd.api.types.is_string_dtype(series.dtype):
        empty = int((series.astype("object").map(lambda x: isinstance(x, str) and x.strip() == "")).sum())
        return na + empty
    return na


def profile_dataset(
    df_or_records: Union[pd.DataFrame, Sequence[Dict[str, Any]]],
    *,
    expected_keys: Sequence[str] = ("title", "abstract", "url", "venue", "year"),
    text_columns: Sequence[str] = DEFAULT_TEXT_COLUMNS,
    interval_columns: Sequence[str] = DEFAULT_INTERVAL_COLUMNS,
    ordinal_columns: Sequence[str] = (),
    id_columns: Sequence[str] = DEFAULT_ID_COLUMNS,
    missing_tokens: Sequence[str] = DEFAULT_MISSING_TOKENS,
) -> DatasetUnderstandingReport:
    """
    End-to-end helper for Section 3:
      - Builds data dictionary (types/scales/roles)
      - Summarizes missingness + detects missing tokens
      - Detects duplicates
      - Flags leakage risks
      - Flags quality risks

    Accepts either a DataFrame or the raw list[dict] records.
    """
    if isinstance(df_or_records, pd.DataFrame):
        df = df_or_records.copy()
    else:
        # records -> dataframe
        if not isinstance(df_or_records, (list, tuple)):
            raise TypeError(f"df_or_records must be a DataFrame or list/tuple of dicts, got {type(df_or_records)}")
        rows = []
        for i, rec in enumerate(df_or_records):
            if not isinstance(rec, dict):
                raise TypeError(f"Each record must be a dict. Found {type(rec)} at index {i}.")
            rows.append({k: rec.get(k, None) for k in expected_keys})
        df = pd.DataFrame(rows)

    data_dict = build_data_dictionary(
        df,
        text_columns=text_columns,
        interval_columns=interval_columns,
        ordinal_columns=ordinal_columns,
        id_columns=id_columns,
    )
    missing_summary = summarize_missingness(df, treat_empty_as_missing=True)
    missing_tokens_found = detect_missing_tokens(df, missing_tokens=missing_tokens)
    duplicates_summary = duplicates_report(df)
    leakage_risks = leakage_risk_report(df, id_columns=id_columns)
    quality_risks = quality_risk_report(df)

    return DatasetUnderstandingReport(
        data_dictionary=data_dict,
        missing_summary=missing_summary,
        missing_tokens_found=missing_tokens_found,
        duplicates_summary=duplicates_summary,
        leakage_risks=leakage_risks,
        quality_risks=quality_risks,
    )


Overwriting section3_dataset_understanding.py


### This code executes Section 3 using the dataset with the real data.

In [ ]:
import section3_dataset_understanding as s3

rep = s3.profile_dataset(df)

rep.data_dictionary.head(20), rep.missing_summary.head(20)
rep.leakage_risks, rep.quality_risks.head(30)


(         column        risk_type severity  \
 0  encounter_id  identifier_like     high   
 1   patient_nbr  identifier_like     high   
 
                                               reason  unique_ratio  n_unique  \
 0  Column name matches a known identifier field (...      1.000000    101766   
 1  Column name matches a known identifier field (...      0.702769     71518   
 
    n_rows  
 0  101766  
 1  101766  ,
          column                    risk_type severity        metric     value  \
 0   citoglipton  constant_or_almost_constant      low      n_unique  1.000000   
 1       examide  constant_or_almost_constant      low      n_unique  1.000000   
 2  encounter_id             high_cardinality      low  unique_ratio  1.000000   
 3   patient_nbr             high_cardinality      low  unique_ratio  0.702769   
 
    threshold                                               note  
 0        1.0       Usually safe to drop (no predictive signal).  
 1        1.0       Usually s

In [20]:
!python -m pip install -q pandas numpy scikit-learn matplotlib



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## SECTION 4: Exploratory Data Analysis (EDA)

In [ ]:
%%writefile section4_eda_imbalance.py
from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from sklearn import metrics
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split


# Class imbalance after binarization

@dataclass(frozen=True)
class ClassImbalanceSummary:
    """Compact summary of class imbalance for the binary target.

    The positive class is the early readmission (<30 days), i.e. y=1.
    This supports the discussion of why accuracy can be misleading and
    why precision/recall/F1 (for the minority class) are preferred
    under imbalance (see PDF: confusion matrix + metrics). 
    """
    N: int
    n_pos: int
    n_neg: int
    pos_rate: float
    neg_rate: float

    def as_dict(self) -> Dict[str, float]:
        return {
            "N": self.N,
            "n_pos": self.n_pos,
            "n_neg": self.n_neg,
            "pos_rate": self.pos_rate,
            "neg_rate": self.neg_rate,
        }


def compute_class_imbalance(
    y: Iterable[int],
    *,
    positive_label: int = 1,
    negative_label: int = 0,
) -> ClassImbalanceSummary:
    """Quantify class imbalance for a binary target y={0,1}.

    Parameters
    ----------
    y:
        Iterable with the binary labels (after binarization).
    positive_label:
        Label considered "positive" (early readmission).
    negative_label:
        Label considered "negative".

    Returns
    -------
    ClassImbalanceSummary
        N, n_pos, n_neg and prevalence rates.
    """
    y_series = pd.Series(list(y))
    N = int(len(y_series))
    counts = y_series.value_counts(dropna=False).to_dict()
    n_pos = int(counts.get(positive_label, 0))
    n_neg = int(counts.get(negative_label, 0))
    pos_rate = float(n_pos / N) if N else 0.0
    neg_rate = float(n_neg / N) if N else 0.0
    return ClassImbalanceSummary(N=N, n_pos=n_pos, n_neg=n_neg,
                                 pos_rate=pos_rate, neg_rate=neg_rate)


# Missingness per feature and per record

MISSING_TOKENS: Tuple[str, ...] = (
    "", "?", "NA", "N/A", "NULL", "NAN", "UNKNOWN", "UNKNOWN/INVALID"
)


def is_missing(x: Any, missing_tokens: Sequence[str] = MISSING_TOKENS) -> bool:
    """Unified predicate for 'missing' used in Section 4.

    Treats as missing:
    - None / NaN (according to pandas.isna)
    - empty strings (after strip)
    - tokens like '?', 'NA', 'UNKNOWN', 'UNKNOWN/INVALID', etc.
    """
    if x is None:
        return True
    try:
        if pd.isna(x):
            return True
    except Exception:
        pass

    if isinstance(x, str):
        s = x.strip()
        if s == "":
            return True
        s_up = s.upper()
        tokens_up = {t.upper() for t in missing_tokens}
        return s_up in tokens_up

    return False


def _series_missing_mask(series: pd.Series,
                         missing_tokens: Sequence[str]) -> pd.Series:
    """Return a boolean mask indicating missing values in a Series."""
    return series.map(lambda v: is_missing(v, missing_tokens=missing_tokens))


def missingness_by_feature(
    df: pd.DataFrame,
    *,
    feature_cols: Optional[Sequence[str]] = None,
    missing_tokens: Sequence[str] = MISSING_TOKENS,
) -> pd.DataFrame:
    """Column-level missingness summary.

    Parameters
    ----------
    df:
        Full dataframe including target and features.
    feature_cols:
        Columns to treat as features. If None, all columns except common
        target columns are used.
    missing_tokens:
        Encodings to be treated as missing (in addition to NaN / empty).

    Returns
    -------
    DataFrame
        Index = column name, columns:
        - missing_rate
        - missing_count
    """
    if feature_cols is None:
        excluded = {"y", "readmitted", "readmitted_30d", "readmitted_3class"}
        feature_cols = [c for c in df.columns if c not in excluded]

    n = len(df)
    if n == 0:
        raise ValueError("DataFrame is empty; cannot summarize missingness.")

    def _col_missing_rate(col: pd.Series) -> float:
        return float(_series_missing_mask(col, missing_tokens).mean())

    missing_rate_by_col = (
        df[feature_cols]
        .apply(_col_missing_rate)
        .sort_values(ascending=False)
        .to_frame("missing_rate")
    )
    missing_rate_by_col["missing_count"] = (
        (missing_rate_by_col["missing_rate"] * n).round().astype(int)
    )
    return missing_rate_by_col


@dataclass(frozen=True)
class MissingnessByRowResult:
    """Result of row-level missingness analysis."""
    missing_per_row: pd.Series
    missing_rate_per_row: pd.Series
    summary_counts: Dict[str, float]
    high_missing_fraction: float

    def as_dict(self) -> Dict[str, Any]:
        return {
            "missing_per_row_summary": self.summary_counts,
            "high_missing_fraction": self.high_missing_fraction,
        }

def _df_elementwise_map(df: pd.DataFrame, func):
    return df.map(func) if hasattr(df, "map") else df.applymap(func)


def missingness_by_row(
    df: pd.DataFrame,
    *,
    feature_cols: Optional[Sequence[str]] = None,
    missing_tokens: Sequence[str] = MISSING_TOKENS,
    high_missing_threshold: float = 0.30,
) -> MissingnessByRowResult:
    """Row-level missingness (per encounter/patient record).

    Parameters
    ----------
    df:
        DataFrame with at least the feature columns.
    feature_cols:
        Subset of df columns to treat as features. If None, all except target.
    missing_tokens:
        Tokens to treat as missing.
    high_missing_threshold:
        Threshold on missing *rate* to consider a row 'highly incomplete'.

    Returns
    -------
    MissingnessByRowResult
        - missing_per_row: number of missing values in each row.
        - missing_rate_per_row: fraction of missing per row.
        - summary_counts: describe() of missing_per_row (as dict).
        - high_missing_fraction: proportion of rows with missing_rate >= threshold.
    """
    if feature_cols is None:
        excluded = {"y", "readmitted", "readmitted_30d", "readmitted_3class"}
        feature_cols = [c for c in df.columns if c not in excluded]

    if not feature_cols:
        raise ValueError("No feature columns provided for missingness_by_row.")

    miss_matrix = _df_elementwise_map(
        df[feature_cols],
        lambda v: is_missing(v, missing_tokens=missing_tokens),
    )
    missing_per_row = miss_matrix.sum(axis=1)
    missing_rate_per_row = missing_per_row / float(len(feature_cols))

    summary_counts = missing_per_row.describe(
        percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]
    ).to_dict()
    high_missing_fraction = float((missing_rate_per_row >= high_missing_threshold).mean())

    return MissingnessByRowResult(
        missing_per_row=missing_per_row,
        missing_rate_per_row=missing_rate_per_row,
        summary_counts=summary_counts,
        high_missing_fraction=high_missing_fraction,
    )


# Distributions and high-cardinality categoricals

def topk_table(
    series: pd.Series,
    *,
    k: int = 20,
    missing_tokens: Sequence[str] = MISSING_TOKENS,
    missing_label: str = "__MISSING__",
) -> Tuple[pd.DataFrame, int, float]:
    """Frequency table for high-cardinality categorical variables.

    Parameters
    ----------
    series:
        Categorical column (e.g., diag_1, medical_specialty, payer_code).
    k:
        Top-k categories to display.
    missing_tokens:
        Encodings considered missing.
    missing_label:
        Label used to group all missing values into a single bucket.

    Returns
    -------
    (top_df, n_unique, coverage)
        - top_df: DataFrame with columns ['count', 'rate'] for top-k items.
        - n_unique: total number of distinct categories (including missing_label).
        - coverage: fraction of rows covered by the top-k categories.
    """
    s = series.copy()
    s = s.where(~s.map(lambda v: is_missing(v, missing_tokens=missing_tokens)),
                other=missing_label)

    vc = s.value_counts(dropna=False)
    top = vc.head(k).to_frame("count")
    if len(s) > 0:
        top["rate"] = top["count"] / float(len(s))
        coverage = float(top["count"].sum() / float(len(s)))
    else:
        top["rate"] = 0.0
        coverage = 0.0

    return top, int(vc.shape[0]), coverage


# Clinically meaningful errors (FN vs FP) – dummy baseline

@dataclass(frozen=True)
class DummyBaselineResult:
    """Result of a most_frequent DummyClassifier baseline.

    This baseline is meant to *illustrate* the effect of class imbalance:
    it usually predicts only the majority class, leading to apparently
    reasonable accuracy but zero recall and F1 for the minority class, as
    discussed in the PDF for imbalanced problems. 
    """
    confusion_terms: Dict[str, int]
    metrics: Dict[str, float]

    def as_dict(self) -> Dict[str, Dict[str, float]]:
        return {
            "confusion_terms": self.confusion_terms,
            "metrics": self.metrics,
        }


def run_dummy_most_frequent_baseline(
    df: pd.DataFrame,
    *,
    feature_cols: Sequence[str],
    target_col: str = "y",
    test_size: float = 0.2,
    random_state: int = 42,
) -> DummyBaselineResult:
    """Run a 'most_frequent' DummyClassifier to illustrate imbalance.

    Parameters
    ----------
    df:
        DataFrame with features and binary target.
    feature_cols:
        Columns used as input X (raw; preprocessing comes later).
    target_col:
        Name of the binary target column (e.g., 'readmitted_30d' or 'y').
    test_size:
        Fraction reserved for the illustrative test split.
    random_state:
        Seed for the train/test partition (kept fixed for reproducibility).

    Returns
    -------
    DummyBaselineResult
        Confusion-matrix terms TN/FP/FN/TP and metrics:
        accuracy, precision, recall, F1.
    """
    if target_col not in df.columns:
        raise KeyError(f"Target column '{target_col}' not found in df.")

    X = df[feature_cols]
    y = df[target_col]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=test_size,
        stratify=y,
        random_state=random_state,
    )

    clf = DummyClassifier(strategy="most_frequent")
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    tn, fp, fn, tp = metrics.confusion_matrix(y_test, y_pred).ravel()

    metrics_dict = {
        "accuracy": float(metrics.accuracy_score(y_test, y_pred)),
        "precision": float(metrics.precision_score(y_test, y_pred, zero_division=0)),
        "recall": float(metrics.recall_score(y_test, y_pred, zero_division=0)),
        "f1": float(metrics.f1_score(y_test, y_pred, zero_division=0)),
    }

    confusion_terms = {
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }

    return DummyBaselineResult(confusion_terms=confusion_terms, metrics=metrics_dict)


Overwriting section4_eda_imbalance.py


### The following code executes Section 2 using the dataset with the real data.

In [ ]:
import section4_eda_imbalance as s4

# Class imbalance (binary)
imb = s4.compute_class_imbalance(df["readmitted_30d"].astype(int))
imb.as_dict()


{'N': 101766,
 'n_pos': 11357,
 'n_neg': 90409,
 'pos_rate': 0.11159915885462728,
 'neg_rate': 0.8884008411453728}

In [ ]:
# Missingness by feature (using the unified missing-token logic)
mf = s4.missingness_by_feature(df)
mf.head(25)


,missing_rate,missing_count
weight,0.968585,98569
max_glu_serum,0.947468,96420
A1Cresult,0.832773,84748
medical_specialty,0.490822,49949
payer_code,0.395574,40256
admission_source,0.066633,6781
admission_type,0.051992,5291
discharge_disposition,0.036269,3691
race,0.022336,2273
diag_3,0.013983,1423


In [ ]:
# Missingness by row (how incomplete are records?)
mr = s4.missingness_by_row(df, high_missing_threshold=0.30)
mr.as_dict()


c:\Users\Cris-SX\Desktop\UA\CHAD-MASTER-IA\TAU\tau-final-project\code\lvl1\lvl2\tau-final-code\section4_eda_imbalance.py:217: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  miss_matrix = df[feature_cols].applymap(


{'missing_per_row_summary': {'count': 101766.0,
  'mean': 3.830188864650276,
  'std': 0.9344269760640691,
  'min': 1.0,
  '50%': 4.0,
  '75%': 4.0,
  '90%': 5.0,
  '95%': 5.0,
  '99%': 6.0,
  'max': 9.0},
 'high_missing_fraction': 0.0}

In [ ]:
# Top-K category distributions for known high-cardinality columns
for col in ["medical_specialty", "diag_1", "diag_2", "diag_3", "payer_code"]:
    if col in df.columns:
        top, nunique, coverage = s4.topk_table(df[col], k=15)
        print(f"\n=== {col} | nunique={nunique} | top15 coverage={coverage:.3f} ===")
        display(top)



=== medical_specialty | nunique=73 | top15 coverage=0.955 ===


,count,rate
medical_specialty,,
__MISSING__,49949,0.490822
InternalMedicine,14635,0.14381
Emergency/Trauma,7565,0.074337
Family/GeneralPractice,7440,0.073109
Cardiology,5352,0.052591
Surgery-General,3099,0.030452
Nephrology,1613,0.01585
Orthopedics,1400,0.013757
Orthopedics-Reconstructive,1233,0.012116



=== diag_1 | nunique=717 | top15 coverage=0.443 ===


,count,rate
diag_1,,
428,6862,0.067429
414,6581,0.064668
786,4016,0.039463
410,3614,0.035513
486,3508,0.034471
427,2766,0.02718
491,2275,0.022355
715,2151,0.021137
682,2042,0.020066



=== diag_2 | nunique=749 | top15 coverage=0.511 ===


,count,rate
diag_2,,
276,6752,0.066348
428,6662,0.065464
250,6071,0.059656
427,5036,0.049486
401,3736,0.036712
496,3305,0.032476
599,3288,0.032309
403,2823,0.02774
414,2650,0.02604



=== diag_3 | nunique=790 | top15 coverage=0.527 ===


,count,rate
diag_3,,
250,11555,0.113545
401,8289,0.081452
276,5175,0.050852
428,4577,0.044976
427,3955,0.038864
414,3664,0.036004
496,2605,0.025598
403,2357,0.023161
585,1992,0.019574



=== payer_code | nunique=18 | top15 coverage=0.999 ===


,count,rate
payer_code,,
__MISSING__,40256,0.395574
MC,32439,0.318761
HM,6274,0.061651
SP,5007,0.049201
BC,4655,0.045742
MD,3532,0.034707
CP,2533,0.02489
UN,2448,0.024055
CM,1937,0.019034


In [ ]:
# Dummy "most_frequent" baseline on a simple train/test split (illustrative only)
excluded = {"readmitted", "readmitted_30d", "readmitted_3class"}
feature_cols = [c for c in df.columns if c not in excluded]

dummy_res = s4.run_dummy_most_frequent_baseline(
    df,
    feature_cols=feature_cols,
    target_col="readmitted_30d",
    test_size=0.2,
    random_state=42,
)
dummy_res.as_dict()


{'confusion_terms': {'tn': 18083, 'fp': 0, 'fn': 2271, 'tp': 0},
 'metrics': {'accuracy': 0.8884248796305394,
  'precision': 0.0,
  'recall': 0.0,
  'f1': 0.0}}

## SECTION 5: Preprocessing Plan

In [ ]:
%%writefile section5_preprocessing_pipeline.py

from __future__ import annotations

from dataclasses import dataclass, field
from typing import Any, Dict, List, Mapping, Optional, Sequence, Tuple

import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.base import BaseEstimator, TransformerMixin

import math

DEFAULT_MISSING_TOKENS: Tuple[str, ...] = (
    "",
    "?",
    "NA",
    "N/A",
    "NULL",
    "NAN",
    "UNKNOWN",
    "UNKNOWN/INVALID",
)

DEFAULT_ID_COLS: Tuple[str, ...] = ("encounter_id", "patient_nbr")
DEFAULT_TARGET_COLS: Tuple[str, ...] = ("readmitted", "readmitted_30d", "readmitted_3class", "y")

AGE_ORDER: Tuple[str, ...] = (
    "[0-10)",
    "[10-20)",
    "[20-30)",
    "[30-40)",
    "[40-50)",
    "[50-60)",
    "[60-70)",
    "[70-80)",
    "[80-90)",
    "[90-100)",
)

DEFAULT_FORCE_CATEGORICAL: Tuple[str, ...] = (
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id",
)

# Count-like numeric fields (plus weight if present)
DEFAULT_NUMERIC_HINTS: Tuple[str, ...] = (
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses",
    "weight",
)


def _normalize_token(x: str) -> str:
    return x.strip().upper()


def _missing_token_set(tokens: Sequence[str]) -> set:
    return {_normalize_token(t) for t in tokens}

class MissingTokenCleaner(BaseEstimator, TransformerMixin):
    def __init__(
        self,
        missing_tokens=("?", "UNKNOWN/INVALID", ""),
        *,
        case_insensitive: bool = True,
        strip: bool = True,
        treat_empty_as_missing: bool = True,
    ):
        self.missing_tokens = tuple(missing_tokens)
        self.case_insensitive = case_insensitive
        self.strip = strip
        self.treat_empty_as_missing = treat_empty_as_missing

    def fit(self, X, y=None):
        toks = []
        for t in self.missing_tokens:
            s = str(t)
            s = s.strip() if self.strip else s
            s = s.upper() if self.case_insensitive else s
            toks.append(s)
        self._missing_set_ = set(toks)
        return self

    def transform(self, X):
        if isinstance(X, pd.DataFrame):
            out = X.copy()
            missing_set = getattr(self, "_missing_set_", None)
            if missing_set is None:
                self.fit(X)
                missing_set = self._missing_set_

            for c in out.columns:
                s = out[c]
                if not (pd.api.types.is_object_dtype(s.dtype) or pd.api.types.is_string_dtype(s.dtype)):
                    continue

                def _clean(v):
                    if v is None:
                        return np.nan
                    try:
                        if pd.isna(v):
                            return np.nan
                    except Exception:
                        pass

                    if isinstance(v, str):
                        vv = v.strip() if self.strip else v
                        if self.treat_empty_as_missing and vv == "":
                            return np.nan
                        key = vv.upper() if self.case_insensitive else vv
                        return np.nan if key in missing_set else vv

                    vv = str(v)
                    vv = vv.strip() if self.strip else vv
                    key = vv.upper() if self.case_insensitive else vv
                    return np.nan if key in missing_set else v

                out[c] = s.map(_clean)

            return out

        arr = np.asarray(X, dtype=object)
        missing_set = getattr(self, "_missing_set_", None)
        if missing_set is None:
            self.fit(pd.DataFrame(arr))
            missing_set = self._missing_set_

        def _clean_scalar(v):
            if v is None:
                return np.nan
            try:
                if pd.isna(v):
                    return np.nan
            except Exception:
                pass
            if isinstance(v, str):
                vv = v.strip() if self.strip else v
                if self.treat_empty_as_missing and vv == "":
                    return np.nan
                key = vv.upper() if self.case_insensitive else vv
                return np.nan if key in missing_set else vv
            vv = str(v)
            vv = vv.strip() if self.strip else vv
            key = vv.upper() if self.case_insensitive else vv
            return np.nan if key in missing_set else v

        vfunc = np.vectorize(_clean_scalar, otypes=[object])
        return vfunc(arr)



class NumericCoercer(BaseEstimator, TransformerMixin):
    """Coerce selected columns to numeric (float), invalid parsing -> NaN.
    """

    def __init__(self, numeric_cols: Sequence[str]):
        self.numeric_cols = numeric_cols  

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        if not isinstance(X, pd.DataFrame):
            return X
        df = X.copy()
        for col in self.numeric_cols:
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors="coerce")
        return df

class CategoricalCaster(BaseEstimator, TransformerMixin):
    """
    Cast categorical values to strings (keeping missing as np.nan).
    Prevents mixed types (e.g., int-coded categories + string missing filler).
    """

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_arr = self._to_2d_array(X).astype("object", copy=True)
        out = np.empty_like(X_arr, dtype="object")

        for i in range(X_arr.shape[0]):
            for j in range(X_arr.shape[1]):
                v = X_arr[i, j]
                if v is None:
                    out[i, j] = np.nan
                    continue
                try:
                    if pd.isna(v):
                        out[i, j] = np.nan
                        continue
                except Exception:
                    pass
                out[i, j] = str(v)

        return out

    @staticmethod
    def _to_2d_array(X):
        if isinstance(X, pd.DataFrame):
            return X.to_numpy(dtype="object")
        if isinstance(X, pd.Series):
            return X.to_frame().to_numpy(dtype="object")
        X_arr = np.asarray(X, dtype="object")
        if X_arr.ndim == 1:
            X_arr = X_arr.reshape(-1, 1)
        return X_arr
from sklearn.base import BaseEstimator, TransformerMixin

class RareCategoryGrouper(BaseEstimator, TransformerMixin):
    def __init__(
        self,
        min_count: int = 10,
        *,
        min_frequency: float | None = None,
        other_label: str = "__OTHER__",
        missing_label: str = "__MISSING__",
    ):
        self.min_count = int(min_count)
        self.min_frequency = min_frequency
        self.other_label = other_label
        self.missing_label = missing_label

    def fit(self, X, y=None):
        arr = np.asarray(X, dtype=object)
        if arr.ndim == 1:
            arr = arr.reshape(-1, 1)

        n_rows = arr.shape[0]

        freq_count = 0
        if self.min_frequency is not None:
            if not (0.0 < float(self.min_frequency) <= 1.0):
                raise ValueError(f"min_frequency must be in (0,1], got {self.min_frequency}")
            freq_count = int(math.ceil(float(self.min_frequency) * n_rows))

        effective_min = max(1, self.min_count, freq_count)

        allowed = []
        for j in range(arr.shape[1]):
            col = arr[:, j]

            col2 = np.array(
                [
                    self.missing_label
                    if (v is None or pd.isna(v))
                    else v
                    for v in col
                ],
                dtype=object,
            )

            vc = pd.Series(col2).value_counts(dropna=False)
            keep = set(vc[vc >= effective_min].index.tolist())

            keep.add(self.missing_label)

            allowed.append(keep)

        self.allowed_categories_ = allowed
        self.effective_min_count_ = effective_min
        return self

    def transform(self, X):
        arr = np.asarray(X, dtype=object)
        if arr.ndim == 1:
            arr = arr.reshape(-1, 1)

        if not hasattr(self, "allowed_categories_"):
            self.fit(arr)

        out = arr.copy()
        for j in range(out.shape[1]):
            keep = self.allowed_categories_[j]

            def _map(v):
                if v is None or pd.isna(v):
                    return self.missing_label
                if v == self.missing_label:
                    return self.missing_label
                return v if v in keep else self.other_label

            out[:, j] = np.vectorize(_map, otypes=[object])(out[:, j])

        return out

class ToNumeric(BaseEstimator, TransformerMixin):
    """
    Convert input columns to numeric (float) using pandas.to_numeric(errors='coerce').
    This is mandatory for numeric-like string columns (e.g., 'weight' = "70") so that
    median imputation + scaling work reliably.
    """

    def __init__(self, dtype: str = "float64"):
        self.dtype = dtype

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        if isinstance(X, pd.DataFrame):
            return X.apply(pd.to_numeric, errors="coerce").to_numpy(dtype=self.dtype)

        arr = np.asarray(X, dtype=object)
        if arr.ndim == 1:
            arr = arr.reshape(-1, 1)

        out = np.empty(arr.shape, dtype=self.dtype)
        for j in range(arr.shape[1]):
            out[:, j] = pd.to_numeric(pd.Series(arr[:, j]), errors="coerce").to_numpy(dtype=self.dtype)
        return out

def make_onehot_encoder(*, sparse_output: bool):
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=sparse_output)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=sparse_output)

def _make_one_hot_encoder(*, sparse: bool, handle_unknown: str = "ignore") -> OneHotEncoder:
    try:
        return OneHotEncoder(handle_unknown=handle_unknown, sparse_output=sparse)
    except TypeError:
        return OneHotEncoder(handle_unknown=handle_unknown, sparse=sparse)


def _make_ordinal_encoder(categories: List[List[str]]) -> OrdinalEncoder:
    try:
        return OrdinalEncoder(
            categories=categories,
            handle_unknown="use_encoded_value",
            unknown_value=-1,
        )
    except TypeError:
        return OrdinalEncoder(categories=categories)


@dataclass(frozen=True)
class DiabetesPreprocessConfig:
    missing_tokens: Tuple[str, ...] = DEFAULT_MISSING_TOKENS
    id_cols: Tuple[str, ...] = DEFAULT_ID_COLS
    target_cols: Tuple[str, ...] = DEFAULT_TARGET_COLS

    ordinal_cols: Mapping[str, Sequence[str]] = field(default_factory=lambda: {"age": AGE_ORDER})

    force_categorical_cols: Tuple[str, ...] = DEFAULT_FORCE_CATEGORICAL

    numeric_impute_strategy: str = "median"   
    scale_numeric: bool = True               
    scaler: str = "standard"                 

    cat_impute_strategy: str = "constant"    
    cat_impute_fill_value: str = "__MISSING__"
    rare_min_count: int = 50
    rare_min_frequency: Optional[float] = None
    rare_other_label: str = "__OTHER__"

    # Output format
    onehot_sparse: bool = True


@dataclass(frozen=True)
class FeatureGroups:
    numeric: List[str]
    ordinal: List[str]
    categorical: List[str]
    dropped: List[str]


def infer_feature_groups(df: pd.DataFrame, config: DiabetesPreprocessConfig) -> FeatureGroups:
    cols = list(df.columns)

    id_set = set(config.id_cols)
    target_set = set(config.target_cols)

    dropped = [c for c in cols if c in id_set]
    usable = [c for c in cols if c not in id_set and c not in target_set]

    ordinal = [c for c in usable if c in set(config.ordinal_cols.keys())]
    remaining = [c for c in usable if c not in set(ordinal)]

    force_cat = set(config.force_categorical_cols)


    numeric: List[str] = [c for c in remaining if c in set(DEFAULT_NUMERIC_HINTS)]
    numeric_set = set(numeric)

    for c in remaining:
        if c in numeric_set or c in force_cat:
            continue
        if pd.api.types.is_numeric_dtype(df[c].dtype) and not pd.api.types.is_bool_dtype(df[c].dtype):
            numeric.append(c)
            numeric_set.add(c)

    categorical = [c for c in remaining if c not in numeric_set]
    return FeatureGroups(
        numeric=list(numeric),
        ordinal=list(ordinal),
        categorical=list(categorical),
        dropped=list(dropped),
    )


def build_preprocessor(df: pd.DataFrame, config: Optional[DiabetesPreprocessConfig] = None) -> Pipeline:
    """
    Single-Source-of-Truth preprocessing pipeline.

    Guarantees:
    - Unified missingness: tokens -> NaN
    - Categorical: one-hot (handle_unknown='ignore') + optional rare-category grouping
    - Numeric: median imputation + (optional) scaling/normalization
    - Ordinal: explicit ordered encoding (age)
    - Leakage-safe when used inside CV: all learned steps are fit on training folds only.
    """
    if config is None:
        config = DiabetesPreprocessConfig()

    groups = infer_feature_groups(df, config)

    # Numeric pipeline
    num_steps = [
        ("to_numeric", ToNumeric()),  
        ("imputer", SimpleImputer(strategy=config.numeric_impute_strategy)),
    ]

    if config.scale_numeric:
        scaler_name = config.scaler.lower()
        if scaler_name == "standard":
            num_steps.append(("scaler", StandardScaler()))
        elif scaler_name == "minmax":
            num_steps.append(("scaler", MinMaxScaler()))
        else:
            raise ValueError(f"Unknown scaler: {config.scaler}. Use 'standard' or 'minmax'.")

    num_pipe = Pipeline(steps=num_steps)

    # Ordinal pipeline (age)
    if groups.ordinal:
        categories: List[List[str]] = [list(config.ordinal_cols[c]) for c in groups.ordinal]
        ord_pipe = Pipeline(
            steps=[
                ("imputer", SimpleImputer(strategy="constant", fill_value=config.cat_impute_fill_value)),
                ("ordinal", _make_ordinal_encoder(categories=categories)),
            ]
        )
    else:
        ord_pipe = "drop"

    # Categorical pipeline
    if config.cat_impute_strategy == "constant":
        cat_imputer = SimpleImputer(strategy="constant", fill_value=config.cat_impute_fill_value)
    elif config.cat_impute_strategy == "most_frequent":
        cat_imputer = SimpleImputer(strategy="most_frequent")
    else:
        raise ValueError("cat_impute_strategy must be 'constant' or 'most_frequent'.")

    onehot = make_onehot_encoder(sparse_output=config.onehot_sparse)

    cat_pipe = Pipeline(
        steps=[
            ("cast_to_str", CategoricalCaster()),
            ("imputer", cat_imputer),
            ("rare", RareCategoryGrouper(
                min_count=config.rare_min_count,
                min_frequency=config.rare_min_frequency,
                other_label=config.rare_other_label,
                missing_label=config.cat_impute_fill_value,
            )),
            ("onehot", onehot),
        ]
    )

    transformers = []
    if groups.numeric:
        transformers.append(("num", num_pipe, groups.numeric))
    if groups.ordinal:
        transformers.append(("ord", ord_pipe, groups.ordinal))
    if groups.categorical:
        transformers.append(("cat", cat_pipe, groups.categorical))

    ct = ColumnTransformer(
    transformers=transformers,
    remainder="drop",
    sparse_threshold=0.0 if not config.onehot_sparse else 0.3,
)

    return Pipeline(
        steps=[
            ("clean_missing_tokens", MissingTokenCleaner(missing_tokens=config.missing_tokens)),
            ("coerce_numeric", NumericCoercer(numeric_cols=groups.numeric)),
            ("features", ct),
        ]
    )


Writing section5_preprocessing_pipeline.py


### The following code executes Section 5 using the dataset with the real data.

In [ ]:
import section5_preprocessing_pipeline as s5

cfg = s5.DiabetesPreprocessConfig(
    onehot_sparse=True,     
    rare_min_count=50,      
)

groups = s5.infer_feature_groups(df, cfg)
groups


FeatureGroups(numeric=['weight', 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses'], ordinal=['age'], categorical=['race', 'gender', 'payer_code', 'medical_specialty', 'diag_1', 'diag_2', 'diag_3', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed', 'admission_type', 'discharge_disposition', 'admission_source'], dropped=['encounter_id', 'patient_nbr'])

In [33]:
pre = s5.build_preprocessor(df, cfg)

X = df.drop(columns=[c for c in ["readmitted", "readmitted_30d", "readmitted_3class"] if c in df.columns])
y = df["readmitted_30d"].astype(int)

pre.fit(X, y)
Xt = pre.transform(X)

type(Xt), Xt.shape


C:\Users\Cris-SX\AppData\Roaming\Python\Python313\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: [0]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
C:\Users\Cris-SX\AppData\Roaming\Python\Python313\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: [0]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


(scipy.sparse._csr.csr_matrix, (101766, 786))

## SECTION 6: Baseline System

In [20]:
%%writefile section6_baseline_system.py
from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    make_scorer,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline

import section5_preprocessing_pipeline as s5

@dataclass(frozen=True)
class BaselineCVResult:
    strategy: str
    n_splits: int
    fold_scores: Dict[str, np.ndarray]
    mean_scores: Dict[str, float]
    std_scores: Dict[str, float]

    def as_row(self) -> Dict[str, float]:
        row: Dict[str, float] = {"strategy": self.strategy, "n_splits": float(self.n_splits)}
        for k, v in self.mean_scores.items():
            row[f"{k}_mean"] = float(v)
        for k, v in self.std_scores.items():
            row[f"{k}_std"] = float(v)
        return row


def _validate_binary_target(y: pd.Series) -> pd.Series:
    if y.isna().any():
        raise ValueError("Target contains missing values; baseline evaluation requires a clean binary target.")
    uniq = set(pd.Series(y).astype(int).unique().tolist())
    if not uniq.issubset({0, 1}):
        raise ValueError(f"Target must be binary in {{0,1}}, found values: {sorted(list(uniq))}")
    return pd.Series(y).astype(int)


def _default_feature_frame(df: pd.DataFrame, *, target_col: str, config: s5.DiabetesPreprocessConfig) -> pd.DataFrame:
    drop_cols = [c for c in config.target_cols if c in df.columns]
    if target_col in df.columns and target_col not in drop_cols:
        drop_cols.append(target_col)
    X = df.drop(columns=drop_cols, errors="ignore")
    if X.shape[1] == 0:
        raise ValueError("No feature columns available after dropping target columns.")
    return X


def _scorers_binary() -> Dict[str, object]:
    # zero_division=0 avoids warnings when a baseline predicts no positives
    return {
        "roc_auc": "roc_auc",
        "accuracy": make_scorer(accuracy_score),
        "precision": make_scorer(precision_score, zero_division=0),
        "recall": make_scorer(recall_score, zero_division=0),
        "f1": make_scorer(f1_score, zero_division=0),
    }

def evaluate_dummy_baseline_cv(
    df: pd.DataFrame,
    *,
    target_col: str = "readmitted_30d",
    strategy: str = "most_frequent",
    n_splits: int = 10,
    cv_random_state: int = 42,
    dummy_random_state: int = 42,
    preprocess_config: Optional[s5.DiabetesPreprocessConfig] = None,
    cv_splits: Optional[Sequence[Tuple[np.ndarray, np.ndarray]]] = None,
) -> BaselineCVResult:
    """
    Evaluate a DummyClassifier baseline under Stratified K-Fold CV.
      - If cv_splits is provided, those exact folds are used (identical partitions across experiments).
      - Otherwise, a StratifiedKFold is created as before.
    """
    if preprocess_config is None:
        preprocess_config = s5.DiabetesPreprocessConfig()

    if target_col not in df.columns:
        raise KeyError(f"Target column '{target_col}' not found. Available: {list(df.columns)}")

    y = _validate_binary_target(df[target_col])
    X = _default_feature_frame(df, target_col=target_col, config=preprocess_config)

    pre = s5.build_preprocessor(df, preprocess_config)

    clf = DummyClassifier(strategy=strategy, random_state=dummy_random_state)
    pipe = Pipeline(steps=[("pre", pre), ("dummy", clf)])

    if cv_splits is not None:
        cv_used = list(cv_splits)
        n_splits_effective = len(cv_used)
    else:
        cv_used = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=cv_random_state)
        n_splits_effective = n_splits

    scoring = _scorers_binary()

    out = cross_validate(
        pipe,
        X,
        y,
        cv=cv_used,
        scoring=scoring,
        return_train_score=False,
        n_jobs=None,
    )

    fold_scores: Dict[str, np.ndarray] = {}
    for key in scoring.keys():
        fold_scores[key] = np.asarray(out[f"test_{key}"], dtype=float)

    mean_scores = {k: float(np.mean(v)) for k, v in fold_scores.items()}
    std_scores = {k: float(np.std(v, ddof=1)) if len(v) > 1 else 0.0 for k, v in fold_scores.items()}

    return BaselineCVResult(
        strategy=strategy,
        n_splits=n_splits_effective,
        fold_scores=fold_scores,
        mean_scores=mean_scores,
        std_scores=std_scores,
    )


def run_dummy_baselines(
    df: pd.DataFrame,
    *,
    target_col: str = "readmitted_30d",
    strategies: Sequence[str] = ("most_frequent", "stratified"),
    n_splits: int = 10,
    cv_random_state: int = 42,
    dummy_random_state: int = 42,
    preprocess_config: Optional[s5.DiabetesPreprocessConfig] = None,
    cv_splits: Optional[Sequence[Tuple[np.ndarray, np.ndarray]]] = None,
) -> pd.DataFrame:
    """
    Convenience helper: evaluate multiple DummyClassifier baselines
    and return a compact results table (mean ± std across folds).
    cv_splits can be passed to force identical folds across baselines and later models.
    """
    rows: List[Dict[str, float]] = []
    for strat in strategies:
        res = evaluate_dummy_baseline_cv(
            df,
            target_col=target_col,
            strategy=strat,
            n_splits=n_splits,
            cv_random_state=cv_random_state,
            dummy_random_state=dummy_random_state,
            preprocess_config=preprocess_config,
            cv_splits=cv_splits,
        )
        rows.append(res.as_row())

    table = pd.DataFrame(rows)
    preferred = [
        "strategy",
        "n_splits",
        "roc_auc_mean",
        "roc_auc_std",
        "precision_mean",
        "precision_std",
        "recall_mean",
        "recall_std",
        "f1_mean",
        "f1_std",
        "accuracy_mean",
        "accuracy_std",
    ]
    cols = [c for c in preferred if c in table.columns] + [c for c in table.columns if c not in preferred]
    return table[cols]


Overwriting section6_baseline_system.py


### The following code executes Section 6 using the dataset with the real data.

In [37]:
import section6_baseline_system as s6

baseline_table = s6.run_dummy_baselines(
    df,
    target_col="readmitted_30d",
    strategies=("most_frequent", "stratified"),
    n_splits=10,
    cv_random_state=42,
    dummy_random_state=42,
    preprocess_config=cfg,
)
baseline_table


C:\Users\Cris-SX\AppData\Roaming\Python\Python313\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: [0]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
C:\Users\Cris-SX\AppData\Roaming\Python\Python313\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: [0]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
C:\Users\Cris-SX\AppData\Roaming\Python\Python313\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: [0]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
C:\Users\Cris-SX\AppData\Roaming\Python\Python313\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: [0]. At least one non-missing value is needed for imputation with strategy='median

,strategy,n_splits,roc_auc_mean,roc_auc_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,accuracy_mean,accuracy_std
0,most_frequent,10.0,0.500000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.888401,0.000043
1,stratified,10.0,0.502107,0.004082,0.115465,0.00749,0.111825,0.007248,0.113616,0.007367,0.805279,0.001621


### SECTION 7: Validation Protocol: 10-Fold Cross-Validation

In [21]:
%%writefile section7_validation_protocol.py
from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import hashlib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold


Split = Tuple[np.ndarray, np.ndarray]

def make_stratified_cv_splits(
    y: Iterable[int],
    *,
    n_splits: int = 10,
    shuffle: bool = True,
    random_state: int = 42,
) -> List[Split]:
    """
    Create one fixed set of Stratified K-Fold splits (train_idx, test_idx),
    to be REUSED across all experiments (baseline, models, imbalance methods,
    selection, PCA, tuning outer loop).

    This is the "identical folds across comparisons" contract required by the practice.
    """
    y_arr = np.asarray(list(y), dtype=int)
    if y_arr.ndim != 1:
        raise ValueError("y must be a 1D iterable of binary labels.")
    if len(y_arr) == 0:
        raise ValueError("y is empty.")
    uniq = set(np.unique(y_arr).tolist())
    if not uniq.issubset({0, 1}):
        raise ValueError(f"y must be binary in {{0,1}}. Found: {sorted(list(uniq))}")

    cv = StratifiedKFold(n_splits=n_splits, shuffle=shuffle, random_state=random_state)
    splits: List[Split] = []
    X_dummy = np.zeros((len(y_arr), 1), dtype=float)  
    for tr, te in cv.split(X_dummy, y_arr):
        splits.append((np.asarray(tr, dtype=np.int64), np.asarray(te, dtype=np.int64)))

    validate_cv_splits(splits, n_samples=len(y_arr), n_splits_expected=n_splits)
    return splits


def validate_cv_splits(
    splits: Sequence[Split],
    *,
    n_samples: int,
    n_splits_expected: Optional[int] = None,
) -> None:
    """
    Sanity checks:
      - correct number of folds (if provided)
      - each test fold disjoint
      - union of test indices covers all samples exactly once
      - train/test disjoint per fold
    """
    if n_samples <= 0:
        raise ValueError("n_samples must be > 0.")
    if n_splits_expected is not None and len(splits) != int(n_splits_expected):
        raise ValueError(f"Expected {n_splits_expected} folds, got {len(splits)}.")

    seen_test = np.zeros(n_samples, dtype=int)

    for i, (tr, te) in enumerate(splits):
        tr = np.asarray(tr, dtype=np.int64)
        te = np.asarray(te, dtype=np.int64)

        if tr.ndim != 1 or te.ndim != 1:
            raise ValueError(f"Fold {i}: indices must be 1D arrays.")
        if len(te) == 0:
            raise ValueError(f"Fold {i}: empty test fold.")
        if np.intersect1d(tr, te).size != 0:
            raise ValueError(f"Fold {i}: train/test overlap detected.")

        if tr.min() < 0 or te.min() < 0 or tr.max() >= n_samples or te.max() >= n_samples:
            raise ValueError(f"Fold {i}: indices out of bounds for n_samples={n_samples}.")

        # track test coverage
        for idx in te:
            seen_test[idx] += 1

    if not np.all(seen_test == 1):
        bad = np.where(seen_test != 1)[0]
        raise ValueError(
            "Test folds must cover every sample exactly once. "
            f"Violations at indices: {bad[:20].tolist()} (showing up to 20)."
        )


def cv_splits_fingerprint(splits: Sequence[Split]) -> str:
    """
    Stable hash identifying the exact fold partitioning.
    Useful to assert fold identity across experiments and for reproducibility logs.
    """
    h = hashlib.sha256()
    h.update(str(len(splits)).encode("utf-8"))
    for tr, te in splits:
        tr = np.asarray(tr, dtype=np.int64)
        te = np.asarray(te, dtype=np.int64)
        h.update(tr.tobytes())
        h.update(b"|")
        h.update(te.tobytes())
        h.update(b";")
    return h.hexdigest()


def _final_estimator(pipeline_or_estimator):
    if hasattr(pipeline_or_estimator, "steps") and pipeline_or_estimator.steps:
        return pipeline_or_estimator.steps[-1][1]
    return pipeline_or_estimator


def _get_score_vector(pipeline_or_estimator, X, *, positive_label: int = 1) -> np.ndarray:
    """
    Get continuous scores for ROC-AUC:
      - predict_proba[:, pos] if available
      - else decision_function if available
      - else raise (to avoid meaningless AUC from hard labels)
    """
    if hasattr(pipeline_or_estimator, "predict_proba"):
        proba = pipeline_or_estimator.predict_proba(X)
        proba = np.asarray(proba)
        if proba.ndim != 2 or proba.shape[1] < 2:
            raise ValueError("predict_proba output must be [n_samples, 2+] for binary ROC-AUC.")
        est = _final_estimator(pipeline_or_estimator)
        classes = getattr(est, "classes_", None)
        if classes is None:
            pos_idx = 1
        else:
            classes = np.asarray(classes)
            if positive_label in set(classes.tolist()):
                pos_idx = int(np.where(classes == positive_label)[0][0])
            else:
                pos_idx = 1
        return proba[:, pos_idx].astype(float)

    if hasattr(pipeline_or_estimator, "decision_function"):
        s = pipeline_or_estimator.decision_function(X)
        return np.asarray(s, dtype=float).ravel()

    raise ValueError("Estimator must expose predict_proba or decision_function to compute ROC-AUC.")


@dataclass(frozen=True)
class CVEvaluationResult:
    n_splits: int
    split_fingerprint: str
    fold_indices: List[Split]
    fold_scores: Dict[str, np.ndarray]
    mean_scores: Dict[str, float]
    std_scores: Dict[str, float]
    oof_pred: Optional[np.ndarray] = None
    oof_score: Optional[np.ndarray] = None


def evaluate_binary_pipeline_cv(
    pipeline,
    X,
    y: Iterable[int],
    *,
    cv_splits: Sequence[Split],
    positive_label: int = 1,
    metrics: Sequence[str] = ("roc_auc", "accuracy", "precision", "recall", "f1", "balanced_accuracy"),
    return_oof: bool = True,
) -> CVEvaluationResult:
    """
    Manual CV loop to guarantee:
      - identical folds (cv_splits provided)
      - per-fold metric vectors (needed later for Wilcoxon)
      - optional out-of-fold predictions/scores (useful for error analysis / XAI)
    """
    y_arr = np.asarray(list(y), dtype=int)
    n = len(y_arr)
    validate_cv_splits(cv_splits, n_samples=n)
    fp = cv_splits_fingerprint(cv_splits)

    wanted = set(metrics)
    allowed = {"roc_auc", "accuracy", "precision", "recall", "f1", "balanced_accuracy"}
    if not wanted.issubset(allowed):
        raise ValueError(f"Unsupported metrics requested: {sorted(list(wanted - allowed))}")

    fold_scores: Dict[str, List[float]] = {m: [] for m in metrics}

    oof_pred = np.full(n, fill_value=-1, dtype=int) if return_oof else None
    oof_score = np.full(n, fill_value=np.nan, dtype=float) if return_oof else None

    for tr, te in cv_splits:
        tr = np.asarray(tr, dtype=np.int64)
        te = np.asarray(te, dtype=np.int64)

        model = clone(pipeline)
        model.fit(_safe_index(X, tr), y_arr[tr])

        y_pred = np.asarray(model.predict(_safe_index(X, te)), dtype=int)

        if "roc_auc" in wanted:
            y_score = _get_score_vector(model, _safe_index(X, te), positive_label=positive_label)
            fold_scores["roc_auc"].append(float(roc_auc_score(y_arr[te], y_score)))
        else:
            y_score = None

        if "accuracy" in wanted:
            fold_scores["accuracy"].append(float(accuracy_score(y_arr[te], y_pred)))
        if "precision" in wanted:
            fold_scores["precision"].append(float(precision_score(y_arr[te], y_pred, pos_label=positive_label, zero_division=0)))
        if "recall" in wanted:
            fold_scores["recall"].append(float(recall_score(y_arr[te], y_pred, pos_label=positive_label, zero_division=0)))
        if "f1" in wanted:
            fold_scores["f1"].append(float(f1_score(y_arr[te], y_pred, pos_label=positive_label, zero_division=0)))
        if "balanced_accuracy" in wanted:
            fold_scores["balanced_accuracy"].append(float(balanced_accuracy_score(y_arr[te], y_pred)))

        if return_oof:
            oof_pred[te] = y_pred
            if y_score is not None:
                oof_score[te] = np.asarray(y_score, dtype=float)

    fold_scores_arr: Dict[str, np.ndarray] = {k: np.asarray(v, dtype=float) for k, v in fold_scores.items()}
    mean_scores = {k: float(np.mean(v)) for k, v in fold_scores_arr.items()}
    std_scores = {k: float(np.std(v, ddof=1)) if len(v) > 1 else 0.0 for k, v in fold_scores_arr.items()}

    return CVEvaluationResult(
        n_splits=len(cv_splits),
        split_fingerprint=fp,
        fold_indices=[(np.asarray(tr, dtype=np.int64), np.asarray(te, dtype=np.int64)) for tr, te in cv_splits],
        fold_scores=fold_scores_arr,
        mean_scores=mean_scores,
        std_scores=std_scores,
        oof_pred=oof_pred,
        oof_score=oof_score,
    )


def _safe_index(X, idx: np.ndarray):
    """
    Index X robustly whether it's a pandas DataFrame/Series or a numpy array / sparse matrix.
    """
    if isinstance(X, (pd.DataFrame, pd.Series)):
        return X.iloc[idx]
    return X[idx]


Writing section7_validation_protocol.py


### The following code executes Section 7 using the dataset with the real data.

In [ ]:
import section7_validation_protocol as s7

y = df["readmitted_30d"].astype(int).to_numpy()

splits = s7.make_stratified_cv_splits(
    y,
    n_splits=10,
    shuffle=True,
    random_state=42,
)

fingerprint = s7.cv_splits_fingerprint(splits)
fingerprint


'f5968b2fab085e7f54fd828ac2d60e09701e9a727701c3d6a99a4e5b64150ef5'

In [11]:
%%writefile section8_core_modeling.py
from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier

import section5_preprocessing_pipeline as s5
import section7_validation_protocol as s7

class SparseToCSC(BaseEstimator, TransformerMixin):
    """Convert sparse matrices to CSC (DecisionTree often expects CSC when input is sparse)."""

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        try:
            from scipy import sparse  
        except Exception as e:
            raise RuntimeError(
            ) from e

        if sparse.issparse(X):
            return X.tocsc()
        return X


def _validate_binary_target(y: pd.Series) -> np.ndarray:
    if y.isna().any():
        raise ValueError("Target contains missing values; Section 8 requires a clean binary target.")
    y_arr = np.asarray(pd.Series(y).astype(int).tolist(), dtype=int)
    uniq = set(np.unique(y_arr).tolist())
    if not uniq.issubset({0, 1}):
        raise ValueError(f"Target must be binary in {{0,1}}, found values: {sorted(list(uniq))}")
    return y_arr


def _default_X_y(
    df: pd.DataFrame,
    *,
    target_col: str,
    preprocess_config: s5.DiabetesPreprocessConfig,
) -> Tuple[pd.DataFrame, np.ndarray]:
    if target_col not in df.columns:
        raise KeyError(f"Target column '{target_col}' not found. Available: {list(df.columns)}")

    y_arr = _validate_binary_target(df[target_col])

    drop_cols = [c for c in preprocess_config.target_cols if c in df.columns]
    if target_col in df.columns and target_col not in drop_cols:
        drop_cols.append(target_col)

    X = df.drop(columns=drop_cols, errors="ignore")
    if X.shape[1] == 0:
        raise ValueError("No feature columns available after dropping target columns.")
    return X, y_arr


@dataclass(frozen=True)
class ModelSpec:
    name: str
    estimator: Any
    require_csc: bool = False


def build_section8_model_specs(
    *,
    random_state: int = 42,
) -> List[ModelSpec]:
    """
    Two conceptually different classifiers:
      1) Logistic Regression (linear, probabilistic)
      2) Decision Tree (rule-based, non-linear)

    We pick a LR solver compatible with sparse one-hot features ("saga").
    """
    lr = LogisticRegression(
        solver="saga",
        penalty="l2",
        max_iter=2000,
        random_state=random_state,
        n_jobs=-1,
    )
    dt = DecisionTreeClassifier(
        random_state=random_state,
    )

    return [
        ModelSpec(name="logistic_regression", estimator=lr, require_csc=False),
        ModelSpec(name="decision_tree", estimator=dt, require_csc=True),
    ]


def build_section8_pipelines(
    df: pd.DataFrame,
    *,
    preprocess_config: Optional[s5.DiabetesPreprocessConfig] = None,
    model_random_state: int = 42,
) -> Dict[str, Pipeline]:
    """
    Build model pipelines = preprocessing (Section 5) + optional format adapter + classifier.
    """
    if preprocess_config is None:
        preprocess_config = s5.DiabetesPreprocessConfig()

    pre = s5.build_preprocessor(df, preprocess_config)
    specs = build_section8_model_specs(random_state=model_random_state)

    pipes: Dict[str, Pipeline] = {}
    for spec in specs:
        steps = [("pre", pre)]
        if spec.require_csc:
            steps.append(("to_csc", SparseToCSC()))
        steps.append(("clf", spec.estimator))
        pipes[spec.name] = Pipeline(steps=steps)

    return pipes


@dataclass(frozen=True)
class Section8RunResult:
    """
    Holds everything you need for the report:
      - identical-fold CV metrics (mean/std + per-fold vectors)
      - fold fingerprint for reproducibility
    """
    target_col: str
    n_splits: int
    cv_random_state: int
    split_fingerprint: str
    results_by_model: Dict[str, s7.CVEvaluationResult]
    summary_table: pd.DataFrame


def _summary_table_from_results(
    results_by_model: Dict[str, s7.CVEvaluationResult],
    *,
    model_order: Sequence[str],
    metrics: Sequence[str],
) -> pd.DataFrame:
    rows: List[Dict[str, float]] = []

    for name in model_order:
        r = results_by_model[name]
        row: Dict[str, float] = {"model": name}
        for m in metrics:
            row[f"{m}_mean"] = float(r.mean_scores[m])
            row[f"{m}_std"] = float(r.std_scores[m])
        rows.append(row)

    cols = ["model"] + [f"{m}_{s}" for m in metrics for s in ("mean", "std")]
    return pd.DataFrame(rows)[cols]


def run_section8_core_models(
    df: pd.DataFrame,
    *,
    target_col: str = "readmitted_30d",
    preprocess_config: Optional[s5.DiabetesPreprocessConfig] = None,
    n_splits: int = 10,
    cv_random_state: int = 42,
    model_random_state: int = 42,
    metrics: Sequence[str] = ("roc_auc", "precision", "recall", "f1", "accuracy", "balanced_accuracy"),
    return_oof: bool = True,
    cv_splits: Optional[Sequence[s7.Split]] = None,
) -> Section8RunResult:
    if preprocess_config is None:
        preprocess_config = s5.DiabetesPreprocessConfig()

    X, y = _default_X_y(df, target_col=target_col, preprocess_config=preprocess_config)

    if cv_splits is None:
        cv_splits = s7.make_stratified_cv_splits(
            y,
            n_splits=n_splits,
            shuffle=True,
            random_state=cv_random_state,
        )
    else:
        s7.validate_cv_splits(cv_splits, n_samples=len(y), n_splits_expected=n_splits)

    fp = s7.cv_splits_fingerprint(cv_splits)

    pipes = build_section8_pipelines(
        df,
        preprocess_config=preprocess_config,
        model_random_state=model_random_state,
    )

    # Evaluate with identical folds
    results_by_model: Dict[str, s7.CVEvaluationResult] = {}
    for name, pipe in pipes.items():
        res = s7.evaluate_binary_pipeline_cv(
            pipe,
            X,
            y,
            cv_splits=cv_splits,
            metrics=metrics,
            return_oof=return_oof,
        )
        results_by_model[name] = res

    model_order = list(pipes.keys())
    table = _summary_table_from_results(results_by_model, model_order=model_order, metrics=metrics)

    return Section8RunResult(
        target_col=target_col,
        n_splits=n_splits,
        cv_random_state=cv_random_state,
        split_fingerprint=fp,
        results_by_model=results_by_model,
        summary_table=table,
    )


Overwriting section8_core_modeling.py


## IMPORTANT NOTE

From this point onward, some of the results from each execution are not shown in this file. This is because, in order to include all the results from these sections in the report, each section was run in parallel in separate files. If I had not done it this way, I would not have had enough time to complete all the runs within a single file.

In [ ]:
import section8_core_modeling as s8

out8 = s8.run_section8_core_models(
    df,
    target_col="readmitted_30d",
    preprocess_config=cfg,
    n_splits=10,
    cv_random_state=42,
    model_random_state=42,
    cv_splits=splits,     
    return_oof=True,      
)

out8.split_fingerprint, out8.summary_table


C:\Users\Cris-SX\AppData\Roaming\Python\Python313\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: [0]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


KeyboardInterrupt: 

## SECTION 9: COMPARISON OF MODELS

In [ ]:
%%writefile section9_expanded_modeling.py
from __future__ import annotations

from dataclasses import dataclass, replace
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier

import section5_preprocessing_pipeline as s5
import section7_validation_protocol as s7

Split = s7.Split


class SparseToCSC(BaseEstimator, TransformerMixin):
    """Convert sparse matrices to CSC (tree-based estimators often prefer CSC when input is sparse)."""

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        try:
            from scipy import sparse
        except Exception as e:
            raise RuntimeError(
                "scipy is required for sparse matrix conversion. Install with: pip install scipy"
            ) from e

        if sparse.issparse(X):
            return X.tocsc()
        return X


def _validate_binary_target(y: pd.Series) -> np.ndarray:
    if y.isna().any():
        raise ValueError("Target contains missing values; Section 9 requires a clean binary target.")
    y_arr = np.asarray(pd.Series(y).astype(int).tolist(), dtype=int)
    uniq = set(np.unique(y_arr).tolist())
    if not uniq.issubset({0, 1}):
        raise ValueError(f"Target must be binary in {{0,1}}, found values: {sorted(list(uniq))}")
    return y_arr


def _default_X_y(
    df: pd.DataFrame,
    *,
    target_col: str,
    preprocess_config: s5.DiabetesPreprocessConfig,
) -> Tuple[pd.DataFrame, np.ndarray]:
    if target_col not in df.columns:
        raise KeyError(f"Target column '{target_col}' not found. Available: {list(df.columns)}")

    y = _validate_binary_target(df[target_col])

    drop_cols = [c for c in preprocess_config.target_cols if c in df.columns]
    if target_col not in drop_cols:
        drop_cols.append(target_col)

    X = df.drop(columns=drop_cols, errors="ignore")
    if X.shape[1] == 0:
        raise ValueError("No feature columns available after dropping target columns.")
    return X, y


@dataclass(frozen=True)
class ModelSpec:
    name: str
    pipeline: Pipeline
    track: str  # helps your report: scaled / tree / nb_nonneg / svd_dense


def _make_configs(
    base: s5.DiabetesPreprocessConfig,
) -> Dict[str, s5.DiabetesPreprocessConfig]:
    """
    Section 9 allows deterministic preprocessing tracks:
      - scaled: for scale-sensitive models (SVM, MLP)
      - tree: for tree-based models (DT, RF) (scaling not required)
      - nb_nonneg: ensures non-negative features for MultinomialNB:
          * disables ordinal encoding (age becomes one-hot categorical)
          * disables numeric scaling
    """
    cfg_scaled = replace(base, scale_numeric=True, scaler="standard")
    cfg_tree = replace(base, scale_numeric=False)


    cfg_nb = replace(cfg_tree, ordinal_cols={})

    return {"scaled": cfg_scaled, "tree": cfg_tree, "nb_nonneg": cfg_nb}


def build_section9_model_specs(
    df: pd.DataFrame,
    *,
    preprocess_config_base: Optional[s5.DiabetesPreprocessConfig] = None,
    model_random_state: int = 42,
    svd_components: int = 100,
    fast_mode: bool = False,
) -> List[ModelSpec]:
    """
    Expanded suite (>=5, includes ensembles):
      - decision_tree (anchor)
      - naive_bayes (MultinomialNB; non-negative track)
      - svm_linear (LinearSVC; margin-based)
      - mlp_svd (MLP on TruncatedSVD components; avoids densifying huge one-hot)
      - random_forest (bagging)
      - gradient_boosting_svd (boosting on SVD components; avoids sparse incompatibility)

    fast_mode reduces compute for unit tests / quick sanity runs.
    """
    if preprocess_config_base is None:
        preprocess_config_base = s5.DiabetesPreprocessConfig()

    cfgs = _make_configs(preprocess_config_base)

    pre_tree = s5.build_preprocessor(df, cfgs["tree"])
    pre_scaled = s5.build_preprocessor(df, cfgs["scaled"])
    pre_nb = s5.build_preprocessor(df, cfgs["nb_nonneg"])

    rf_estimators = 80 if fast_mode else 300
    gb_estimators = 100 if fast_mode else 200
    mlp_max_iter = 60 if fast_mode else 200
    mlp_hidden = (32,) if fast_mode else (64,)

    dt_pipe = Pipeline(
        steps=[
            ("pre", pre_tree),
            ("to_csc", SparseToCSC()),
            ("clf", DecisionTreeClassifier(random_state=model_random_state)),
        ]
    )

    nb_pipe = Pipeline(
        steps=[
            ("pre", pre_nb),
            ("clf", MultinomialNB(alpha=1.0)),
        ]
    )

    svm_pipe = Pipeline(
        steps=[
            ("pre", pre_scaled),
            ("clf", LinearSVC(C=1.0, random_state=model_random_state)),
        ]
    )

    mlp_pipe = Pipeline(
        steps=[
            ("pre", pre_scaled),
            ("svd", TruncatedSVD(n_components=int(svd_components), random_state=model_random_state)),
            ("clf", MLPClassifier(
                hidden_layer_sizes=mlp_hidden,
                activation="relu",
                solver="adam",
                alpha=1e-4,
                max_iter=int(mlp_max_iter),
                random_state=model_random_state,
                early_stopping=True,
                n_iter_no_change=10,
            )),
        ]
    )

    rf_pipe = Pipeline(
        steps=[
            ("pre", pre_tree),
            ("to_csc", SparseToCSC()),
            ("clf", RandomForestClassifier(
                n_estimators=int(rf_estimators),
                random_state=model_random_state,
                n_jobs=-1,
            )),
        ]
    )

    gb_pipe = Pipeline(
        steps=[
            ("pre", pre_tree),
            ("svd", TruncatedSVD(n_components=int(svd_components), random_state=model_random_state)),
            ("clf", GradientBoostingClassifier(
                n_estimators=int(gb_estimators),
                learning_rate=0.1,
                random_state=model_random_state,
            )),
        ]
    )

    return [
        ModelSpec("decision_tree", dt_pipe, track="tree"),
        ModelSpec("naive_bayes", nb_pipe, track="nb_nonneg"),
        ModelSpec("svm_linear", svm_pipe, track="scaled"),
        ModelSpec("mlp_svd", mlp_pipe, track="svd_dense"),
        ModelSpec("random_forest", rf_pipe, track="tree"),
        ModelSpec("gradient_boosting_svd", gb_pipe, track="svd_dense"),
    ]


@dataclass(frozen=True)
class Section9RunResult:
    target_col: str
    n_splits: int
    cv_random_state: int
    split_fingerprint: str
    results_by_model: Dict[str, s7.CVEvaluationResult]
    summary_table: pd.DataFrame


def _summary_table(
    specs: Sequence[ModelSpec],
    results_by_model: Dict[str, s7.CVEvaluationResult],
    *,
    metrics: Sequence[str],
) -> pd.DataFrame:
    rows: List[Dict[str, Any]] = []
    for spec in specs:
        r = results_by_model[spec.name]
        row: Dict[str, Any] = {"model": spec.name, "track": spec.track}
        for m in metrics:
            row[f"{m}_mean"] = float(r.mean_scores[m])
            row[f"{m}_std"] = float(r.std_scores[m])
        rows.append(row)

    cols = ["model", "track"] + [f"{m}_{s}" for m in metrics for s in ("mean", "std")]
    return pd.DataFrame(rows)[cols]


def run_section9_expanded_suite(
    df: pd.DataFrame,
    *,
    target_col: str = "readmitted_30d",
    preprocess_config_base: Optional[s5.DiabetesPreprocessConfig] = None,
    n_splits: int = 10,
    cv_random_state: int = 42,
    model_random_state: int = 42,
    metrics: Sequence[str] = ("roc_auc", "precision", "recall", "f1", "accuracy", "balanced_accuracy"),
    return_oof: bool = False,
    cv_splits: Optional[Sequence[Split]] = None,
    svd_components: int = 100,
    fast_mode: bool = False,
) -> Section9RunResult:
    """
    End-to-end Section 9:
      - builds 6-model suite (>=5, includes ensembles)
      - uses identical StratifiedKFold splits (must be reused across comparisons)
      - reports mean ± std across folds (table-ready)
    """
    if preprocess_config_base is None:
        preprocess_config_base = s5.DiabetesPreprocessConfig()

    X, y = _default_X_y(df, target_col=target_col, preprocess_config=preprocess_config_base)

    if cv_splits is None:
        cv_splits = s7.make_stratified_cv_splits(y, n_splits=n_splits, random_state=cv_random_state)
    else:
        s7.validate_cv_splits(cv_splits, n_samples=len(y), n_splits_expected=n_splits)

    fp = s7.cv_splits_fingerprint(cv_splits)

    specs = build_section9_model_specs(
        df,
        preprocess_config_base=preprocess_config_base,
        model_random_state=model_random_state,
        svd_components=svd_components,
        fast_mode=fast_mode,
    )

    results_by_model: Dict[str, s7.CVEvaluationResult] = {}
    for spec in specs:
        res = s7.evaluate_binary_pipeline_cv(
            spec.pipeline,
            X,
            y,
            cv_splits=cv_splits,
            metrics=metrics,
            return_oof=return_oof,
        )
        results_by_model[spec.name] = res

    table = _summary_table(specs, results_by_model, metrics=metrics)

    return Section9RunResult(
        target_col=target_col,
        n_splits=n_splits,
        cv_random_state=cv_random_state,
        split_fingerprint=fp,
        results_by_model=results_by_model,
        summary_table=table,
    )


Writing section9_expanded_modeling.py


### The following code executes Section 9 using the dataset with the real data.

In [ ]:

CSV_PATH = "database/diabetic_data.csv"

import pandas as pd

df_raw = pd.read_csv(CSV_PATH)


df = project_code.engineer_targets(
    df_raw,
    source_col="readmitted",
    binary_col="readmitted_30d",
    multiclass_col="readmitted_3class",
    add_multiclass=True,
    drop_source=False,
    unknown_policy="error",
)

import section5_preprocessing_pipeline as s5

cfg = s5.DiabetesPreprocessConfig(
    onehot_sparse=True,   
    rare_min_count=50,    _
)

import section7_validation_protocol as s7

y = df["readmitted_30d"].astype(int).to_numpy()
splits = s7.make_stratified_cv_splits(
    y,
    n_splits=10,
    shuffle=True,
    random_state=42,
)

print("CV fingerprint:", s7.cv_splits_fingerprint(splits))

import section9_expanded_modeling as s9

out9 = s9.run_section9_expanded_suite(
    df,
    target_col="readmitted_30d",
    preprocess_config_base=cfg,
    n_splits=10,
    cv_random_state=42,
    model_random_state=42,
    cv_splits=splits,       
    svd_components=100,     
    fast_mode=False,
    return_oof=False,       
)

out9.summary_table


CV fingerprint: f5968b2fab085e7f54fd828ac2d60e09701e9a727701c3d6a99a4e5b64150ef5


C:\Users\Cris-SX\AppData\Roaming\Python\Python313\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: [0]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


## SECTION 10: Class Imbalance Handling Experiments

In [ ]:
%%writefile section10_imbalance_handling.py

from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple, Literal

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.utils.class_weight import compute_sample_weight

import section5_preprocessing_pipeline as s5
import section7_validation_protocol as s7

Split = s7.Split

ImbalanceMethod = Literal[
    "none",
    "class_weight_balanced",
    "random_over",
    "random_under",
    "threshold_f1",
]


@dataclass(frozen=True)
class ImbalanceExperimentSpec:
    model: str
    method: ImbalanceMethod
    sampling_ratio: float = 1.0  
    random_state: int = 42


@dataclass(frozen=True)
class Section10EvaluationResult:
    spec: ImbalanceExperimentSpec
    n_splits: int
    split_fingerprint: str
    fold_scores: Dict[str, np.ndarray]
    mean_scores: Dict[str, float]
    std_scores: Dict[str, float]
    fold_thresholds: Optional[np.ndarray] = None  

    def as_row(self) -> Dict[str, Any]:
        row: Dict[str, Any] = {
            "model": self.spec.model,
            "method": self.spec.method,
            "sampling_ratio": float(self.spec.sampling_ratio),
            "n_splits": int(self.n_splits),
            "split_fingerprint": self.split_fingerprint,
        }
        if self.fold_thresholds is not None:
            row["threshold_mean"] = float(np.mean(self.fold_thresholds))
            row["threshold_std"] = float(np.std(self.fold_thresholds, ddof=1)) if len(self.fold_thresholds) > 1 else 0.0

        for m, v in self.mean_scores.items():
            row[f"{m}_mean"] = float(v)
        for m, v in self.std_scores.items():
            row[f"{m}_std"] = float(v)
        return row

def _ensure_1d_binary(y: Iterable[int]) -> np.ndarray:
    y_arr = np.asarray(list(y), dtype=int).ravel()
    if y_arr.size == 0:
        raise ValueError("y is empty.")
    uniq = set(np.unique(y_arr).tolist())
    if not uniq.issubset({0, 1}):
        raise ValueError(f"Binary target must be in {{0,1}}. Found: {sorted(list(uniq))}")
    return y_arr


def _safe_index(X, idx: np.ndarray):
    if isinstance(X, (pd.DataFrame, pd.Series)):
        return X.iloc[idx]
    return X[idx]


def _get_score_vector(estimator, X, *, positive_label: int = 1) -> np.ndarray:
    """
    Continuous scores for ROC-AUC + threshold moving:
    - predict_proba[:, pos] if available
    - else decision_function
    """
    if hasattr(estimator, "predict_proba"):
        proba = estimator.predict_proba(X)
        proba = np.asarray(proba)
        if proba.ndim != 2 or proba.shape[1] < 2:
            raise ValueError("predict_proba must return [n_samples, 2+] for binary tasks.")
        classes = getattr(estimator, "classes_", None)
        if classes is None:
            pos_idx = 1
        else:
            classes = np.asarray(classes)
            if positive_label in set(classes.tolist()):
                pos_idx = int(np.where(classes == positive_label)[0][0])
            else:
                pos_idx = 1
        return proba[:, pos_idx].astype(float)

    if hasattr(estimator, "decision_function"):
        s = estimator.decision_function(X)
        return np.asarray(s, dtype=float).ravel()

    raise ValueError("Estimator must provide predict_proba or decision_function for ROC-AUC/thresholding.")


def _to_csc_if_sparse(X):
    try:
        from scipy import sparse
    except Exception as e:
        raise RuntimeError("scipy is required for sparse matrix operations. Install with: pip install scipy") from e
    if sparse.issparse(X):
        return X.tocsc()
    return X


def random_oversample(X, y: np.ndarray, *, ratio: float = 1.0, random_state: int = 42):
    """
    Randomly oversample the minority class with replacement to reach:
        n_minority_new ~= ratio * n_majority
    """
    y = _ensure_1d_binary(y)
    if not (ratio > 0.0):
        raise ValueError("ratio must be > 0.")

    rng = np.random.RandomState(int(random_state))
    idx0 = np.where(y == 0)[0]
    idx1 = np.where(y == 1)[0]
    if len(idx0) == 0 or len(idx1) == 0:
        return X, y

    if len(idx1) <= len(idx0):
        idx_min, idx_maj = idx1, idx0
        min_label = 1
    else:
        idx_min, idx_maj = idx0, idx1
        min_label = 0

    n_maj = len(idx_maj)
    n_min = len(idx_min)
    target_min = int(np.round(float(ratio) * n_maj))
    if target_min <= n_min:
        new_idx = np.concatenate([idx_maj, idx_min])
    else:
        extra = rng.choice(idx_min, size=(target_min - n_min), replace=True)
        new_idx = np.concatenate([idx_maj, idx_min, extra])

    rng.shuffle(new_idx)
    X_new = _safe_index(X, new_idx)
    y_new = y[new_idx]
    return X_new, y_new


def random_undersample(X, y: np.ndarray, *, ratio: float = 1.0, random_state: int = 42):
    """
    Randomly undersample the majority class without replacement to reach:
        n_majority_new ~= n_minority / ratio
    where ratio = minority/majority after sampling (ratio=1 => balanced).
    """
    y = _ensure_1d_binary(y)
    if not (ratio > 0.0):
        raise ValueError("ratio must be > 0.")

    rng = np.random.RandomState(int(random_state))
    idx0 = np.where(y == 0)[0]
    idx1 = np.where(y == 1)[0]
    if len(idx0) == 0 or len(idx1) == 0:
        return X, y

    if len(idx1) <= len(idx0):
        idx_min, idx_maj = idx1, idx0
    else:
        idx_min, idx_maj = idx0, idx1

    n_min = len(idx_min)
    n_maj = len(idx_maj)

    target_maj = int(np.round(float(n_min) / float(ratio)))
    target_maj = max(1, min(target_maj, n_maj))

    keep_maj = rng.choice(idx_maj, size=target_maj, replace=False)
    new_idx = np.concatenate([idx_min, keep_maj])
    rng.shuffle(new_idx)

    X_new = _safe_index(X, new_idx)
    y_new = y[new_idx]
    return X_new, y_new


def best_threshold_max_f1(
    y_true: np.ndarray,
    scores: np.ndarray,
    *,
    positive_label: int = 1,
    n_grid: int = 200,
) -> float:
    """
    Choose threshold that maximizes F1 on the provided (y_true, scores).
    Uses quantile grid for stability and speed.
    """
    y_true = _ensure_1d_binary(y_true)
    scores = np.asarray(scores, dtype=float).ravel()
    if scores.size != y_true.size:
        raise ValueError("scores and y_true must have the same length.")

    if scores.size <= 500:
        candidates = np.unique(scores)
    else:
        qs = np.linspace(0.0, 1.0, int(n_grid))
        candidates = np.unique(np.quantile(scores, qs))

    best_t = float(candidates[0])
    best_f1 = -1.0
    best_recall = -1.0

    for t in candidates:
        y_pred = (scores >= t).astype(int)
        f1 = float(f1_score(y_true, y_pred, pos_label=positive_label, zero_division=0))
        if f1 > best_f1:
            best_f1 = f1
            best_t = float(t)
            best_recall = float(recall_score(y_true, y_pred, pos_label=positive_label, zero_division=0))
        elif f1 == best_f1:
            rec = float(recall_score(y_true, y_pred, pos_label=positive_label, zero_division=0))
            if rec > best_recall or (rec == best_recall and float(t) < best_t):
                best_t = float(t)
                best_recall = rec

    return best_t


def _compute_metrics(y_true: np.ndarray, y_pred: np.ndarray, y_score: np.ndarray) -> Dict[str, float]:
    return {
        "roc_auc": float(roc_auc_score(y_true, y_score)),
        "precision": float(precision_score(y_true, y_pred, pos_label=1, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, pos_label=1, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, pos_label=1, zero_division=0)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
    }


def _supports_sample_weight(estimator) -> bool:
    import inspect
    try:
        sig = inspect.signature(estimator.fit)
        return "sample_weight" in sig.parameters
    except Exception:
        return False

def build_section10_estimator(model: str, *, random_state: int = 42, fast_mode: bool = False):
    """
    Minimal, rubric-safe set for Section 10:
    - logistic_regression (probabilistic; classic for class weights + threshold moving)
    - decision_tree (PDF notes sensitivity to imbalance)
    - random_forest (ensemble/bagging; can respond well to imbalance methods)
    """
    model = str(model).strip().lower()
    if model == "logistic_regression":
        from sklearn.linear_model import LogisticRegression
        return LogisticRegression(
            solver="saga",
            penalty="l2",
            max_iter=2000,
            random_state=int(random_state),
            n_jobs=-1,
        )

    if model == "decision_tree":
        from sklearn.tree import DecisionTreeClassifier
        return DecisionTreeClassifier(random_state=int(random_state))

    if model == "random_forest":
        from sklearn.ensemble import RandomForestClassifier
        n_estimators = 80 if fast_mode else 300
        return RandomForestClassifier(
            n_estimators=int(n_estimators),
            random_state=int(random_state),
            n_jobs=-1,
        )

    if model == "linear_svm":
        from sklearn.svm import LinearSVC
        return LinearSVC(C=1.0, random_state=int(random_state))

    raise ValueError(f"Unknown model '{model}'. Use: logistic_regression, decision_tree, random_forest, linear_svm.")


def evaluate_section10_experiment_cv(
    df: pd.DataFrame,
    *,
    target_col: str,
    preprocess_config: Optional[s5.DiabetesPreprocessConfig],
    cv_splits: Sequence[Split],
    spec: ImbalanceExperimentSpec,
    positive_label: int = 1,
    fast_mode: bool = False,
) -> Section10EvaluationResult:
    """
    Manual fold loop to guarantee:
    - resampling inside TRAIN only
    - threshold selection inside TRAIN only
    - identical folds reused across all experiments
    """
    if preprocess_config is None:
        preprocess_config = s5.DiabetesPreprocessConfig()

    if target_col not in df.columns:
        raise KeyError(f"Target column '{target_col}' not found in df.")

    y = _ensure_1d_binary(df[target_col].astype(int).to_numpy())

    drop_cols = [c for c in preprocess_config.target_cols if c in df.columns]
    if target_col not in drop_cols:
        drop_cols.append(target_col)
    X = df.drop(columns=drop_cols, errors="ignore")
    if X.shape[1] == 0:
        raise ValueError("No feature columns available after dropping target columns.")

    s7.validate_cv_splits(cv_splits, n_samples=len(y))
    fp = s7.cv_splits_fingerprint(cv_splits)

    pre_template = s5.build_preprocessor(df, preprocess_config)

    fold_metrics: Dict[str, List[float]] = {k: [] for k in ["roc_auc", "precision", "recall", "f1", "accuracy", "balanced_accuracy"]}
    fold_thresholds: List[float] = []

    for tr, te in cv_splits:
        tr = np.asarray(tr, dtype=np.int64)
        te = np.asarray(te, dtype=np.int64)

        X_tr_raw = _safe_index(X, tr)
        y_tr = y[tr]
        X_te_raw = _safe_index(X, te)
        y_te = y[te]

        pre = clone(pre_template)
        pre.fit(X_tr_raw, y_tr)
        X_tr = pre.transform(X_tr_raw)
        X_te = pre.transform(X_te_raw)

        sample_weight = None
        if spec.method == "class_weight_balanced":
            sample_weight = compute_sample_weight(class_weight="balanced", y=y_tr).astype(float)

        if spec.method == "random_over":
            X_tr, y_tr = random_oversample(X_tr, y_tr, ratio=spec.sampling_ratio, random_state=spec.random_state)
            sample_weight = None  

        if spec.method == "random_under":
            X_tr, y_tr = random_undersample(X_tr, y_tr, ratio=spec.sampling_ratio, random_state=spec.random_state)
            sample_weight = None

        est = build_section10_estimator(spec.model, random_state=spec.random_state, fast_mode=fast_mode)

        if spec.model in {"decision_tree", "random_forest"}:
            X_tr_fit = _to_csc_if_sparse(X_tr)
            X_te_fit = _to_csc_if_sparse(X_te)
        else:
            X_tr_fit, X_te_fit = X_tr, X_te

        if sample_weight is not None:
            if not _supports_sample_weight(est):
                raise RuntimeError(f"Estimator '{spec.model}' does not support sample_weight; cannot run class_weight_balanced.")
            est.fit(X_tr_fit, y_tr, sample_weight=sample_weight)
        else:
            est.fit(X_tr_fit, y_tr)

        tr_scores = _get_score_vector(est, X_tr_fit, positive_label=positive_label)
        te_scores = _get_score_vector(est, X_te_fit, positive_label=positive_label)

        if spec.method == "threshold_f1":
            thr = best_threshold_max_f1(y_tr, tr_scores, positive_label=positive_label)
            fold_thresholds.append(float(thr))
            y_pred = (te_scores >= thr).astype(int)
        else:
            y_pred = np.asarray(est.predict(X_te_fit), dtype=int)

        m = _compute_metrics(y_te, y_pred, te_scores)
        for k in fold_metrics.keys():
            fold_metrics[k].append(float(m[k]))

    fold_scores = {k: np.asarray(v, dtype=float) for k, v in fold_metrics.items()}
    mean_scores = {k: float(np.mean(v)) for k, v in fold_scores.items()}
    std_scores = {k: float(np.std(v, ddof=1)) if len(v) > 1 else 0.0 for k, v in fold_scores.items()}

    thr_arr = np.asarray(fold_thresholds, dtype=float) if spec.method == "threshold_f1" else None

    return Section10EvaluationResult(
        spec=spec,
        n_splits=len(cv_splits),
        split_fingerprint=fp,
        fold_scores=fold_scores,
        mean_scores=mean_scores,
        std_scores=std_scores,
        fold_thresholds=thr_arr,
    )


def run_section10_imbalance_experiments(
    df: pd.DataFrame,
    *,
    target_col: str = "readmitted_30d",
    preprocess_config: Optional[s5.DiabetesPreprocessConfig] = None,
    cv_splits: Sequence[Split],
    models: Sequence[str] = ("logistic_regression", "decision_tree", "random_forest"),
    methods: Sequence[ImbalanceMethod] = ("none", "class_weight_balanced", "random_over", "random_under", "threshold_f1"),
    sampling_ratio: float = 1.0,
    random_state: int = 42,
    fast_mode: bool = False,
) -> Tuple[pd.DataFrame, Dict[Tuple[str, str], Section10EvaluationResult]]:
    """
    Returns:
      - results_table: mean ± std per (model, method) for ROC-AUC + precision/recall/F1 (+ accuracy, balanced_accuracy)
      - results_map: raw per-fold vectors (needed later for Wilcoxon / deeper analysis)
    """
    if preprocess_config is None:
        preprocess_config = s5.DiabetesPreprocessConfig()

    rows: List[Dict[str, Any]] = []
    results_map: Dict[Tuple[str, str], Section10EvaluationResult] = {}

    for model in models:
        for method in methods:
            spec = ImbalanceExperimentSpec(
                model=str(model),
                method=method,
                sampling_ratio=float(sampling_ratio),
                random_state=int(random_state),
            )
            res = evaluate_section10_experiment_cv(
                df,
                target_col=target_col,
                preprocess_config=preprocess_config,
                cv_splits=cv_splits,
                spec=spec,
                fast_mode=fast_mode,
            )
            rows.append(res.as_row())
            results_map[(spec.model, spec.method)] = res

    table = pd.DataFrame(rows)

    metric_order = ["roc_auc", "precision", "recall", "f1", "accuracy", "balanced_accuracy"]
    cols = (
        ["model", "method", "sampling_ratio", "n_splits", "split_fingerprint"]
        + (["threshold_mean", "threshold_std"] if "threshold_mean" in table.columns else [])
        + [f"{m}_mean" for m in metric_order]
        + [f"{m}_std" for m in metric_order]
    )
    cols = [c for c in cols if c in table.columns] + [c for c in table.columns if c not in cols]
    table = table[cols].sort_values(["model", "method"]).reset_index(drop=True)

    return table, results_map


Writing section10_imbalance_handling.py


## SECTION 11: Feature Engineering to Create or Transform Predictors

In [ ]:
%%writefile section11_feature_engineering.py
from __future__ import annotations

from dataclasses import dataclass
from functools import lru_cache
from typing import List, Sequence, Tuple

import numpy as np
import pandas as pd



DEFAULT_MISSING_TOKENS: Tuple[str, ...] = (
    "", "?", "NA", "N/A", "NULL", "NAN", "UNKNOWN", "UNKNOWN/INVALID"
)

def _norm_token(x: object) -> str:
    return str(x).strip().upper()

_DEFAULT_MISSING_TOKENS_UP = frozenset(_norm_token(t) for t in DEFAULT_MISSING_TOKENS)

@lru_cache(maxsize=64)
def _missing_token_set(tokens: Tuple[str, ...]) -> frozenset[str]:
    return frozenset(_norm_token(t) for t in tokens)

def is_missing_token(x: object, *, missing_tokens: Sequence[str] = DEFAULT_MISSING_TOKENS) -> bool:
    if x is None:
        return True
    try:
        if pd.isna(x):
            return True
    except Exception:
        pass

    s = str(x)
    if s.strip() == "":
        return True

    up = _norm_token(s)
    if missing_tokens is DEFAULT_MISSING_TOKENS:
        return up in _DEFAULT_MISSING_TOKENS_UP

    toks = _missing_token_set(tuple(missing_tokens))
    return up in toks


DEFAULT_MED_COLS: Tuple[str, ...] = (
    "metformin","repaglinide","nateglinide","chlorpropamide","glimepiride","acetohexamide",
    "glipizide","glyburide","tolbutamide","pioglitazone","rosiglitazone","acarbose","miglitol",
    "troglitazone","tolazamide","examide","citoglipton","insulin",
    "glyburide-metformin","glipizide-metformin","glimepiride-pioglitazone",
    "metformin-rosiglitazone","metformin-pioglitazone",
)

DIAG_COLS_DEFAULT: Tuple[str, ...] = ("diag_1", "diag_2", "diag_3")


def _to_num(s: pd.Series) -> pd.Series:
    return pd.to_numeric(s, errors="coerce")

def _df_elementwise_map(df: pd.DataFrame, func):
    return df.map(func) if hasattr(df, "map") else df.applymap(func)



def age_midpoint(age_str: object) -> float:
    """
    Convert age bucket like "[50-60)" -> 55.0.
    Missing/unknown -> np.nan.
    """
    if is_missing_token(age_str):
        return np.nan
    s = str(age_str).strip()
    if not s:
        return np.nan

    # Expect formats like "[a-b)" or "(a-b]" etc.
    if "-" not in s:
        return np.nan

    try:
        inside = s.strip("[]()")
        a, b = inside.split("-", 1)
        a = float(a)
        b = float(b)
        return (a + b) / 2.0
    except Exception:
        return np.nan


def bin_numeric_fixed(
    x: pd.Series,
    bins: Sequence[float],
    labels: Sequence[str],
    *,
    include_lowest: bool = True,
    right: bool = True,
) -> pd.Series:
    """
    Deterministic, fixed-threshold binning via pd.cut.
    Keeps reproducibility and avoids data-driven thresholds (leakage).
    """
    x_num = pd.to_numeric(x, errors="coerce")
    return pd.cut(
        x_num,
        bins=bins,
        labels=labels,
        include_lowest=include_lowest,
        right=right,
        ordered=True,
    )


def _diag_token(v: object) -> str | None:
    """Normalize and validate a diagnosis token."""
    if v is None:
        return None
    try:
        if pd.isna(v):
            return None
    except Exception:
        pass

    s = str(v).strip()
    if s == "":
        return None

    up = s.upper()
    if up in _DEFAULT_MISSING_TOKENS_UP:
        return None

    return up

def _extract_leading_numeric(tok: str) -> float | None:
    """
    Extract a leading numeric value from an ICD-9 token (handles '250.13', '401', '530.81').
    Returns None if it can't parse a leading numeric prefix.
    """
    buf = []
    dot_used = False
    for ch in tok:
        if ch.isdigit():
            buf.append(ch)
        elif ch == "." and not dot_used:
            buf.append(ch)
            dot_used = True
        else:
            break
    if not buf:
        return None
    try:
        return float("".join(buf))
    except Exception:
        return None

def icd9_group(code: object) -> str:
    """
    Robust ICD-9 grouping:
      - V* -> supplementary_v
      - E* -> external_e
      - numeric ranges -> major ICD-9 chapter
      - 250.* -> diabetes (special case)
    """
    tok = _diag_token(code)
    if tok is None:
        return "__MISSING__"

    if tok.startswith("V"):
        return "supplementary_v"
    if tok.startswith("E"):
        return "external_e"

    num = _extract_leading_numeric(tok)
    if num is None:
        return "other"

    # Special case diabetes: 250.xx
    if 250.0 <= num < 251.0:
        return "diabetes"

    # Major ICD-9 chapters (coarse)
    if 1.0 <= num <= 139.0:
        return "infectious"
    if 140.0 <= num <= 239.0:
        return "neoplasms"
    if 240.0 <= num <= 279.0:
        return "endocrine_metabolic"
    if 280.0 <= num <= 289.0:
        return "blood"
    if 290.0 <= num <= 319.0:
        return "mental"
    if 320.0 <= num <= 389.0:
        return "nervous"
    if 390.0 <= num <= 459.0:
        return "circulatory"
    if 460.0 <= num <= 519.0:
        return "respiratory"
    if 520.0 <= num <= 579.0:
        return "digestive"
    if 580.0 <= num <= 629.0:
        return "genitourinary"
    if 630.0 <= num <= 679.0:
        return "pregnancy"
    if 680.0 <= num <= 709.0:
        return "skin"
    if 710.0 <= num <= 739.0:
        return "musculoskeletal"
    if 740.0 <= num <= 759.0:
        return "congenital"
    if 760.0 <= num <= 779.0:
        return "perinatal"
    if 780.0 <= num <= 799.0:
        return "symptoms"
    if 800.0 <= num <= 999.0:
        return "injury"

    return "other"

def add_diag_group_features(df: pd.DataFrame, diag_cols: Sequence[str] = DIAG_COLS_DEFAULT) -> pd.DataFrame:
    out = df.copy()
    present = [c for c in diag_cols if c in out.columns]
    for c in present:
        out[f"{c}_group"] = out[c].map(icd9_group)

    group_cols = [f"{c}_group" for c in present]
    if group_cols:
        out["n_diabetes_diags"] = (out[group_cols] == "diabetes").sum(axis=1).astype(int)

        if "diag_1_group" in out.columns:
            out["primary_diag_is_diabetes"] = (out["diag_1_group"] == "diabetes").astype(int)
        else:
            out["primary_diag_is_diabetes"] = pd.Series(0, index=out.index, dtype=int)

        tmp = out[group_cols].replace("__MISSING__", np.nan)
        out["n_unique_diag_groups"] = tmp.nunique(axis=1, dropna=True).fillna(0).astype(int)

        if "diag_1_group" in out.columns:
            out["n_diags_same_as_primary_group"] = (
                out[group_cols].eq(out["diag_1_group"], axis=0).sum(axis=1).astype(int)
            )

    return out


# Labs & tests (A1C / glucose)
_A1C_MAP = {"NONE": 0, "NORM": 1, ">7": 2, ">8": 3}
_GLU_MAP = {"NONE": 0, "NORM": 1, ">200": 2, ">300": 3}

def add_lab_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    if "A1Cresult" in out.columns:
        a1c_raw = out["A1Cresult"].astype("object")
        a1c_norm = a1c_raw.map(lambda v: "MISSING" if is_missing_token(v) else str(v).strip().upper())
        out["a1c_measured"] = (a1c_norm.notna() & (a1c_norm != "MISSING") & (a1c_norm != "NONE")).astype(int)
        out["a1c_high"] = a1c_norm.isin({">7", ">8"}).astype(int)
        out["a1c_level"] = a1c_norm.map(_A1C_MAP).fillna(0).astype(int)

    if "max_glu_serum" in out.columns:
        glu_raw = out["max_glu_serum"].astype("object")
        glu_norm = glu_raw.map(lambda v: "MISSING" if is_missing_token(v) else str(v).strip().upper())
        out["glu_measured"] = (glu_norm.notna() & (glu_norm != "MISSING") & (glu_norm != "NONE")).astype(int)
        out["glu_high"] = glu_norm.isin({">200", ">300"}).astype(int)
        out["glu_level"] = glu_norm.map(_GLU_MAP).fillna(0).astype(int)

    if "num_lab_procedures" in out.columns:
        out["num_lab_procedures"] = _to_num(out["num_lab_procedures"])
        out["num_lab_procedures_bin"] = bin_numeric_fixed(
            out["num_lab_procedures"],
            bins=[-np.inf, 20, 40, 60, np.inf],
            labels=["low", "medium", "high", "very_high"],
        )

    return out

def add_utilization_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    num_cols = [
        "number_outpatient", "number_emergency", "number_inpatient",
        "time_in_hospital", "num_medications", "number_diagnoses"
    ]
    for c in num_cols:
        if c in out.columns:
            out[c] = _to_num(out[c])

    if all(c in out.columns for c in ["number_outpatient", "number_emergency", "number_inpatient"]):
        out["total_visits"] = (
            out["number_outpatient"].fillna(0)
            + out["number_emergency"].fillna(0)
            + out["number_inpatient"].fillna(0)
        ).astype(float)

        out["any_emergency"] = (out["number_emergency"].fillna(0) > 0).astype(int)
        out["any_inpatient"] = (out["number_inpatient"].fillna(0) > 0).astype(int)
        out["any_outpatient"] = (out["number_outpatient"].fillna(0) > 0).astype(int)
        out["any_prior_visit"] = (out["total_visits"].fillna(0) > 0).astype(int)

        out["total_visits_bin"] = bin_numeric_fixed(
            out["total_visits"],
            bins=[-np.inf, 0, 2, 5, np.inf],
            labels=["none", "low", "medium", "high"],
        )

    if "time_in_hospital" in out.columns:
        out["stay_bin"] = bin_numeric_fixed(
            out["time_in_hospital"],
            bins=[-np.inf, 3, 7, 14, np.inf],
            labels=["short", "medium", "long", "very_long"],
        )

    if "num_medications" in out.columns:
        out["num_medications_bin"] = bin_numeric_fixed(
            out["num_medications"],
            bins=[-np.inf, 5, 10, 20, np.inf],
            labels=["low", "medium", "high", "very_high"],
        )
        out["polypharmacy"] = (out["num_medications"].fillna(0) >= 10).astype(int)

    if "number_diagnoses" in out.columns:
        out["number_diagnoses_bin"] = bin_numeric_fixed(
            out["number_diagnoses"],
            bins=[-np.inf, 3, 6, np.inf],
            labels=["low", "medium", "high"],
        )

    return out


# Medication burden from per-drug columns
def add_medication_burden_features(
    df: pd.DataFrame,
    *,
    med_cols: Sequence[str] = DEFAULT_MED_COLS,
) -> pd.DataFrame:
    out = df.copy()
    present = [c for c in med_cols if c in out.columns]
    if not present:
        return out

    meds = out[present].astype("object")
    meds_norm = _df_elementwise_map(
        meds, lambda v: "MISSING" if is_missing_token(v) else str(v).strip().upper()
    )

    active_mask = meds_norm.isin({"STEADY", "UP", "DOWN"})
    changed_mask = meds_norm.isin({"UP", "DOWN"})

    out["n_active_diabetes_meds"] = active_mask.sum(axis=1).astype(int)
    out["n_changed_diabetes_meds"] = changed_mask.sum(axis=1).astype(int)
    out["any_active_diabetes_med"] = (out["n_active_diabetes_meds"] > 0).astype(int)
    out["any_changed_diabetes_med"] = (out["n_changed_diabetes_meds"] > 0).astype(int)

    # insulin-specific flags
    if "insulin" in out.columns:
        insulin = out["insulin"].astype("object").map(
            lambda v: "MISSING" if is_missing_token(v) else str(v).strip().upper()
        )
        out["insulin_active"] = insulin.isin({"STEADY", "UP", "DOWN"}).astype(int)
        out["insulin_changed"] = insulin.isin({"UP", "DOWN"}).astype(int)

    return out


# Simple interactions (numeric only; low risk)
def add_interaction_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    # age midpoint
    if "age" in out.columns:
        out["age_mid"] = out["age"].map(age_midpoint)
        out["elderly"] = (out["age_mid"].fillna(-1) >= 65).astype(int)

    # ensure numeric coercion for interactions
    for c in ["time_in_hospital", "num_medications", "number_diagnoses", "total_visits"]:
        if c in out.columns:
            out[c] = _to_num(out[c])

    if all(c in out.columns for c in ["time_in_hospital", "num_medications"]):
        out["stay_x_meds"] = (out["time_in_hospital"].fillna(0) * out["num_medications"].fillna(0)).astype(float)

    if all(c in out.columns for c in ["time_in_hospital", "number_diagnoses"]):
        out["stay_x_diagnoses"] = (out["time_in_hospital"].fillna(0) * out["number_diagnoses"].fillna(0)).astype(float)

    if all(c in out.columns for c in ["total_visits", "time_in_hospital"]):
        out["visits_x_stay"] = (out["total_visits"].fillna(0) * out["time_in_hospital"].fillna(0)).astype(float)

    if all(c in out.columns for c in ["age_mid", "num_medications"]):
        out["age_x_meds"] = (out["age_mid"].fillna(0) * out["num_medications"].fillna(0)).astype(float)

    if all(c in out.columns for c in ["age_mid", "total_visits"]):
        out["age_x_visits"] = (out["age_mid"].fillna(0) * out["total_visits"].fillna(0)).astype(float)

    return out

@dataclass(frozen=True)
class FeatureEngineeringOutput:
    df: pd.DataFrame
    added_columns: List[str]


def apply_feature_engineering(
    df: pd.DataFrame,
    *,
    diag_cols: Sequence[str] = DIAG_COLS_DEFAULT,
    med_cols: Sequence[str] = DEFAULT_MED_COLS,
) -> FeatureEngineeringOutput:
    """
    Apply Section 11 transformations and return:
      - df with engineered features added
      - list of added column names (useful for report + debugging)

    IMPORTANT: run this BEFORE building the Section 5 preprocessing pipeline,
    so the new columns are included in modeling.
    """
    if not isinstance(df, pd.DataFrame):
        raise TypeError("apply_feature_engineering expects a pandas DataFrame.")

    before = set(df.columns)

    out = df.copy()
    out = add_diag_group_features(out, diag_cols=diag_cols)
    out = add_lab_features(out)
    out = add_utilization_features(out)
    out = add_medication_burden_features(out, med_cols=med_cols)
    out = add_interaction_features(out)

    after = set(out.columns)
    added = sorted(list(after - before))
    return FeatureEngineeringOutput(df=out, added_columns=added)


Writing section11_feature_engineering.py


### The following code executes Section 11 using the dataset with the real data.

In [ ]:
import os, sys, importlib.util
import pandas as pd

def import_module_from_path(module_name: str, file_path: str):
    spec = importlib.util.spec_from_file_location(module_name, file_path)
    if spec is None or spec.loader is None:
        raise ImportError(f"Could not load spec for {module_name} from {file_path}")

    mod = importlib.util.module_from_spec(spec)

    # ✅ CRITICAL: register BEFORE exec_module (fixes dataclasses on Py3.13)
    sys.modules.pop(module_name, None)
    sys.modules[module_name] = mod

    spec.loader.exec_module(mod)
    return mod

PROJECT_DIR = os.getcwd()
project_code = import_module_from_path("project_code", os.path.join(PROJECT_DIR, "code.py"))

df_raw = pd.read_csv("database/diabetic_data.csv")

df = project_code.engineer_targets(
    df_raw,
    source_col="readmitted",
    binary_col="readmitted_30d",
    multiclass_col="readmitted_3class",
    add_multiclass=True,
    drop_source=False,
    unknown_policy="error",
)

import section11_feature_engineering as s11

fe_out = s11.apply_feature_engineering(df)
df_fe = fe_out.df

print("Added columns (Section 11):")
print(fe_out.added_columns)

cols_to_preview = [
    "readmitted_30d",
    "age", "age_mid", "elderly",
    "diag_1", "diag_1_group",
    "n_diabetes_diags", "primary_diag_is_diabetes",
    "total_visits", "total_visits_bin",
    "stay_bin", "num_medications_bin",
    "n_active_diabetes_meds", "n_changed_diabetes_meds",
    "a1c_measured", "a1c_high",
    "glu_measured", "glu_high",
    "stay_x_meds", "stay_x_diagnoses"
]
cols_to_preview = [c for c in cols_to_preview if c in df_fe.columns]
df_fe[cols_to_preview].head(10)


Added columns (Section 11):
['a1c_high', 'a1c_measured', 'age_mid', 'any_emergency', 'any_inpatient', 'any_outpatient', 'diag_1_group', 'diag_2_group', 'diag_3_group', 'elderly', 'glu_high', 'glu_measured', 'insulin_active', 'insulin_changed', 'n_active_diabetes_meds', 'n_changed_diabetes_meds', 'n_diabetes_diags', 'num_lab_procedures_bin', 'num_medications_bin', 'number_diagnoses_bin', 'polypharmacy', 'primary_diag_is_diabetes', 'stay_bin', 'stay_x_diagnoses', 'stay_x_meds', 'total_visits', 'total_visits_bin', 'visits_x_stay']


,readmitted_30d,age,age_mid,elderly,diag_1,diag_1_group,n_diabetes_diags,primary_diag_is_diabetes,total_visits,total_visits_bin,stay_bin,num_medications_bin,n_active_diabetes_meds,n_changed_diabetes_meds,a1c_measured,a1c_high,glu_measured,glu_high,stay_x_meds,stay_x_diagnoses
0,0,[0-10),5.0,0,250.83,diabetes,1,1,0.0,none,short,low,0,0,0,0,0,0,1.0,1.0
1,0,[10-20),15.0,0,276,endocrine_metabolic,1,0,0.0,none,short,high,1,1,0,0,0,0,54.0,27.0
2,0,[20-30),25.0,0,648,pregnancy,1,0,3.0,medium,short,high,1,0,0,0,0,0,26.0,12.0
3,0,[30-40),35.0,0,8,infectious,1,0,0.0,none,short,high,1,1,0,0,0,0,32.0,14.0
4,0,[40-50),45.0,0,197,neoplasms,1,0,0.0,none,short,medium,2,0,0,0,0,0,8.0,5.0
5,0,[50-60),55.0,0,414,circulatory,1,0,0.0,none,short,high,1,0,0,0,0,0,48.0,27.0
6,0,[60-70),65.0,1,414,circulatory,0,0,0.0,none,medium,very_high,3,0,0,0,0,0,84.0,28.0
7,0,[70-80),75.0,1,428,circulatory,1,0,0.0,none,medium,high,1,0,0,0,0,0,60.0,40.0
8,0,[80-90),85.0,1,398,circulatory,0,0,0.0,none,long,very_high,2,0,0,0,0,0,364.0,104.0
9,0,[90-100),95.0,1,434,circulatory,0,0,0.0,none,long,high,2,0,0,0,0,0,216.0,96.0


In [27]:
# A1C sanity
df_fe["A1Cresult"].value_counts(dropna=False).head(10)
df_fe["a1c_measured"].value_counts(dropna=False)
df_fe["a1c_high"].value_counts(dropna=False)

# Glucose sanity
df_fe["max_glu_serum"].value_counts(dropna=False).head(10)
df_fe["glu_measured"].value_counts(dropna=False)
df_fe["glu_high"].value_counts(dropna=False)


glu_high
0    99017
1     2749
Name: count, dtype: int64

In [28]:
assert (df_fe.loc[df_fe["glu_high"] == 1, "glu_measured"] == 1).all()
assert (df_fe.loc[df_fe["a1c_high"] == 1, "a1c_measured"] == 1).all()


valid = df_fe["max_glu_serum"].astype(str).str.upper().str.strip()
is_missing = valid.isin(["", "?", "NA", "N/A", "NULL", "NAN", "UNKNOWN", "UNKNOWN/INVALID"])
expected_measured = ~(is_missing | (valid == "NONE"))

assert (df_fe["glu_measured"] == expected_measured).all()

raw = df_fe["A1Cresult"].astype(str).str.upper().str.strip()

is_missing = raw.isin(["", "?", "NA", "N/A", "NULL", "NAN", "UNKNOWN", "UNKNOWN/INVALID"])
expected_measured_a1c = ~(is_missing | (raw == "NONE"))

assert (df_fe["a1c_measured"] == expected_measured_a1c).all()


df_fe["glu_high"].equals(df_fe["max_glu_serum"].isin([">200", ">300"]))

df_fe["a1c_high"].equals(df_fe["A1Cresult"].isin([">7", ">8"]))


pd.crosstab(df_fe["max_glu_serum"], df_fe["glu_high"], margins=True)
pd.crosstab(df_fe["max_glu_serum"], df_fe["glu_measured"], margins=True)

pd.crosstab(df_fe["A1Cresult"], df_fe["a1c_high"], margins=True)
pd.crosstab(df_fe["A1Cresult"], df_fe["a1c_measured"], margins=True)


a1c_measured,1,All
A1Cresult,,
>7,3812,3812
>8,8216,8216
Norm,4990,4990
All,17018,17018


## SECTION 12: Feature Selection and Hypothesis About Selected Groups

In [30]:
%%writefile section12_feature_selection.py
from __future__ import annotations

from dataclasses import dataclass, replace
from functools import partial
from typing import Any, Dict, Iterable, List, Literal, Optional, Sequence, Tuple

import numpy as np
import pandas as pd

from sklearn.feature_selection import SelectFromModel, SelectKBest, RFE, chi2, mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

import section5_preprocessing_pipeline as s5
import section7_validation_protocol as s7


SelectionMethod = Literal[
    "filter_chi2",        
    "filter_mi",          
    "embedded_l1",        
    "wrapper_rfe_mi",     
]



def _ensure_binary_target(y: Iterable[int]) -> np.ndarray:
    y_arr = np.asarray(list(y), dtype=int).ravel()
    if y_arr.size == 0:
        raise ValueError("Empty target.")
    uniq = set(np.unique(y_arr).tolist())
    if not uniq.issubset({0, 1}):
        raise ValueError(f"Target must be binary in {{0,1}}. Found: {sorted(list(uniq))}")
    return y_arr


def _default_X_y(
    df: pd.DataFrame,
    *,
    target_col: str,
    cfg: s5.DiabetesPreprocessConfig,
) -> Tuple[pd.DataFrame, np.ndarray]:
    if target_col not in df.columns:
        raise KeyError(f"Target column '{target_col}' not found in df.")

    y = _ensure_binary_target(df[target_col].astype(int).to_numpy())

    drop_cols = [c for c in cfg.target_cols if c in df.columns]
    if target_col not in drop_cols:
        drop_cols.append(target_col)

    X = df.drop(columns=drop_cols, errors="ignore")
    if X.shape[1] == 0:
        raise ValueError("No feature columns left after dropping target columns.")
    return X, y


def _logreg_l2(*, random_state: int = 42) -> LogisticRegression:
    return LogisticRegression(
        solver="saga",
        penalty="l2",
        C=1.0,
        max_iter=3000,
        random_state=int(random_state),
        n_jobs=-1,
    )


def _logreg_l1(*, C: float = 0.05, random_state: int = 42) -> LogisticRegression:
    return LogisticRegression(
        solver="saga",
        penalty="l1",
        C=float(C),
        max_iter=4000,
        random_state=int(random_state),
        n_jobs=-1,
    )


def _nonneg_config(base: s5.DiabetesPreprocessConfig) -> s5.DiabetesPreprocessConfig:
    """
    Make preprocessing non-negative so chi2 works:
    - disable scaling (StandardScaler can create negatives)
    - remove ordinal encoding (age ordinal can create -1 for unknown); treat age as categorical one-hot instead
    """
    return replace(base, scale_numeric=False, ordinal_cols={})


def _ohe_feature_names(onehot, input_features: List[str]) -> np.ndarray:
    """Sklearn compatibility: get_feature_names_out (new) vs get_feature_names (old)."""
    if hasattr(onehot, "get_feature_names_out"):
        return np.asarray(onehot.get_feature_names_out(input_features), dtype=object)
    if hasattr(onehot, "get_feature_names"):
        return np.asarray(onehot.get_feature_names(input_features), dtype=object)
    raise AttributeError("OneHotEncoder has no get_feature_names_out/get_feature_names.")


def _cols_to_names(cols, feature_names_in: List[str]) -> List[str]:
    """
    Convert ColumnTransformer column selectors to a list of column names.
    Handles: list[str], list[int], numpy arrays, slices.
    """
    if cols is None:
        return []
    if isinstance(cols, slice):
        return [str(c) for c in feature_names_in[cols]]
    if isinstance(cols, (list, tuple, np.ndarray, pd.Index)):
        cols_list = list(cols)
        if len(cols_list) == 0:
            return []
        if isinstance(cols_list[0], (int, np.integer)):
            return [str(feature_names_in[int(i)]) for i in cols_list]
        return [str(c) for c in cols_list]
    return [str(cols)]


def _feature_names_after_preprocessing(pre_fitted: Pipeline) -> np.ndarray:
    """
    Extract output feature names in the EXACT order produced by the fitted preprocessor.
    This avoids mismatches with selector.get_support().
    """
    ct = pre_fitted.named_steps["features"]

    feature_names_in = list(getattr(ct, "feature_names_in_", []))

    names: List[str] = []

    for name, trans, cols in getattr(ct, "transformers_", []):
        if name == "remainder" or trans == "drop":
            continue

        cols_names = _cols_to_names(cols, feature_names_in)

        if name == "num":
            names.extend([f"num__{c}" for c in cols_names])

        elif name == "ord":
            names.extend([f"ord__{c}" for c in cols_names])

        elif name == "cat":
            onehot = getattr(trans, "named_steps", {}).get("onehot", None)
            if onehot is None:
                raise RuntimeError("Expected categorical transformer to be a Pipeline with a 'onehot' step.")
            ohe = _ohe_feature_names(onehot, cols_names)
            names.extend([f"cat__{n}" for n in ohe])

        else:
            names.extend([f"{name}__{c}" for c in cols_names])

    return np.asarray(names, dtype=object)

def _base_column_from_feature_name(feature_name: str, base_cols: Sequence[str]) -> str:
    """
    Map expanded one-hot feature names back to original column names.
    Works with names like:
      - num__time_in_hospital
      - ord__age
      - cat__medical_specialty_Cardiology
    """
    s = str(feature_name)
    if s.startswith("num__"):
        return s[len("num__"):]
    if s.startswith("ord__"):
        return s[len("ord__"):]
    if s.startswith("cat__"):
        rest = s[len("cat__"):]
        for col in sorted(base_cols, key=len, reverse=True):
            if rest == col or rest.startswith(col + "_"):
                return col
        return rest.split("_", 1)[0]
    return s

def _build_interpretation_tables(
    fitted_pipe: Pipeline,
    df: pd.DataFrame,
    cfg: s5.DiabetesPreprocessConfig,
    *,
    method: SelectionMethod,
    top_n_features: int = 40,
    top_n_groups: int = 25,
) -> Dict[str, pd.DataFrame]:
    """
    Fit is already done. Extract:
      - top features (expanded)
      - top groups (original columns)
    """
    pre = fitted_pipe.named_steps["pre"]
    ct = pre.named_steps["features"]

    feature_names = _feature_names_after_preprocessing(pre)

    support = None
    importance = None

    if method in {"filter_chi2", "filter_mi", "embedded_l1"}:
        sel = fitted_pipe.named_steps["select"]
        support = sel.get_support() if hasattr(sel, "get_support") else None

        if method in {"filter_chi2", "filter_mi"} and hasattr(sel, "scores_"):
            importance = np.asarray(sel.scores_, dtype=float)
        elif method == "embedded_l1":
            est = getattr(sel, "estimator_", None)
            if est is not None and hasattr(est, "coef_"):
                importance = np.abs(np.asarray(est.coef_, dtype=float)).ravel()

    elif method == "wrapper_rfe_mi":
        kb = fitted_pipe.named_steps["kbest"]
        rfe = fitted_pipe.named_steps["select"]

        kb_support = kb.get_support()
        kb_idx = np.where(kb_support)[0]

        rfe_support = rfe.get_support()
        picked_in_kb = np.where(rfe_support)[0]
        picked_idx = kb_idx[picked_in_kb]

        support = np.zeros(len(feature_names), dtype=bool)
        support[picked_idx] = True

        imp_full = np.zeros(len(feature_names), dtype=float)
        ranking = np.asarray(getattr(rfe, "ranking_", np.ones(len(kb_idx))), dtype=float)
        imp_kb = 1.0 / np.maximum(1.0, ranking)
        imp_full[kb_idx] = imp_kb
        importance = imp_full

    if support is None:
        support = np.ones(len(feature_names), dtype=bool)
    if importance is None:
        importance = np.zeros(len(feature_names), dtype=float)

    if len(feature_names) != len(support) or len(feature_names) != len(importance):
        raise ValueError(
            "Interpretation shape mismatch:\n"
            f"  n_feature_names={len(feature_names)}\n"
            f"  n_support={len(support)}\n"
            f"  n_importance={len(importance)}\n"
            "This usually means feature-name reconstruction is out of sync with the fitted preprocessor."
        )

    base_cols = list(getattr(ct, "feature_names_in_", []))
    if not base_cols:
        g = s5.infer_feature_groups(df, cfg)
        base_cols = list(g.numeric) + list(g.ordinal) + list(g.categorical)

    base_col = [_base_column_from_feature_name(fn, base_cols=base_cols) for fn in feature_names]

    feat_df = pd.DataFrame(
        {
            "feature": feature_names,
            "base_column": base_col,
            "selected": support.astype(bool),
            "importance": importance.astype(float),
        }
    )

    feat_df_sorted = feat_df.sort_values(["selected", "importance"], ascending=[False, False])
    top_features = feat_df_sorted.head(int(top_n_features)).reset_index(drop=True)

    grp = (
        feat_df.assign(importance_selected=feat_df["importance"] * feat_df["selected"].astype(int))
        .groupby("base_column", as_index=False)
        .agg(
            n_selected=("selected", "sum"),
            n_total=("selected", "size"),
            importance_sum=("importance_selected", "sum"),
        )
    )
    grp["selected_rate"] = grp["n_selected"] / grp["n_total"].replace(0, np.nan)
    grp = grp.sort_values(["importance_sum", "n_selected", "selected_rate"], ascending=False)
    top_groups = grp.head(int(top_n_groups)).reset_index(drop=True)

    return {"top_features": top_features, "top_groups": top_groups, "all_features_table": feat_df}


def build_selection_pipeline(
    df: pd.DataFrame,
    *,
    method: SelectionMethod,
    cfg: s5.DiabetesPreprocessConfig,
    random_state: int = 42,
    k_best: int = 800,
    chi2_k: int = 800,
    l1_C: float = 0.05,
    l1_threshold: str = "median",
    l1_max_features: Optional[int] = None,
    wrapper_k_pre: int = 1200,
    wrapper_n: int = 250,
    wrapper_step: float = 0.2,
) -> Tuple[Pipeline, s5.DiabetesPreprocessConfig]:
    """
    Returns (pipeline, cfg_used).
    """
    cfg_used = cfg
    if method == "filter_chi2":
        cfg_used = _nonneg_config(cfg_used)

    pre = s5.build_preprocessor(df, cfg_used)
    clf = _logreg_l2(random_state=random_state)

    if method == "filter_chi2":
        selector = SelectKBest(score_func=chi2, k=int(chi2_k))
        pipe = Pipeline([("pre", pre), ("select", selector), ("clf", clf)])
        return pipe, cfg_used

    if method == "filter_mi":
        mi = partial(mutual_info_classif, discrete_features="auto", random_state=int(random_state))
        selector = SelectKBest(score_func=mi, k=int(k_best))
        pipe = Pipeline([("pre", pre), ("select", selector), ("clf", clf)])
        return pipe, cfg_used

    if method == "embedded_l1":
        base_est = _logreg_l1(C=l1_C, random_state=random_state)
        selector = SelectFromModel(
            estimator=base_est,
            threshold=l1_threshold,   
            max_features=l1_max_features,
        )
        pipe = Pipeline([("pre", pre), ("select", selector), ("clf", clf)])
        return pipe, cfg_used

    if method == "wrapper_rfe_mi":
        mi = partial(mutual_info_classif, discrete_features="auto", random_state=int(random_state))
        kbest = SelectKBest(score_func=mi, k=int(wrapper_k_pre))

        rfe_est = _logreg_l2(random_state=random_state)
        rfe = RFE(estimator=rfe_est, n_features_to_select=int(wrapper_n), step=float(wrapper_step))

        pipe = Pipeline([("pre", pre), ("kbest", kbest), ("select", rfe), ("clf", clf)])
        return pipe, cfg_used

    raise ValueError(f"Unknown method: {method}")


@dataclass(frozen=True)
class Section12Result:
    split_fingerprint: str
    summary_table: pd.DataFrame
    interpretations: Dict[str, Dict[str, pd.DataFrame]]  


def run_section12_feature_selection(
    df: pd.DataFrame,
    *,
    target_col: str = "readmitted_30d",
    preprocess_config_base: Optional[s5.DiabetesPreprocessConfig] = None,
    cv_splits: Optional[Sequence[s7.Split]] = None,
    n_splits: int = 10,
    cv_random_state: int = 42,
    model_random_state: int = 42,
    methods: Sequence[SelectionMethod] = ("filter_chi2", "embedded_l1"),
    k_best: int = 800,
    chi2_k: int = 800,
    l1_C: float = 0.05,
    l1_threshold: str = "median",
    l1_max_features: Optional[int] = None,
    wrapper_k_pre: int = 1200,
    wrapper_n: int = 250,
    wrapper_step: float = 0.2,
    build_interpretation: bool = True,
    top_n_features: int = 40,
    top_n_groups: int = 25,
) -> Section12Result:
    """
    Produces:
      - CV comparison table: all-features vs selected-features (same splits)
      - Interpretation tables to support your “hypothesis about selected groups”
    """
    if preprocess_config_base is None:
        preprocess_config_base = s5.DiabetesPreprocessConfig()

    X_base, y = _default_X_y(df, target_col=target_col, cfg=preprocess_config_base)

    if cv_splits is None:
        cv_splits = s7.make_stratified_cv_splits(
            y, n_splits=int(n_splits), shuffle=True, random_state=int(cv_random_state)
        )
    else:
        s7.validate_cv_splits(cv_splits, n_samples=len(y), n_splits_expected=int(n_splits))

    fp = s7.cv_splits_fingerprint(cv_splits)

    rows: List[Dict[str, Any]] = []
    interpretations: Dict[str, Dict[str, pd.DataFrame]] = {}

    def _eval_and_row(method_name: str, variant: str, pipe: Pipeline) -> s7.CVEvaluationResult:
        res = s7.evaluate_binary_pipeline_cv(
            pipe, X_base, y,
            cv_splits=cv_splits,
            metrics=("roc_auc", "precision", "recall", "f1", "accuracy", "balanced_accuracy"),
            return_oof=False,
        )
        row = {
            "method": method_name,
            "variant": variant,  
            "split_fingerprint": fp,
        }
        for m in res.mean_scores:
            row[f"{m}_mean"] = float(res.mean_scores[m])
            row[f"{m}_std"] = float(res.std_scores[m])
        rows.append(row)
        return res

    for method in methods:
        sel_pipe, cfg_used = build_selection_pipeline(
            df,
            method=method,
            cfg=preprocess_config_base,
            random_state=model_random_state,
            k_best=k_best,
            chi2_k=chi2_k,
            l1_C=l1_C,
            l1_threshold=l1_threshold,
            l1_max_features=l1_max_features,
            wrapper_k_pre=wrapper_k_pre,
            wrapper_n=wrapper_n,
            wrapper_step=wrapper_step,
        )

        pre_all = s5.build_preprocessor(df, cfg_used)
        all_pipe = Pipeline([("pre", pre_all), ("clf", _logreg_l2(random_state=model_random_state))])

        _eval_and_row(str(method), "all_features", all_pipe)
        _eval_and_row(str(method), "selected", sel_pipe)

        if build_interpretation:
            fitted = Pipeline(sel_pipe.steps)  
            fitted.fit(X_base, y)
            tables = _build_interpretation_tables(
                fitted, df, cfg_used,
                method=method,
                top_n_features=top_n_features,
                top_n_groups=top_n_groups,
            )
            interpretations[str(method)] = tables

    summary = pd.DataFrame(rows).sort_values(["method", "variant"]).reset_index(drop=True)
    return Section12Result(split_fingerprint=fp, summary_table=summary, interpretations=interpretations)


Overwriting section12_feature_selection.py


### The following code executes Section 12 using the dataset with the real data.

In [ ]:
import section5_preprocessing_pipeline as s5
import section7_validation_protocol as s7
import section12_feature_selection as s12

cfg = s5.DiabetesPreprocessConfig(
    onehot_sparse=True,
    rare_min_count=50,
    scale_numeric=True,
    scaler="standard",
)

y = df["readmitted_30d"].astype(int).to_numpy()

splits = s7.make_stratified_cv_splits(
    y,
    n_splits=10,
    random_state=42,   
)

out12_wrapper = s12.run_section12_feature_selection(
    df,
    target_col="readmitted_30d",
    preprocess_config_base=cfg,
    cv_splits=splits,
    n_splits=10,
    methods=("wrapper_rfe_mi",),   
    wrapper_k_pre=1200,            
    wrapper_n=200,                 
    wrapper_step=0.2,              
    top_n_features=40,             
    top_n_groups=25,               
)

out12_wrapper.summary_table


C:\Users\Cris-SX\AppData\Roaming\Python\Python313\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: [0]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


## SECTION 13: Dimensionality Reduction

In [ ]:
%%writefile section13_dimensionality_reduction.py

from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Sequence, Tuple, Literal

import numpy as np
import pandas as pd
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

import section5_preprocessing_pipeline as s5
import section7_validation_protocol as s7

Split = s7.Split
ReductionMethod = Literal["none", "svd"]


def _validate_binary_target(y: pd.Series) -> np.ndarray:
    if pd.Series(y).isna().any():
        raise ValueError("Target contains missing values; Section 13 requires a clean binary target.")
    y_arr = np.asarray(pd.Series(y).astype(int).tolist(), dtype=int)
    uniq = set(np.unique(y_arr).tolist())
    if not uniq.issubset({0, 1}):
        raise ValueError(f"Target must be binary in {{0,1}}, found values: {sorted(list(uniq))}")
    return y_arr


def _default_X_y(
    df: pd.DataFrame,
    *,
    target_col: str,
    preprocess_config: s5.DiabetesPreprocessConfig,
) -> Tuple[pd.DataFrame, np.ndarray]:
    if target_col not in df.columns:
        raise KeyError(f"Target column '{target_col}' not found. Available: {list(df.columns)}")

    y = _validate_binary_target(df[target_col])

    drop_cols = [c for c in preprocess_config.target_cols if c in df.columns]
    if target_col not in drop_cols:
        drop_cols.append(target_col)

    X = df.drop(columns=drop_cols, errors="ignore")
    if X.shape[1] == 0:
        raise ValueError("No feature columns available after dropping target columns.")
    return X, y


def _validate_components(components: Sequence[int]) -> List[int]:
    out: List[int] = []
    for c in components:
        try:
            ci = int(c)
        except Exception:
            raise ValueError(f"All n_components must be integers. Got: {c!r}")
        if ci <= 0:
            raise ValueError(f"All n_components must be > 0. Got: {ci}")
        out.append(ci)
    seen = set()
    uniq = []
    for ci in out:
        if ci not in seen:
            uniq.append(ci)
            seen.add(ci)
    return uniq


@dataclass(frozen=True)
class DimRedVariant:
    name: str
    method: ReductionMethod
    n_components: Optional[int] = None


@dataclass(frozen=True)
class Section13RunResult:
    """
    Section 13: Dimensionality Reduction ablation.

    We implement "PCA-like" reduction for sparse one-hot data via TruncatedSVD:
    - After one-hot encoding, X is typically a high-dimensional sparse matrix.
    - Classic PCA requires centering and generally dense matrices.
    - TruncatedSVD is the standard practical alternative on sparse matrices,
      and is commonly used as a PCA analogue for sparse feature spaces.

    Returns:
      - per-variant CV results with identical folds (for fair comparisons)
      - a compact mean±std summary table for the report
    """
    target_col: str
    n_splits: int
    cv_random_state: int
    split_fingerprint: str
    variants: List[DimRedVariant]
    results_by_variant: Dict[str, s7.CVEvaluationResult]
    summary_table: pd.DataFrame


def build_section13_pipelines(
    df: pd.DataFrame,
    *,
    preprocess_config: Optional[s5.DiabetesPreprocessConfig] = None,
    model_random_state: int = 42,
    svd_components: Sequence[int] = (50, 100, 200),
) -> Tuple[List[DimRedVariant], Dict[str, Pipeline]]:
    """
    Build pipelines for:
      - no reduction (baseline)
      - TruncatedSVD(k) for each k in svd_components
    """
    if preprocess_config is None:
        preprocess_config = s5.DiabetesPreprocessConfig()

    comps = _validate_components(svd_components)

    pre = s5.build_preprocessor(df, preprocess_config)

    clf = LogisticRegression(
        solver="saga",         
        penalty="l2",
        max_iter=3000,
        random_state=model_random_state,
        n_jobs=-1,
    )

    variants: List[DimRedVariant] = []
    pipes: Dict[str, Pipeline] = {}

    v0 = DimRedVariant(name="no_reduction", method="none", n_components=None)
    variants.append(v0)
    pipes[v0.name] = Pipeline(steps=[("pre", pre), ("clf", clf)])

    for k in comps:
        vk = DimRedVariant(name=f"svd_{k}", method="svd", n_components=int(k))
        variants.append(vk)
        pipes[vk.name] = Pipeline(
            steps=[
                ("pre", pre),
                ("svd", TruncatedSVD(n_components=int(k), random_state=model_random_state)),
                ("clf", clf),
            ]
        )

    return variants, pipes


def _summary_table(
    variants: Sequence[DimRedVariant],
    results_by_variant: Dict[str, s7.CVEvaluationResult],
    *,
    metrics: Sequence[str],
) -> pd.DataFrame:
    rows: List[Dict[str, Any]] = []
    for v in variants:
        r = results_by_variant[v.name]
        row: Dict[str, Any] = {
            "variant": v.name,
            "method": v.method,
            "n_components": (np.nan if v.n_components is None else int(v.n_components)),
        }
        for m in metrics:
            row[f"{m}_mean"] = float(r.mean_scores[m])
            row[f"{m}_std"] = float(r.std_scores[m])
        rows.append(row)

    cols = ["variant", "method", "n_components"] + [f"{m}_{s}" for m in metrics for s in ("mean", "std")]
    return pd.DataFrame(rows)[cols]


def run_section13_dimensionality_reduction(
    df: pd.DataFrame,
    *,
    target_col: str = "readmitted_30d",
    preprocess_config: Optional[s5.DiabetesPreprocessConfig] = None,
    svd_components: Sequence[int] = (50, 100, 200),
    metrics: Sequence[str] = ("roc_auc", "precision", "recall", "f1", "accuracy", "balanced_accuracy"),
    n_splits: int = 10,
    cv_random_state: int = 42,
    model_random_state: int = 42,
    cv_splits: Optional[Sequence[Split]] = None,
    return_oof: bool = False,
) -> Section13RunResult:
    """
    End-to-end runner for Section 13 (ablation):
      - builds baseline (no reduction) + SVD(k) variants
      - evaluates each variant with the SAME cv_splits (required for fair comparison)
      - returns mean±std table + fold vectors (usable later for Wilcoxon)
    """
    if preprocess_config is None:
        preprocess_config = s5.DiabetesPreprocessConfig()

    X, y = _default_X_y(df, target_col=target_col, preprocess_config=preprocess_config)

    if cv_splits is None:
        cv_splits = s7.make_stratified_cv_splits(y, n_splits=n_splits, shuffle=True, random_state=cv_random_state)
    else:
        s7.validate_cv_splits(cv_splits, n_samples=len(y), n_splits_expected=n_splits)

    fp = s7.cv_splits_fingerprint(cv_splits)

    variants, pipes = build_section13_pipelines(
        df,
        preprocess_config=preprocess_config,
        model_random_state=model_random_state,
        svd_components=svd_components,
    )

    results_by_variant: Dict[str, s7.CVEvaluationResult] = {}
    for v in variants:
        res = s7.evaluate_binary_pipeline_cv(
            pipes[v.name],
            X,
            y,
            cv_splits=cv_splits,
            metrics=metrics,
            return_oof=return_oof,
        )
        results_by_variant[v.name] = res

    table = _summary_table(variants, results_by_variant, metrics=metrics)

    return Section13RunResult(
        target_col=target_col,
        n_splits=n_splits,
        cv_random_state=cv_random_state,
        split_fingerprint=fp,
        variants=list(variants),
        results_by_variant=results_by_variant,
        summary_table=table,
    )


def pick_best_variant_by_auc(out: Section13RunResult) -> str:
    """Convenience: return the variant name with best mean ROC-AUC."""
    if "roc_auc_mean" not in out.summary_table.columns:
        raise ValueError("summary_table has no roc_auc_mean column. Did you include roc_auc in metrics?")
    best_idx = int(out.summary_table["roc_auc_mean"].astype(float).idxmax())
    return str(out.summary_table.loc[best_idx, "variant"])


Writing section13_dimensionality_reduction.py


### The following code executes Section 13 using the dataset with the real data.

In [ ]:
import os, sys, importlib.util
import pandas as pd

def import_module_from_path(module_name: str, file_path: str):
    spec = importlib.util.spec_from_file_location(module_name, file_path)
    if spec is None or spec.loader is None:
        raise ImportError(f"Could not load spec for {module_name} from {file_path}")

    mod = importlib.util.module_from_spec(spec)

    sys.modules.pop(module_name, None)
    sys.modules[module_name] = mod

    spec.loader.exec_module(mod)
    return mod

PROJECT_DIR = os.getcwd()
CODE_PATH = os.path.join(PROJECT_DIR, "code.py")
project_code = import_module_from_path("project_code", CODE_PATH)

DATA_PATH = os.path.join(PROJECT_DIR, "database", "diabetic_data.csv")
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"CSV not found at: {DATA_PATH}")

df_raw = pd.read_csv(DATA_PATH, low_memory=False)
print("Loaded:", df_raw.shape)
display(df_raw.head(3))


df = df_raw.copy()

if "id_maps" not in globals():
    id_maps = {}  

def apply_mapping(df: pd.DataFrame, id_col: str, new_col: str) -> pd.DataFrame:
    if id_col not in df.columns or id_col not in id_maps:
        return df
    map_df = id_maps[id_col]
    if id_col not in map_df.columns or "description" not in map_df.columns:
        raise ValueError(f"id_maps['{id_col}'] must have columns [{id_col}, 'description'].")

    mapper = map_df.set_index(id_col)["description"]
    mapper.index = mapper.index.astype("string")

    keys = df[id_col].astype("string").str.strip()
    df[new_col] = keys.map(mapper)  
    return df

df = apply_mapping(df, "admission_type_id", "admission_type")
df = apply_mapping(df, "discharge_disposition_id", "discharge_disposition")
df = apply_mapping(df, "admission_source_id", "admission_source")

to_drop = [c for c in ["admission_type_id", "discharge_disposition_id", "admission_source_id"]
           if c in df.columns and c in id_maps]
if to_drop:
    df = df.drop(columns=to_drop)

df = project_code.engineer_targets(
    df,
    source_col="readmitted",
    binary_col="readmitted_30d",
    multiclass_col="readmitted_3class",
    add_multiclass=True,
    drop_source=False,
    unknown_policy="error",  
)

display(df[["readmitted", "readmitted_30d", "readmitted_3class"]].head(10))
print("\nreadmitted value counts:")
print(df["readmitted"].value_counts(dropna=False))
print("\nBinary target counts:")
print(df["readmitted_30d"].value_counts(dropna=False))


Loaded: (101766, 50)


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO


,readmitted,readmitted_30d,readmitted_3class
0,NO,0,NO
1,>30,0,>30
2,NO,0,NO
3,NO,0,NO
4,NO,0,NO
5,>30,0,>30
6,NO,0,NO
7,>30,0,>30
8,NO,0,NO
9,NO,0,NO



readmitted value counts:
readmitted
NO     54864
>30    35545
<30    11357
Name: count, dtype: int64

Binary target counts:
readmitted_30d
0    90409
1    11357
Name: count, dtype: int64


## SECTION 14: Hyperparameter Tuning

In [33]:
%%writefile section14_hyperparameter_tuning.py
from __future__ import annotations

from dataclasses import dataclass
from collections import Counter
from typing import Any, Dict, List, Optional, Sequence, Tuple, Literal

import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
)

import section5_preprocessing_pipeline as s5
import section7_validation_protocol as s7

Split = s7.Split

class SparseToCSC(BaseEstimator, TransformerMixin):
    """Convert sparse matrices to CSC (helps some tree models)."""
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        try:
            from scipy import sparse
        except Exception as e:
            raise RuntimeError("scipy is required. Install with: pip install scipy") from e
        return X.tocsc() if sparse.issparse(X) else X


def _safe_index(X, idx: np.ndarray):
    if isinstance(X, (pd.DataFrame, pd.Series)):
        return X.iloc[idx]
    return X[idx]


def _score_vector(estimator, X, *, positive_label: int = 1) -> np.ndarray:
    if hasattr(estimator, "predict_proba"):
        proba = np.asarray(estimator.predict_proba(X))
        if proba.ndim != 2 or proba.shape[1] < 2:
            raise ValueError("predict_proba must return [n_samples, 2+] for binary AUC.")
        classes = getattr(estimator, "classes_", None)
        if classes is None:
            pos_idx = 1
        else:
            classes = np.asarray(classes)
            pos_idx = int(np.where(classes == positive_label)[0][0]) if positive_label in set(classes.tolist()) else 1
        return proba[:, pos_idx].astype(float)

    if hasattr(estimator, "decision_function"):
        return np.asarray(estimator.decision_function(X), dtype=float).ravel()

    raise ValueError("Estimator must expose predict_proba or decision_function for ROC-AUC.")


def _compute_metrics(y_true: np.ndarray, y_pred: np.ndarray, y_score: np.ndarray) -> Dict[str, float]:
    return {
        "roc_auc": float(roc_auc_score(y_true, y_score)),
        "precision": float(precision_score(y_true, y_pred, pos_label=1, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, pos_label=1, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, pos_label=1, zero_division=0)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
    }


def default_X_y(
    df: pd.DataFrame,
    *,
    target_col: str,
    preprocess_config: s5.DiabetesPreprocessConfig,
) -> Tuple[pd.DataFrame, np.ndarray]:
    if target_col not in df.columns:
        raise KeyError(f"Target column '{target_col}' not found in df.")
    y = df[target_col].astype(int).to_numpy()
    drop_cols = [c for c in preprocess_config.target_cols if c in df.columns]
    if target_col not in drop_cols:
        drop_cols.append(target_col)
    X = df.drop(columns=drop_cols, errors="ignore")
    if X.shape[1] == 0:
        raise ValueError("No features left after dropping target columns.")
    return X, y

def manual_sweep_logreg_C(
    df: pd.DataFrame,
    *,
    target_col: str = "readmitted_30d",
    preprocess_config: Optional[s5.DiabetesPreprocessConfig] = None,
    cv_splits: Sequence[Split],
    C_values: Sequence[float] = (0.01, 0.1, 1.0, 3.0, 10.0),
    class_weight: Optional[str] = None,  
    random_state: int = 42,
) -> pd.DataFrame:
    """
    Evaluate LogisticRegression for several C values on the SAME fixed outer folds.
    This is exactly the "manual hyperparameter exploration" objective.
    """
    if preprocess_config is None:
        preprocess_config = s5.DiabetesPreprocessConfig(scale_numeric=True, scaler="standard")

    X, y = default_X_y(df, target_col=target_col, preprocess_config=preprocess_config)
    s7.validate_cv_splits(cv_splits, n_samples=len(y))

    from sklearn.linear_model import LogisticRegression

    pre = s5.build_preprocessor(df, preprocess_config)

    rows: List[Dict[str, Any]] = []
    for C in C_values:
        pipe = Pipeline(
            steps=[
                ("pre", pre),
                ("clf", LogisticRegression(
                    solver="saga",
                    penalty="l2",
                    C=float(C),
                    class_weight=class_weight,
                    max_iter=4000,
                    n_jobs=-1,
                    random_state=int(random_state),
                )),
            ]
        )

        res = s7.evaluate_binary_pipeline_cv(
            pipe, X, y,
            cv_splits=cv_splits,
            metrics=("roc_auc","precision","recall","f1","accuracy","balanced_accuracy"),
            return_oof=False,
        )
        rows.append({
            "model": "logreg",
            "C": float(C),
            "class_weight": (class_weight if class_weight is not None else "None"),
            **{f"{m}_mean": float(res.mean_scores[m]) for m in res.mean_scores},
            **{f"{m}_std": float(res.std_scores[m]) for m in res.std_scores},
        })

    out = pd.DataFrame(rows).sort_values("roc_auc_mean", ascending=False).reset_index(drop=True)
    return out

def gridsearch_logreg(
    df: pd.DataFrame,
    *,
    target_col: str = "readmitted_30d",
    preprocess_config: Optional[s5.DiabetesPreprocessConfig] = None,
    inner_folds: int = 5,
    random_state: int = 42,
    n_jobs: int = -1,
    verbose: int = 1,
):
    if preprocess_config is None:
        preprocess_config = s5.DiabetesPreprocessConfig(scale_numeric=True, scaler="standard")

    X, y = default_X_y(df, target_col=target_col, preprocess_config=preprocess_config)
    from sklearn.linear_model import LogisticRegression

    pre = s5.build_preprocessor(df, preprocess_config)

    pipe = Pipeline(
        steps=[
            ("pre", pre),
            ("clf", LogisticRegression(
                solver="saga",
                max_iter=5000,
                n_jobs=-1,
                random_state=int(random_state),
            )),
        ]
    )

    param_grid = [
        {"clf__penalty": ["l2"], "clf__C": [0.01, 0.1, 1.0, 3.0, 10.0], "clf__class_weight": [None, "balanced"]},
        {"clf__penalty": ["l1"], "clf__C": [0.01, 0.1, 1.0, 3.0, 10.0], "clf__class_weight": [None, "balanced"]},
    ]

    inner_cv = StratifiedKFold(n_splits=int(inner_folds), shuffle=True, random_state=int(random_state))

    gs = GridSearchCV(
        estimator=pipe,
        param_grid=param_grid,
        scoring="roc_auc",
        refit=True,
        cv=inner_cv,
        n_jobs=int(n_jobs),
        verbose=int(verbose),
        return_train_score=False,
    )
    gs.fit(X, y)
    return gs


# Nested CV
SearchKind = Literal["grid", "random"]


@dataclass(frozen=True)
class NestedCVResult:
    model_name: str
    outer_split_fingerprint: str
    fold_metrics: pd.DataFrame
    mean_scores: Dict[str, float]
    std_scores: Dict[str, float]
    best_params_per_fold: List[Dict[str, Any]]
    best_params_counts: List[Tuple[str, int]]


def nested_cv_tuning(
    df: pd.DataFrame,
    *,
    model_name: Literal["logreg","random_forest"] = "logreg",
    target_col: str = "readmitted_30d",
    preprocess_config: Optional[s5.DiabetesPreprocessConfig] = None,
    outer_splits: Sequence[Split],
    inner_folds: int = 3,
    search: SearchKind = "random",
    n_iter: int = 20,                
    random_state: int = 42,
    n_jobs: int = -1,
    verbose: int = 0,
) -> NestedCVResult:
    if preprocess_config is None:
        preprocess_config = s5.DiabetesPreprocessConfig()

    X, y = default_X_y(df, target_col=target_col, preprocess_config=preprocess_config)
    s7.validate_cv_splits(outer_splits, n_samples=len(y))
    fp_outer = s7.cv_splits_fingerprint(outer_splits)

    inner_cv = StratifiedKFold(n_splits=int(inner_folds), shuffle=True, random_state=int(random_state))

    if model_name == "logreg":
        from sklearn.linear_model import LogisticRegression
        cfg = preprocess_config
        if not cfg.scale_numeric:
            cfg = s5.DiabetesPreprocessConfig(**{**cfg.__dict__, "scale_numeric": True, "scaler": "standard"})  

        pre = s5.build_preprocessor(df, cfg)
        base = Pipeline([
            ("pre", pre),
            ("clf", LogisticRegression(
                solver="saga",
                max_iter=5000,
                n_jobs=-1,
                random_state=int(random_state),
            )),
        ])
        param_grid = [
            {"clf__penalty": ["l2"], "clf__C": [0.001,0.01,0.1,1,3,10], "clf__class_weight": [None,"balanced"]},
            {"clf__penalty": ["l1"], "clf__C": [0.001,0.01,0.1,1,3,10], "clf__class_weight": [None,"balanced"]},
        ]
        param_dist = {
            "clf__penalty": ["l2","l1"],
            "clf__C": list(np.logspace(-3, 1, 20)),
            "clf__class_weight": [None, "balanced"],
        }

    elif model_name == "random_forest":
        from sklearn.ensemble import RandomForestClassifier
        cfg = preprocess_config
        pre = s5.build_preprocessor(df, cfg)
        base = Pipeline([
            ("pre", pre),
            ("to_csc", SparseToCSC()),
            ("clf", RandomForestClassifier(
                n_estimators=300,
                n_jobs=-1,
                random_state=int(random_state),
            )),
        ])
        param_grid = {
            "clf__n_estimators": [200, 400],
            "clf__max_depth": [None, 10, 20],
            "clf__min_samples_split": [2, 10],
            "clf__min_samples_leaf": [1, 5],
            "clf__max_features": ["sqrt", 0.3],
            "clf__class_weight": [None, "balanced", "balanced_subsample"],
        }
        param_dist = {
            "clf__n_estimators": [200, 300, 400, 600],
            "clf__max_depth": [None, 8, 12, 18, 25],
            "clf__min_samples_split": [2, 5, 10, 20],
            "clf__min_samples_leaf": [1, 2, 5, 10],
            "clf__max_features": ["sqrt", 0.2, 0.3, 0.5],
            "clf__class_weight": [None, "balanced", "balanced_subsample"],
        }

    else:
        raise ValueError("model_name must be 'logreg' or 'random_forest'")

    fold_rows: List[Dict[str, Any]] = []
    best_params: List[Dict[str, Any]] = []

    for fold_id, (tr, te) in enumerate(outer_splits, start=1):
        tr = np.asarray(tr, dtype=np.int64)
        te = np.asarray(te, dtype=np.int64)

        X_tr, y_tr = _safe_index(X, tr), y[tr]
        X_te, y_te = _safe_index(X, te), y[te]

        if search == "grid":
            searcher = GridSearchCV(
                estimator=base,
                param_grid=param_grid,
                scoring="roc_auc",
                refit=True,
                cv=inner_cv,
                n_jobs=int(n_jobs),
                verbose=int(verbose),
                return_train_score=False,
            )
        else:
            searcher = RandomizedSearchCV(
                estimator=base,
                param_distributions=param_dist,
                n_iter=int(n_iter),
                scoring="roc_auc",
                refit=True,
                cv=inner_cv,
                random_state=int(random_state),
                n_jobs=int(n_jobs),
                verbose=int(verbose),
                return_train_score=False,
            )

        searcher.fit(X_tr, y_tr)
        best_est = searcher.best_estimator_
        best_params.append(dict(searcher.best_params_))

        y_pred = np.asarray(best_est.predict(X_te), dtype=int)
        y_score = _score_vector(best_est, X_te, positive_label=1)
        m = _compute_metrics(y_te, y_pred, y_score)

        fold_rows.append({
            "fold": fold_id,
            "roc_auc": m["roc_auc"],
            "precision": m["precision"],
            "recall": m["recall"],
            "f1": m["f1"],
            "accuracy": m["accuracy"],
            "balanced_accuracy": m["balanced_accuracy"],
        })

    fold_df = pd.DataFrame(fold_rows)
    mean_scores = {c: float(fold_df[c].mean()) for c in fold_df.columns if c != "fold"}
    std_scores = {c: float(fold_df[c].std(ddof=1)) for c in fold_df.columns if c != "fold"}

    sigs = [str(sorted(p.items())) for p in best_params]
    counts = Counter(sigs).most_common(10)

    return NestedCVResult(
        model_name=model_name,
        outer_split_fingerprint=fp_outer,
        fold_metrics=fold_df,
        mean_scores=mean_scores,
        std_scores=std_scores,
        best_params_per_fold=best_params,
        best_params_counts=counts,
    )


Overwriting section14_hyperparameter_tuning.py


### The following code executes Section 14 using the dataset with the real data.

In [ ]:
import os, sys, importlib.util
import pandas as pd

import section5_preprocessing_pipeline as s5
import section7_validation_protocol as s7
import section14_hyperparameter_tuning as s14

def import_module_from_path(module_name: str, file_path: str):
    spec = importlib.util.spec_from_file_location(module_name, file_path)
    if spec is None or spec.loader is None:
        raise ImportError(f"Could not load spec for {module_name} from {file_path}")
    mod = importlib.util.module_from_spec(spec)
    sys.modules.pop(module_name, None)
    sys.modules[module_name] = mod
    spec.loader.exec_module(mod)
    return mod

PROJECT_DIR = os.getcwd()
project_code = import_module_from_path("project_code", os.path.join(PROJECT_DIR, "code.py"))

DATA_PATH = os.path.join(PROJECT_DIR, "database", "diabetic_data.csv")
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"CSV not found at: {DATA_PATH}")

df_raw = pd.read_csv(DATA_PATH, low_memory=False)
print("Loaded:", df_raw.shape)

df = project_code.engineer_targets(
    df_raw,
    source_col="readmitted",
    binary_col="readmitted_30d",
    multiclass_col="readmitted_3class",
    add_multiclass=True,
    drop_source=False,
    unknown_policy="error",
)

cfg = s5.DiabetesPreprocessConfig(
    onehot_sparse=True,
    rare_min_count=50,
    scale_numeric=True,
    scaler="standard",
)

y = df["readmitted_30d"].astype(int).to_numpy()
splits = s7.make_stratified_cv_splits(y, n_splits=10, shuffle=True, random_state=42)
print("Fold fingerprint:", s7.cv_splits_fingerprint(splits))



[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Loaded: (101766, 50)
Fold fingerprint: f5968b2fab085e7f54fd828ac2d60e09701e9a727701c3d6a99a4e5b64150ef5


In [ ]:
manual_table = s14.manual_sweep_logreg_C(
    df,
    target_col="readmitted_30d",
    preprocess_config=cfg,
    cv_splits=splits,
    C_values=(0.001, 0.01, 0.1, 1.0, 3.0, 10.0),
    class_weight=None,         
    random_state=42,
)
manual_table


C:\Users\Cris-SX\AppData\Roaming\Python\Python313\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: [0]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
C:\Users\Cris-SX\AppData\Roaming\Python\Python313\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: [0]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
C:\Users\Cris-SX\AppData\Roaming\Python\Python313\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: [0]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
C:\Users\Cris-SX\AppData\Roaming\Python\Python313\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: [0]. At least one non-missing value is needed for imputation with strategy='median

KeyboardInterrupt: 

In [ ]:
gs = s14.gridsearch_logreg(
    df,
    target_col="readmitted_30d",
    preprocess_config=cfg,
    inner_folds=5,
    random_state=42,
    n_jobs=-1,
    verbose=1,
)
print("Best params:", gs.best_params_)
print("Best inner-CV ROC-AUC:", gs.best_score_)


In [ ]:
nested_lr = s14.nested_cv_tuning(
    df,
    model_name="logreg",
    target_col="readmitted_30d",
    preprocess_config=cfg,
    outer_splits=splits,
    inner_folds=3,       
    search="random",     
    n_iter=20,           
    random_state=42,
    n_jobs=-1,
    verbose=0,
)

nested_lr.fold_metrics, nested_lr.mean_scores, nested_lr.std_scores, nested_lr.best_params_counts


In [ ]:
nested_rf = s14.nested_cv_tuning(
    df,
    model_name="random_forest",
    target_col="readmitted_30d",
    preprocess_config=cfg,
    outer_splits=splits,
    inner_folds=3,
    search="random",
    n_iter=15,          
    random_state=42,
    n_jobs=-1,
    verbose=0,
)

nested_rf.fold_metrics, nested_rf.mean_scores, nested_rf.std_scores, nested_rf.best_params_counts


## SECTION 15: Statistical Comparison of the Best Models

In [34]:
%%writefile section15_statistical_comparison.py
from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Dict, Optional, Sequence, Tuple

import numpy as np
from scipy.stats import rankdata, wilcoxon


@dataclass(frozen=True)
class WilcoxonSignedRankReport:
    model_a: str
    model_b: str
    metric: str
    n_pairs: int
    alpha: float
    alternative: str
    statistic: float
    p_value: float
    mean_diff: float
    median_diff: float
    effect_r_rank_biserial: float
    split_fingerprint: Optional[str]
    reject_h0: bool
    winner: str

    def to_text(self) -> str:
        h0 = f"H0: median({self.model_a} − {self.model_b}) = 0"
        h1 = f"H1 ({self.alternative}): median difference ≠ 0" if self.alternative == "two-sided" else f"H1 ({self.alternative})"
        decision = "REJECT H0" if self.reject_h0 else "FAIL TO REJECT H0"
        return (
            f"Wilcoxon signed-rank test on '{self.metric}' (paired folds)\n"
            f"- Models: A={self.model_a} vs B={self.model_b}\n"
            f"- {h0}\n"
            f"- {h1}\n"
            f"- alpha={self.alpha:.3f}, n_pairs_used={self.n_pairs}\n"
            f"- statistic={self.statistic:.6g}, p_value={self.p_value:.6g}\n"
            f"- mean(A−B)={self.mean_diff:.6g}, median(A−B)={self.median_diff:.6g}\n"
            f"- rank-biserial r={self.effect_r_rank_biserial:.6g} (≈effect size)\n"
            f"- split_fingerprint={self.split_fingerprint}\n"
            f"- decision: {decision}\n"
            f"- winner (by median diff): {self.winner}\n"
        )


def _as_1d_float(x: Sequence[float]) -> np.ndarray:
    arr = np.asarray(list(x), dtype=float).ravel()
    if arr.ndim != 1:
        raise ValueError("Input must be 1D.")
    return arr


def _rank_biserial_from_differences(d: np.ndarray) -> Tuple[float, int]:
    """
    Rank-biserial correlation for Wilcoxon signed-rank:
      r = (W+ - W-) / (W+ + W-)
    where W+ is sum of ranks for positive diffs, W- for negative diffs,
    after removing zeros.
    """
    d = np.asarray(d, dtype=float).ravel()
    nz = d != 0
    d_nz = d[nz]
    n = int(d_nz.size)
    if n == 0:
        return 0.0, 0

    ranks = rankdata(np.abs(d_nz), method="average")
    w_plus = float(ranks[d_nz > 0].sum())
    w_minus = float(ranks[d_nz < 0].sum())
    denom = w_plus + w_minus
    if denom == 0:
        return 0.0, n
    r = (w_plus - w_minus) / denom
    return float(r), n


def wilcoxon_compare_fold_scores(
    scores_a: Sequence[float],
    scores_b: Sequence[float],
    *,
    model_a_name: str = "model_A",
    model_b_name: str = "model_B",
    metric: str = "roc_auc",
    alpha: float = 0.05,
    alternative: str = "two-sided",
    zero_method: str = "wilcox",
    correction: bool = False,
    split_fingerprint: Optional[str] = None,
) -> WilcoxonSignedRankReport:
    """
    Paired Wilcoxon signed-rank test comparing two models across identical CV folds.
    """
    a = _as_1d_float(scores_a)
    b = _as_1d_float(scores_b)
    if a.size != b.size:
        raise ValueError(f"Fold score vectors must have same length. Got {a.size} vs {b.size}.")

    d = a - b
    effect_r, n_used = _rank_biserial_from_differences(d)

    stat, p = wilcoxon(
        a,
        b,
        alternative=alternative,
        zero_method=zero_method,
        correction=correction,
        method="auto",
    )

    mean_diff = float(np.mean(d))
    median_diff = float(np.median(d))
    reject = bool(float(p) < float(alpha))

    if median_diff > 0:
        winner = model_a_name
    elif median_diff < 0:
        winner = model_b_name
    else:
        winner = "tie"

    return WilcoxonSignedRankReport(
        model_a=model_a_name,
        model_b=model_b_name,
        metric=metric,
        n_pairs=n_used,
        alpha=float(alpha),
        alternative=str(alternative),
        statistic=float(stat),
        p_value=float(p),
        mean_diff=mean_diff,
        median_diff=median_diff,
        effect_r_rank_biserial=float(effect_r),
        split_fingerprint=split_fingerprint,
        reject_h0=reject,
        winner=winner,
    )


def _get_attr(obj: Any, name: str) -> Any:
    if hasattr(obj, name):
        return getattr(obj, name)
    raise AttributeError(f"Object has no attribute '{name}'.")


def compare_two_cv_results(
    res_a: Any,
    res_b: Any,
    *,
    metric: str = "roc_auc",
    model_a_name: str = "model_A",
    model_b_name: str = "model_B",
    alpha: float = 0.05,
    alternative: str = "two-sided",
) -> WilcoxonSignedRankReport:
    """
    Convenience wrapper to compare two result objects that expose:
      - fold_scores: Dict[str, np.ndarray]
      - split_fingerprint: str (optional but recommended)

    Works with your Section 7 CVEvaluationResult and also with other section results
    that store fold_scores similarly (as long as they keep identical folds).
    """
    fold_scores_a: Dict[str, np.ndarray] = _get_attr(res_a, "fold_scores")
    fold_scores_b: Dict[str, np.ndarray] = _get_attr(res_b, "fold_scores")

    if metric not in fold_scores_a or metric not in fold_scores_b:
        raise KeyError(f"Metric '{metric}' not found in both fold_scores dicts.")

    fp_a = getattr(res_a, "split_fingerprint", None)
    fp_b = getattr(res_b, "split_fingerprint", None)
    if fp_a is not None and fp_b is not None and fp_a != fp_b:
        raise ValueError(
            "split_fingerprint mismatch: folds are NOT identical. "
            "You must reuse the SAME cv_splits for Wilcoxon.\n"
            f"fp_a={fp_a}\nfp_b={fp_b}"
        )

    return wilcoxon_compare_fold_scores(
        fold_scores_a[metric],
        fold_scores_b[metric],
        model_a_name=model_a_name,
        model_b_name=model_b_name,
        metric=metric,
        alpha=alpha,
        alternative=alternative,
        split_fingerprint=fp_a if fp_a is not None else fp_b,
    )


def pick_best_two(
    summary_table,
    *,
    id_col: str,
    metric_col: str = "roc_auc_mean",
    higher_is_better: bool = True,
) -> Tuple[str, str]:
    """
    Pick top-2 rows from a summary_table by metric_col.
    Example:
      pick_best_two(out9.summary_table, id_col="model", metric_col="roc_auc_mean")
    """
    df = summary_table.copy()
    if id_col not in df.columns or metric_col not in df.columns:
        raise KeyError(f"summary_table must have columns '{id_col}' and '{metric_col}'.")

    df[metric_col] = df[metric_col].astype(float)
    df = df.sort_values(metric_col, ascending=not higher_is_better).reset_index(drop=True)
    if len(df) < 2:
        raise ValueError("Need at least 2 rows to pick best two models.")
    return str(df.loc[0, id_col]), str(df.loc[1, id_col])


Writing section15_statistical_comparison.py


### The following code executes Section 15 using the dataset with the real data.

In [ ]:
import os, sys, importlib.util
import pandas as pd

import section5_preprocessing_pipeline as s5
import section7_validation_protocol as s7
import section9_expanded_modeling as s9
import section15_statistical_comparison as s15

def import_module_from_path(module_name: str, file_path: str):
    spec = importlib.util.spec_from_file_location(module_name, file_path)
    if spec is None or spec.loader is None:
        raise ImportError(f"Could not load spec for {module_name} from {file_path}")

    mod = importlib.util.module_from_spec(spec)
    sys.modules.pop(module_name, None)
    sys.modules[module_name] = mod
    spec.loader.exec_module(mod)
    return mod

PROJECT_DIR = os.getcwd()
project_code = import_module_from_path("project_code", os.path.join(PROJECT_DIR, "code.py"))

DATA_PATH = os.path.join(PROJECT_DIR, "database", "diabetic_data.csv")
df_raw = pd.read_csv(DATA_PATH, low_memory=False)

df = project_code.engineer_targets(
    df_raw,
    source_col="readmitted",
    binary_col="readmitted_30d",
    multiclass_col="readmitted_3class",
    add_multiclass=True,
    drop_source=False,
    unknown_policy="error",
)

cfg = s5.DiabetesPreprocessConfig(
    onehot_sparse=True,
    rare_min_count=50,
    scale_numeric=True,
    scaler="standard",
)

y = df["readmitted_30d"].astype(int).to_numpy()
splits = s7.make_stratified_cv_splits(y, n_splits=10, shuffle=True, random_state=42)

out9 = s9.run_section9_expanded_suite(
    df,
    target_col="readmitted_30d",
    preprocess_config_base=cfg,
    cv_splits=splits,     
    n_splits=10,
    cv_random_state=42,
    model_random_state=42,
    return_oof=False,
    svd_components=100,
    fast_mode=True,       
)

display(out9.summary_table.sort_values("roc_auc_mean", ascending=False))

best_a, best_b = s15.pick_best_two(out9.summary_table, id_col="model", metric_col="roc_auc_mean")
res = s15.compare_two_cv_results(
    out9.results_by_model[best_a],
    out9.results_by_model[best_b],
    metric="roc_auc",
    model_a_name=best_a,
    model_b_name=best_b,
    alpha=0.05,
    alternative="two-sided",
)

print(res.to_text())


C:\Users\Cris-SX\AppData\Roaming\Python\Python313\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: [0]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


## SECTION 16: Error Analysis

In [ ]:
%%writefile section16_error_analysis.py
from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.metrics import confusion_matrix, roc_auc_score

MISSING_TOKENS = ("", "?", "NA", "N/A", "NULL", "NAN", "UNKNOWN", "UNKNOWN/INVALID")

def is_missing(v: object, missing_tokens: Sequence[str] = MISSING_TOKENS) -> bool:
    if v is None:
        return True
    try:
        if pd.isna(v):
            return True
    except Exception:
        pass
    if isinstance(v, str):
        s = v.strip()
        if s == "":
            return True
        return s.upper() in {t.upper() for t in missing_tokens}
    return False


def _safe_index(X, idx: np.ndarray):
    if isinstance(X, (pd.DataFrame, pd.Series)):
        return X.iloc[idx]
    return X[idx]


def get_score_vector(estimator, X, *, positive_label: int = 1) -> np.ndarray:
    """
    Continuous scores for AUC and confidence:
    - predict_proba[:, pos] if available
    - else decision_function
    """
    if hasattr(estimator, "predict_proba"):
        proba = np.asarray(estimator.predict_proba(X))
        if proba.ndim != 2 or proba.shape[1] < 2:
            raise ValueError("predict_proba must return [n_samples, 2+] for binary tasks.")
        classes = getattr(estimator, "classes_", None)
        if classes is None:
            pos_idx = 1
        else:
            classes = np.asarray(classes)
            pos_idx = int(np.where(classes == positive_label)[0][0]) if positive_label in set(classes.tolist()) else 1
        return proba[:, pos_idx].astype(float)

    if hasattr(estimator, "decision_function"):
        s = estimator.decision_function(X)
        return np.asarray(s, dtype=float).ravel()

    raise ValueError("Estimator must provide predict_proba or decision_function.")


def confidence_from_scores(scores: np.ndarray) -> np.ndarray:
    """
    Model-agnostic confidence proxy:
    - if scores look like probabilities in [0,1], use |p-0.5|
    - else use |z| after standardization
    """
    s = np.asarray(scores, dtype=float).ravel()
    if np.all(s >= 0.0) and np.all(s <= 1.0):
        return np.abs(s - 0.5)
    mu = float(np.nanmean(s))
    sd = float(np.nanstd(s) + 1e-12)
    z = (s - mu) / sd
    return np.abs(z)


def row_missingness_counts(df: pd.DataFrame, feature_cols: Sequence[str]) -> Tuple[np.ndarray, np.ndarray]:
    miss = df[feature_cols].applymap(is_missing)
    miss_count = miss.sum(axis=1).to_numpy(dtype=int)
    miss_rate = (miss_count / float(len(feature_cols))).astype(float)
    return miss_count, miss_rate


@dataclass(frozen=True)
class OOFResult:
    oof_pred: np.ndarray
    oof_score: np.ndarray
    auc: float
    cm: np.ndarray


def compute_oof_for_model(
    pipeline,
    X,
    y: np.ndarray,
    *,
    cv_splits: Sequence[Tuple[np.ndarray, np.ndarray]],
    positive_label: int = 1,
) -> OOFResult:
    """
    Manual CV (no leakage): fit on TRAIN fold, predict on TEST fold.
    Produces OOF predictions and OOF scores for every sample.
    """
    y = np.asarray(y, dtype=int).ravel()
    n = len(y)

    oof_pred = np.full(n, -1, dtype=int)
    oof_score = np.full(n, np.nan, dtype=float)

    for tr, te in cv_splits:
        tr = np.asarray(tr, dtype=np.int64)
        te = np.asarray(te, dtype=np.int64)

        model = clone(pipeline)
        model.fit(_safe_index(X, tr), y[tr])

        yhat = np.asarray(model.predict(_safe_index(X, te)), dtype=int)
        scr = get_score_vector(model, _safe_index(X, te), positive_label=positive_label)

        oof_pred[te] = yhat
        oof_score[te] = np.asarray(scr, dtype=float)

    if np.any(oof_pred < 0) or np.any(np.isnan(oof_score)):
        raise RuntimeError("OOF predictions/scores incomplete. Check your cv_splits coverage.")

    auc = float(roc_auc_score(y, oof_score))
    cm = confusion_matrix(y, oof_pred, labels=[0, 1])
    return OOFResult(oof_pred=oof_pred, oof_score=oof_score, auc=auc, cm=cm)


def select_systematic_failures(
    y: np.ndarray,
    *,
    best_model: str,
    oof_by_model: Dict[str, OOFResult],
    candidate_models: Sequence[str],
    k: int = 3,
    min_models: int = 2,
) -> List[int]:
    """
    "Systematic failure" = misclassified by best model AND misclassified by >= min_models-1 other models
    among candidate_models (top models).
    """
    y = np.asarray(y, dtype=int).ravel()

    best = oof_by_model[best_model]
    mis_best = (best.oof_pred != y)

    mis_counts = np.zeros_like(y, dtype=int)
    for m in candidate_models:
        r = oof_by_model[m]
        mis_counts += (r.oof_pred != y).astype(int)

    keep = np.where(mis_best & (mis_counts >= int(min_models)))[0]

    if keep.size == 0:
        keep = np.where(mis_best)[0]

    conf = confidence_from_scores(best.oof_score)
    order = keep[np.argsort(-conf[keep])]  

    return order[: int(k)].astype(int).tolist()


def rare_category_flags(df: pd.DataFrame, idx: int, cols: Sequence[str], min_count: int = 50) -> Dict[str, Tuple[Any, int]]:
    out: Dict[str, Tuple[Any, int]] = {}
    for c in cols:
        if c not in df.columns:
            continue
        v = df.loc[idx, c]
        vv = "__MISSING__" if is_missing(v) else str(v)
        counts = df[c].astype("object").map(lambda x: "__MISSING__" if is_missing(x) else str(x)).value_counts()
        out[c] = (vv, int(counts.get(vv, 0)))
    return {c: (v, n) for c, (v, n) in out.items() if n < min_count}


def hypothesize_case(
    df: pd.DataFrame,
    idx: int,
    *,
    feature_cols: Sequence[str],
    rare_cols: Sequence[str],
    rare_min_count: int = 50,
) -> List[str]:
    reasons: List[str] = []

    miss_count, miss_rate = row_missingness_counts(df.loc[[idx]], feature_cols)
    miss_count = int(miss_count[0])
    miss_rate = float(miss_rate[0])
    if miss_rate >= 0.30:
        reasons.append(f"High missingness: {miss_count}/{len(feature_cols)} features missing ({miss_rate:.1%}).")

    rares = rare_category_flags(df, idx, rare_cols, min_count=rare_min_count)
    if rares:
        msg = "Rare categories: " + ", ".join([f"{c}={v} (count={n})" for c, (v, n) in rares.items()])
        reasons.append(msg)

    if "a1c_high" in df.columns and "insulin_active" in df.columns:
        a1c_high = int(pd.to_numeric(df.loc[idx, "a1c_high"], errors="coerce") or 0)
        ins_act = int(pd.to_numeric(df.loc[idx, "insulin_active"], errors="coerce") or 0)
        if a1c_high == 1 and ins_act == 0:
            reasons.append("Possible lab/therapy mismatch: A1C high but insulin not active (could confuse model).")

    if "total_visits" in df.columns and "time_in_hospital" in df.columns:
        tv = float(pd.to_numeric(df.loc[idx, "total_visits"], errors="coerce") or 0.0)
        tih = float(pd.to_numeric(df.loc[idx, "time_in_hospital"], errors="coerce") or 0.0)
        if tv >= 5 and tih <= 3:
            reasons.append("Utilization pattern: high prior visits but short stay (non-typical combination).")

    if "n_changed_diabetes_meds" in df.columns:
        nch = float(pd.to_numeric(df.loc[idx, "n_changed_diabetes_meds"], errors="coerce") or 0.0)
        if nch >= 2:
            reasons.append("Multiple medication changes (complex management) may increase uncertainty.")

    return reasons

def plot_confusion_matrix(cm: np.ndarray, *, title: str = "Confusion matrix (OOF)") -> None:
    import matplotlib.pyplot as plt

    cm = np.asarray(cm, dtype=int)
    fig = plt.figure()
    ax = fig.add_subplot(111)
    im = ax.imshow(cm)
    ax.set_title(title)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(["0", "1"])
    ax.set_yticklabels(["0", "1"])

    for (i, j), v in np.ndenumerate(cm):
        ax.text(j, i, str(v), ha="center", va="center")

    fig.colorbar(im, ax=ax)
    plt.show()


def plot_fp_fn_by_fold(
    y: np.ndarray,
    oof_pred: np.ndarray,
    cv_splits: Sequence[Tuple[np.ndarray, np.ndarray]],
    *,
    title: str = "False Positives / False Negatives by fold",
) -> None:
    import matplotlib.pyplot as plt

    y = np.asarray(y, dtype=int).ravel()
    oof_pred = np.asarray(oof_pred, dtype=int).ravel()

    fps, fns = [], []
    for _, te in cv_splits:
        te = np.asarray(te, dtype=np.int64)
        yt = y[te]
        yp = oof_pred[te]
        fp = int(((yt == 0) & (yp == 1)).sum())
        fn = int(((yt == 1) & (yp == 0)).sum())
        fps.append(fp)
        fns.append(fn)

    x = np.arange(len(fps))
    fig = plt.figure()
    ax = fig.add_subplot(111)
    ax.bar(x, fps, label="FP")
    ax.bar(x, fns, bottom=fps, label="FN")
    ax.set_title(title)
    ax.set_xlabel("Fold")
    ax.set_ylabel("Count")
    ax.legend()
    plt.show()


def plot_missingness_distributions(
    df: pd.DataFrame,
    y: np.ndarray,
    oof_pred: np.ndarray,
    *,
    feature_cols: Sequence[str],
    title: str = "Row missingness: correct vs misclassified",
) -> None:
    import matplotlib.pyplot as plt

    y = np.asarray(y, dtype=int).ravel()
    oof_pred = np.asarray(oof_pred, dtype=int).ravel()
    mis = (oof_pred != y)

    miss_count, miss_rate = row_missingness_counts(df, feature_cols)

    fig = plt.figure()
    ax = fig.add_subplot(111)
    ax.hist(miss_rate[~mis], bins=30, alpha=0.7, label="Correct")
    ax.hist(miss_rate[mis], bins=30, alpha=0.7, label="Misclassified")
    ax.set_title(title)
    ax.set_xlabel("Missing-rate per row")
    ax.set_ylabel("Frequency")
    ax.legend()
    plt.show()


def plot_case_profile(
    df: pd.DataFrame,
    idx: int,
    *,
    numeric_cols: Sequence[str],
    title: Optional[str] = None,
) -> None:
    """
    Compare case numeric features vs dataset medians (overall + by class if available).
    """
    import matplotlib.pyplot as plt

    present = [c for c in numeric_cols if c in df.columns]
    if not present:
        return

    vals = []
    meds = []
    for c in present:
        v = pd.to_numeric(df.loc[idx, c], errors="coerce")
        vals.append(float(v) if not pd.isna(v) else np.nan)
        meds.append(float(pd.to_numeric(df[c], errors="coerce").median(skipna=True)))

    x = np.arange(len(present))
    fig = plt.figure(figsize=(max(8, len(present) * 0.6), 4))
    ax = fig.add_subplot(111)
    ax.bar(x - 0.2, vals, width=0.4, label="Case")
    ax.bar(x + 0.2, meds, width=0.4, label="Dataset median")
    ax.set_xticks(x)
    ax.set_xticklabels(present, rotation=45, ha="right")
    ax.set_ylabel("Value")
    ax.set_title(title or f"Case profile idx={idx}")
    ax.legend()
    plt.tight_layout()
    plt.show()


Writing section16_error_analysis.py


### The following code executes Section 2 using the dataset with the real data.

In [ ]:
import os, sys, importlib.util
import numpy as np
import pandas as pd

import section5_preprocessing_pipeline as s5
import section7_validation_protocol as s7
import section9_expanded_modeling as s9
import section11_feature_engineering as s11
import section16_error_analysis as s16

def import_module_from_path(module_name: str, file_path: str):
    spec = importlib.util.spec_from_file_location(module_name, file_path)
    if spec is None or spec.loader is None:
        raise ImportError(f"Could not load spec for {module_name} from {file_path}")
    mod = importlib.util.module_from_spec(spec)
    sys.modules.pop(module_name, None)
    sys.modules[module_name] = mod
    spec.loader.exec_module(mod)
    return mod

PROJECT_DIR = os.getcwd()
project_code = import_module_from_path("project_code", os.path.join(PROJECT_DIR, "code.py"))

DATA_PATH = os.path.join(PROJECT_DIR, "database", "diabetic_data.csv")
df_raw = pd.read_csv(DATA_PATH, low_memory=False)
print("Loaded df_raw:", df_raw.shape)

df = project_code.engineer_targets(
    df_raw,
    source_col="readmitted",
    binary_col="readmitted_30d",
    multiclass_col="readmitted_3class",
    add_multiclass=True,
    drop_source=False,
    unknown_policy="error",
)

fe_out = s11.apply_feature_engineering(df)
df_model = fe_out.df
print("Section 11 added columns:", len(fe_out.added_columns))

cfg = s5.DiabetesPreprocessConfig(
    onehot_sparse=True,
    rare_min_count=50,
    scale_numeric=True,
    scaler="standard",
)

y = df_model["readmitted_30d"].astype(int).to_numpy()
splits = s7.make_stratified_cv_splits(y, n_splits=10, shuffle=True, random_state=42)
print("CV fingerprint:", s7.cv_splits_fingerprint(splits))

suite = s9.run_section9_expanded_suite(
    df_model,
    target_col="readmitted_30d",
    preprocess_config_base=cfg,
    cv_splits=splits,
    n_splits=10,
    return_oof=False,
    fast_mode=False,          
    svd_components=100,
)

summary = suite.summary_table.sort_values("roc_auc_mean", ascending=False).reset_index(drop=True)
display(summary)

best_model_name = str(summary.loc[0, "model"])
top_models = summary.head(3)["model"].astype(str).tolist()
print("Best model:", best_model_name)
print("Top-3 models:", top_models)

specs = s9.build_section9_model_specs(
    df_model,
    preprocess_config_base=cfg,
    model_random_state=42,
    svd_components=100,
    fast_mode=False,
)
pipelines = {sp.name: sp.pipeline for sp in specs}

drop_cols = [c for c in cfg.target_cols if c in df_model.columns]
X = df_model.drop(columns=drop_cols, errors="ignore")

oof_by_model = {}
for m in top_models:
    print(f"OOF for model: {m}")
    oof_by_model[m] = s16.compute_oof_for_model(
        pipelines[m],
        X,
        y,
        cv_splits=splits,
        positive_label=1,
    )
    print("  OOF AUC:", oof_by_model[m].auc)

best_oof = oof_by_model[best_model_name]

s16.plot_confusion_matrix(best_oof.cm, title=f"OOF confusion matrix — best model = {best_model_name}")
s16.plot_fp_fn_by_fold(y, best_oof.oof_pred, splits, title=f"FP/FN by fold — best model = {best_model_name}")

feature_cols = [c for c in df_model.columns if c not in cfg.target_cols]
s16.plot_missingness_distributions(
    df_model,
    y,
    best_oof.oof_pred,
    feature_cols=feature_cols,
    title="Row missingness distribution: correct vs misclassified (best model)",
)

case_indices = s16.select_systematic_failures(
    y,
    best_model=best_model_name,
    oof_by_model=oof_by_model,
    candidate_models=top_models,
    k=3,
    min_models=2,
)
print("Selected systematic failure indices:", case_indices)

KEY_NUMERIC = [
    "age_mid", "time_in_hospital", "num_medications", "number_diagnoses",
    "total_visits", "stay_x_meds", "stay_x_diagnoses",
    "n_active_diabetes_meds", "n_changed_diabetes_meds"
]
RARE_COLS = [
    "race", "gender", "payer_code", "medical_specialty",
    "diag_1_group", "diag_2_group", "diag_3_group",
    "admission_type", "discharge_disposition", "admission_source"
]

for idx in case_indices:
    true_y = int(y[idx])
    pred_y = int(best_oof.oof_pred[idx])
    score = float(best_oof.oof_score[idx])
    err_type = "FN" if (true_y == 1 and pred_y == 0) else ("FP" if (true_y == 0 and pred_y == 1) else "OK")

    print("\n" + "="*90)
    print(f"CASE idx={idx} | true={true_y} pred={pred_y} | error={err_type} | score={score:.4f}")
    print("- Key fields snapshot:")
    cols_preview = [
        "encounter_id", "patient_nbr", "readmitted", "readmitted_30d",
        "age", "age_mid", "elderly",
        "diag_1", "diag_1_group", "n_diabetes_diags", "primary_diag_is_diabetes",
        "A1Cresult", "a1c_measured", "a1c_high",
        "max_glu_serum", "glu_measured", "glu_high",
        "insulin", "insulin_active", "insulin_changed",
        "time_in_hospital", "num_medications", "polypharmacy",
        "number_outpatient", "number_emergency", "number_inpatient",
        "total_visits", "any_emergency", "any_inpatient",
        "n_active_diabetes_meds", "n_changed_diabetes_meds",
    ]
    cols_preview = [c for c in cols_preview if c in df_model.columns]
    display(df_model.loc[[idx], cols_preview])

    reasons = s16.hypothesize_case(
        df_model,
        idx,
        feature_cols=feature_cols,
        rare_cols=RARE_COLS,
        rare_min_count=50,
    )
    print("- Hypotheses for failure:")
    if reasons:
        for r in reasons:
            print("  •", r)
    else:
        print("  • (No strong heuristic flags; likely boundary/noisy case or label ambiguity.)")

    s16.plot_case_profile(
        df_model,
        idx,
        numeric_cols=KEY_NUMERIC,
        title=f"Case numeric profile vs dataset median (idx={idx}, error={err_type})"
    )


Loaded df_raw: (101766, 50)
Section 11 added columns: 37
CV fingerprint: f5968b2fab085e7f54fd828ac2d60e09701e9a727701c3d6a99a4e5b64150ef5


C:\Users\Cris-SX\AppData\Roaming\Python\Python313\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: [0]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


## SECTION 17: XAI / Explainability

In [ ]:
%%writefile section17_xai_explainability.py
from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier

import section5_preprocessing_pipeline as s5


class SparseToCSC(BaseEstimator, TransformerMixin):
    """Convert sparse matrices to CSC (tree estimators often prefer CSC)."""

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        try:
            from scipy import sparse
        except Exception as e:
            raise RuntimeError("scipy is required. Install with: pip install scipy") from e
        return X.tocsc() if sparse.issparse(X) else X


def _safe_series_mode(s: pd.Series):
    s2 = s.dropna()
    if s2.empty:
        return np.nan
    return s2.mode(dropna=True).iloc[0]


def _predict_score_pos1(pipe: Pipeline, X: pd.DataFrame) -> np.ndarray:
    """Return continuous score for positive class (1): proba if available else decision_function."""
    if hasattr(pipe, "predict_proba"):
        proba = np.asarray(pipe.predict_proba(X))
        if proba.ndim != 2 or proba.shape[1] < 2:
            raise ValueError("predict_proba must return [n_samples, 2+].")
        return proba[:, 1].astype(float)

    if hasattr(pipe, "decision_function"):
        return np.asarray(pipe.decision_function(X), dtype=float).ravel()

    raise ValueError("Pipeline must expose predict_proba or decision_function.")


def _get_preprocessed_feature_names(pre: Pipeline) -> List[str]:
    """
    Robust feature-name extraction for your Section 5 preprocessor:
      - num: num__{col}
      - ord: ord__{col}
      - cat: onehot feature names if available, else cat__{col} (fallback)
    """
    if "features" not in pre.named_steps:
        raise ValueError("Expected preprocessor Pipeline to contain a 'features' ColumnTransformer step.")

    ct = pre.named_steps["features"]

    names: List[str] = []
    for name, trans, cols in ct.transformers_:
        cols = list(cols) if isinstance(cols, (list, tuple)) else [cols]

        if name == "num":
            names.extend([f"num__{c}" for c in cols])
            continue

        if name == "ord":
            names.extend([f"ord__{c}" for c in cols])
            continue

        if name == "cat":
            try:
                onehot = ct.named_transformers_["cat"].named_steps["onehot"]
                oh_names = list(onehot.get_feature_names_out(cols))
                names.extend([f"cat__{n}" for n in oh_names])
            except Exception:
                names.extend([f"cat__{c}" for c in cols])
            continue

        names.extend([f"{name}__{c}" for c in cols])

    return names


def _plot_top_barh(df_imp: pd.DataFrame, *, title: str, top_n: int = 25):
    if df_imp is None or df_imp.empty:
        print(f"[plot skipped] {title}: empty importance table.")
        return
    d = df_imp.head(top_n).iloc[::-1]  
    plt.figure(figsize=(10, max(4, int(0.22 * len(d) + 2))))
    plt.barh(d["feature"].astype(str), d["importance"].astype(float))
    plt.title(title)
    plt.xlabel("Importance")
    plt.tight_layout()
    plt.show()


def _global_importance_logreg(pipe: Pipeline, feature_names: List[str], top_n: int = 30) -> pd.DataFrame:
    pre = pipe.named_steps["pre"]
    clf: LogisticRegression = pipe.named_steps["clf"]

    coef = np.asarray(clf.coef_).ravel()
    if len(coef) != len(feature_names):
        raise ValueError(f"coef dim {len(coef)} != feature_names dim {len(feature_names)}")

    imp = np.abs(coef)
    out = pd.DataFrame(
        {"feature": feature_names, "importance": imp, "coef": coef}
    ).sort_values("importance", ascending=False).reset_index(drop=True)
    return out.head(top_n)


def _global_importance_tree(pipe: Pipeline, feature_names: List[str], top_n: int = 30) -> pd.DataFrame:
    clf = pipe.named_steps["clf"]
    if not hasattr(clf, "feature_importances_"):
        raise ValueError("Tree-based estimator has no feature_importances_.")
    imp = np.asarray(clf.feature_importances_, dtype=float).ravel()
    if len(imp) != len(feature_names):
        raise ValueError(f"feature_importances dim {len(imp)} != feature_names dim {len(feature_names)}")

    out = pd.DataFrame({"feature": feature_names, "importance": imp})
    out = out.sort_values("importance", ascending=False).reset_index(drop=True)
    return out.head(top_n)


def _compute_elderly_from_age_bucket(age_series: pd.Series) -> pd.Series:
    """
    If you don't have 'elderly' yet, infer from 'age' buckets:
      elderly = midpoint(age) >= 65
    """
    def midpoint(v) -> float:
        if v is None:
            return np.nan
        s = str(v).strip()
        if not s.startswith("[") or "-" not in s:
            return np.nan
        try:
            inside = s.strip("[]()")
            a, b = inside.split("-")
            return (float(a) + float(b)) / 2.0
        except Exception:
            return np.nan

    mid = age_series.map(midpoint)
    return (mid.fillna(-1) >= 65).astype(int)


def _perm_importance_raw_cols(
    pipe: Pipeline,
    X_test: pd.DataFrame,
    y_test: np.ndarray,
    *,
    n_repeats: int = 5,
    random_state: int = 42,
    top_n: int = 25,
) -> pd.DataFrame:
    if len(np.unique(y_test)) < 2:
        return pd.DataFrame(columns=["feature", "importance_mean", "importance_std"])

    r = permutation_importance(
        pipe,
        X_test,
        y_test,
        scoring="roc_auc",
        n_repeats=int(n_repeats),
        random_state=int(random_state),
        n_jobs=None,
    )
    out = pd.DataFrame(
        {
            "feature": list(X_test.columns),
            "importance_mean": r.importances_mean.astype(float),
            "importance_std": r.importances_std.astype(float),
        }
    ).sort_values("importance_mean", ascending=False).reset_index(drop=True)
    return out.head(top_n)


def _pick_top_confident_errors(
    pipe: Pipeline,
    X_test: pd.DataFrame,
    y_test: np.ndarray,
    *,
    k: int = 3,
) -> pd.DataFrame:
    scores = _predict_score_pos1(pipe, X_test)
    pred = (scores >= 0.5).astype(int)
    wrong = pred != y_test

    if wrong.sum() == 0:
        return pd.DataFrame(columns=["row_ix", "y_true", "y_pred", "score", "confidence"])

    confidence = np.abs(scores - 0.5)
    idx = np.where(wrong)[0]
    idx_sorted = idx[np.argsort(confidence[idx])[::-1]]
    idx_pick = idx_sorted[: int(k)]

    return pd.DataFrame(
        {
            "row_ix": idx_pick,
            "y_true": y_test[idx_pick],
            "y_pred": pred[idx_pick],
            "score": scores[idx_pick],
            "confidence": confidence[idx_pick],
        }
    ).sort_values("confidence", ascending=False).reset_index(drop=True)


def _local_explain_logreg_exact(
    pipe: Pipeline,
    X_row: pd.DataFrame,
    feature_names: List[str],
    *,
    top_n: int = 15,
) -> pd.DataFrame:
    """
    Exact local explanation for LogisticRegression on the *preprocessed* feature space:
      contribution_i = coef_i * x_i
    """
    pre = pipe.named_steps["pre"]
    clf: LogisticRegression = pipe.named_steps["clf"]

    Xz = pre.transform(X_row)
    try:
        x = np.asarray(Xz.toarray()).ravel()
    except Exception:
        x = np.asarray(Xz).ravel()

    coef = np.asarray(clf.coef_).ravel()
    if x.shape[0] != coef.shape[0] or len(feature_names) != coef.shape[0]:
        raise ValueError("Dimension mismatch in local_explain_logreg_exact.")

    contrib = x * coef
    out = pd.DataFrame(
        {"feature": feature_names, "x_value": x, "coef": coef, "contribution": contrib, "abs_contribution": np.abs(contrib)}
    ).sort_values("abs_contribution", ascending=False).reset_index(drop=True)

    return out.head(top_n)[["feature", "x_value", "coef", "contribution"]]


def _baseline_values_from_train(X_train: pd.DataFrame) -> Dict[str, object]:
    """
    Baseline replacement values for counterfactual-style local explanation:
      - numeric -> median
      - categorical/object -> mode
    """
    base: Dict[str, object] = {}
    for c in X_train.columns:
        s = X_train[c]
        if pd.api.types.is_numeric_dtype(s):
            base[c] = float(pd.to_numeric(s, errors="coerce").median())
        else:
            base[c] = _safe_series_mode(s.astype("object"))
    return base


def _local_counterfactual_deltas(
    pipe: Pipeline,
    X_row: pd.DataFrame,
    *,
    baseline_values: Dict[str, object],
    candidate_features: List[str],
    top_n: int = 15,
) -> pd.DataFrame:
    """
    Model-agnostic local explanation in RAW feature space:
      For each raw feature f:
        - replace f with baseline(train) value
        - delta = score(original) - score(replaced)
      Positive delta => feature pushes score towards class 1.
    """
    if len(X_row) != 1:
        raise ValueError("X_row must be a single-row DataFrame.")

    base_score = float(_predict_score_pos1(pipe, X_row)[0])

    rows = []
    for f in candidate_features:
        if f not in X_row.columns:
            continue
        x_cf = X_row.copy()
        x_cf.iloc[0, x_cf.columns.get_loc(f)] = baseline_values.get(f, np.nan)
        s_cf = float(_predict_score_pos1(pipe, x_cf)[0])
        rows.append(
            {
                "feature": f,
                "orig_value": X_row.iloc[0][f],
                "baseline_value": baseline_values.get(f, np.nan),
                "delta_score": base_score - s_cf,
                "abs_delta": abs(base_score - s_cf),
            }
        )

    out = pd.DataFrame(rows).sort_values("abs_delta", ascending=False).reset_index(drop=True)
    return out.head(top_n)[["feature", "orig_value", "baseline_value", "delta_score"]]


@dataclass(frozen=True)
class Section17XAIResult:
    split_random_state: int
    test_size: float
    models: Dict[str, Pipeline]
    feature_names: List[str]
    global_importances_encoded: Dict[str, pd.DataFrame]   
    global_importance_raw_perm: pd.DataFrame              
    global_importance_raw_perm_elderly: Optional[pd.DataFrame]
    global_importance_raw_perm_nonelderly: Optional[pd.DataFrame]
    hard_errors: pd.DataFrame                             
    local_logreg_exact: Dict[int, pd.DataFrame]           
    local_best_counterfactual: Dict[int, pd.DataFrame]    


def run_section17_xai(
    df: pd.DataFrame,
    *,
    target_col: str = "readmitted_30d",
    preprocess_config: Optional[s5.DiabetesPreprocessConfig] = None,
    test_size: float = 0.20,
    random_state: int = 42,
    top_n_global: int = 25,
    n_repeats_perm: int = 5,
    top_n_local: int = 15,
) -> Section17XAIResult:
    """
    End-to-end XAI runner:
      - Fit 3 model families: Logistic Regression, Decision Tree, Random Forest
      - Global (encoded-space): coef / feature_importances_
      - Global (raw-space): permutation importance on raw columns (model-agnostic)
      - Subgroup (elderly vs non-elderly): raw permutation importance
      - Local: 3 high-confidence wrong predictions on the best model (RF):
          * LR exact contributions (encoded)
          * RF model-agnostic counterfactual deltas (raw)
    """
    if preprocess_config is None:
        preprocess_config = s5.DiabetesPreprocessConfig(onehot_sparse=True, rare_min_count=50, scale_numeric=True, scaler="standard")

    if target_col not in df.columns:
        raise KeyError(f"Target col '{target_col}' not found in df.")

    y = df[target_col].astype(int).to_numpy()

    drop_cols = [c for c in preprocess_config.target_cols if c in df.columns]
    if target_col not in drop_cols:
        drop_cols.append(target_col)

    X = df.drop(columns=drop_cols, errors="ignore")
    if X.shape[1] == 0:
        raise ValueError("No features left after dropping targets.")

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=float(test_size),
        stratify=y,
        random_state=int(random_state),
    )

    pre = s5.build_preprocessor(df, preprocess_config)

    lr = LogisticRegression(
        solver="saga",
        penalty="l2",
        max_iter=4000,
        n_jobs=-1,
        random_state=int(random_state),
    )
    dt = DecisionTreeClassifier(random_state=int(random_state))
    rf = RandomForestClassifier(
        n_estimators=300,
        n_jobs=-1,
        random_state=int(random_state),
    )

    pipes: Dict[str, Pipeline] = {
        "logistic_regression": Pipeline([("pre", pre), ("clf", lr)]),
        "decision_tree": Pipeline([("pre", pre), ("to_csc", SparseToCSC()), ("clf", dt)]),
        "random_forest": Pipeline([("pre", pre), ("to_csc", SparseToCSC()), ("clf", rf)]),
    }

    for _, p in pipes.items():
        p.fit(X_train, y_train)

    feature_names = _get_preprocessed_feature_names(pipes["logistic_regression"].named_steps["pre"])

    global_encoded: Dict[str, pd.DataFrame] = {
        "logistic_regression": _global_importance_logreg(pipes["logistic_regression"], feature_names, top_n=top_n_global),
        "decision_tree": _global_importance_tree(pipes["decision_tree"], feature_names, top_n=top_n_global),
        "random_forest": _global_importance_tree(pipes["random_forest"], feature_names, top_n=top_n_global),
    }

    perm_raw = _perm_importance_raw_cols(
        pipes["random_forest"], X_test, y_test,
        n_repeats=int(n_repeats_perm),
        random_state=int(random_state),
        top_n=top_n_global,
    )

    if "elderly" in X_test.columns:
        elderly = X_test["elderly"].astype(int)
    elif "age_mid" in X_test.columns:
        elderly = (pd.to_numeric(X_test["age_mid"], errors="coerce").fillna(-1) >= 65).astype(int)
    elif "age" in X_test.columns:
        elderly = _compute_elderly_from_age_bucket(X_test["age"].astype("object"))
    else:
        elderly = None

    perm_elderly = None
    perm_nonelderly = None
    if elderly is not None:
        mask_e = elderly.to_numpy().astype(int) == 1
        mask_ne = ~mask_e

        if mask_e.sum() >= 50 and len(np.unique(y_test[mask_e])) >= 2:
            perm_elderly = _perm_importance_raw_cols(
                pipes["random_forest"], X_test.loc[mask_e], y_test[mask_e],
                n_repeats=int(n_repeats_perm),
                random_state=int(random_state),
                top_n=top_n_global,
            )
        if mask_ne.sum() >= 50 and len(np.unique(y_test[mask_ne])) >= 2:
            perm_nonelderly = _perm_importance_raw_cols(
                pipes["random_forest"], X_test.loc[mask_ne], y_test[mask_ne],
                n_repeats=int(n_repeats_perm),
                random_state=int(random_state),
                top_n=top_n_global,
            )

    hard = _pick_top_confident_errors(pipes["random_forest"], X_test, y_test, k=3)

    baseline_values = _baseline_values_from_train(X_train)

    candidate_raw = perm_raw["feature"].astype(str).tolist() if perm_raw is not None and not perm_raw.empty else list(X_test.columns)[:30]

    local_lr: Dict[int, pd.DataFrame] = {}
    local_best: Dict[int, pd.DataFrame] = {}

    for _, row in hard.iterrows():
        ix = int(row["row_ix"])
        X_one = X_test.iloc[[ix]]

        try:
            local_lr[ix] = _local_explain_logreg_exact(
                pipes["logistic_regression"], X_one, feature_names, top_n=int(top_n_local)
            )
        except Exception as e:
            local_lr[ix] = pd.DataFrame({"error": [str(e)]})

        try:
            local_best[ix] = _local_counterfactual_deltas(
                pipes["random_forest"],
                X_one,
                baseline_values=baseline_values,
                candidate_features=candidate_raw,
                top_n=int(top_n_local),
            )
        except Exception as e:
            local_best[ix] = pd.DataFrame({"error": [str(e)]})

    return Section17XAIResult(
        split_random_state=int(random_state),
        test_size=float(test_size),
        models=pipes,
        feature_names=feature_names,
        global_importances_encoded=global_encoded,
        global_importance_raw_perm=perm_raw,
        global_importance_raw_perm_elderly=perm_elderly,
        global_importance_raw_perm_nonelderly=perm_nonelderly,
        hard_errors=hard,
        local_logreg_exact=local_lr,
        local_best_counterfactual=local_best,
    )


def plot_section17_global(result: Section17XAIResult, *, top_n: int = 25):
    for model_name, df_imp in result.global_importances_encoded.items():
        _plot_top_barh(df_imp, title=f"Global importance (encoded) — {model_name}", top_n=top_n)

    if result.global_importance_raw_perm is not None and not result.global_importance_raw_perm.empty:
        d = result.global_importance_raw_perm.head(top_n).iloc[::-1]
        plt.figure(figsize=(10, max(4, int(0.22 * len(d) + 2))))
        plt.barh(d["feature"].astype(str), d["importance_mean"].astype(float))
        plt.title("Global importance (RAW permutation) — random_forest")
        plt.xlabel("Mean decrease in ROC-AUC (higher = more important)")
        plt.tight_layout()
        plt.show()


def plot_section17_local_tables(result: Section17XAIResult):
    print("\n=== Hard errors picked on best model (random_forest) ===")
    print(result.hard_errors)

    for _, row in result.hard_errors.iterrows():
        ix = int(row["row_ix"])
        print(f"\n--- Case row_ix={ix} | y_true={int(row['y_true'])} | y_pred={int(row['y_pred'])} | score={float(row['score']):.4f} ---")

        print("\n[Local — Logistic Regression exact contributions (encoded space)]")
        print(result.local_logreg_exact.get(ix, pd.DataFrame()).head(20))

        print("\n[Local — Random Forest counterfactual deltas (raw space)]")
        print(result.local_best_counterfactual.get(ix, pd.DataFrame()).head(20))


Overwriting section17_xai_explainability.py


### The following code executes Section 17 using the dataset with the real data.

In [ ]:
import os, sys, importlib.util
import pandas as pd

import section5_preprocessing_pipeline as s5
import section17_xai_explainability as s17

import section11_feature_engineering as s11

def import_module_from_path(module_name: str, file_path: str):
    spec = importlib.util.spec_from_file_location(module_name, file_path)
    if spec is None or spec.loader is None:
        raise ImportError(f"Could not load spec for {module_name} from {file_path}")
    mod = importlib.util.module_from_spec(spec)
    sys.modules.pop(module_name, None)
    sys.modules[module_name] = mod
    spec.loader.exec_module(mod)
    return mod

PROJECT_DIR = os.getcwd()
CODE_PATH = os.path.join(PROJECT_DIR, "code.py")
DATA_PATH = os.path.join(PROJECT_DIR, "database", "diabetic_data.csv")

project_code = import_module_from_path("project_code", CODE_PATH)

df_raw = pd.read_csv(DATA_PATH, low_memory=False)

df = project_code.engineer_targets(
    df_raw,
    source_col="readmitted",
    binary_col="readmitted_30d",
    multiclass_col="readmitted_3class",
    add_multiclass=True,
    drop_source=False,
    unknown_policy="error",
)

fe_out = s11.apply_feature_engineering(df)
df = fe_out.df
print("Section 11 added columns:", len(fe_out.added_columns))

cfg = s5.DiabetesPreprocessConfig(
    onehot_sparse=True,
    rare_min_count=50,
    scale_numeric=True,
    scaler="standard",
)

res17 = s17.run_section17_xai(
    df,
    target_col="readmitted_30d",
    preprocess_config=cfg,
    test_size=0.20,
    random_state=42,
    top_n_global=25,
    n_repeats_perm=5,
    top_n_local=15,
)

s17.plot_section17_global(res17, top_n=25)

display(res17.global_importances_encoded["logistic_regression"])
display(res17.global_importances_encoded["decision_tree"])
display(res17.global_importances_encoded["random_forest"])

display(res17.global_importance_raw_perm)

if res17.global_importance_raw_perm_elderly is not None:
    display(res17.global_importance_raw_perm_elderly)

if res17.global_importance_raw_perm_nonelderly is not None:
    display(res17.global_importance_raw_perm_nonelderly)

s17.plot_section17_local_tables(res17)


# SECTION 18

## IMPORTANT

For greater clarity, I include again within Section 18 all the code cells from each run of every section on the dataset with the real data


## IMPORTANT

Run the three following code cells to ensure that everything works correctly.


In [ ]:
import section11_feature_engineering as s11

fe_out = s11.apply_feature_engineering(df)
df_fe = fe_out.df
print("Added columns (Section 11):", fe_out.added_columns)

DF = df_fe

In [ ]:
import section5_preprocessing_pipeline as s5;

cfg = s5.DiabetesPreprocessConfig(onehot_sparse=True, rare_min_count=50)

In [ ]:
import section7_validation_protocol as s7
import numpy as np

# IMPORTANT: use the SAME dataframe you pass to the models (DF vs df)
y = DF["readmitted_30d"].astype(int).to_numpy()

splits = s7.make_stratified_cv_splits(
    y,
    n_splits=10,
    shuffle=True,
    random_state=42,
)

print("CV fingerprint:", s7.cv_splits_fingerprint(splits))

### START OF THE RESULTS OF SECTION 18

In [ ]:
import section4_eda_imbalance as s4

# Class imbalance (binary)
imb = s4.compute_class_imbalance(DF["readmitted_30d"].astype(int))
imb.as_dict()


{'N': 101766,
 'n_pos': 11357,
 'n_neg': 90409,
 'pos_rate': 0.11159915885462728,
 'neg_rate': 0.8884008411453728}

In [ ]:
# Missingness by feature
mf = s4.missingness_by_feature(DF)
mf.head(25)


,missing_rate,missing_count
weight,0.968585,98569
max_glu_serum,0.947468,96420
A1Cresult,0.832773,84748
medical_specialty,0.490822,49949
payer_code,0.395574,40256
race,0.022336,2273
diag_3,0.013983,1423
diag_2,0.003518,358
diag_1,0.000206,21
gender,0.000029,3


In [ ]:
# Missingness by row
mr = s4.missingness_by_row(DF, high_missing_threshold=0.30)
mr.as_dict()


{'missing_per_row_summary': {'count': 101766.0,
  'mean': 3.675294302615805,
  'std': 0.8034736709543345,
  'min': 1.0,
  '50%': 4.0,
  '75%': 4.0,
  '90%': 5.0,
  '95%': 5.0,
  '99%': 5.0,
  'max': 7.0},
 'high_missing_fraction': 0.0}

In [ ]:
# Top-K category distributions
from IPython.display import display

for col in ["medical_specialty", "diag_1", "diag_2", "diag_3", "payer_code"]:
    if col in DF.columns:
        top, nunique, coverage = s4.topk_table(DF[col], k=15)
        print(f"\n=== {col} | nunique={nunique} | top15 coverage={coverage:.3f} ===")
        display(top)



=== medical_specialty | nunique=73 | top15 coverage=0.955 ===


,count,rate
medical_specialty,,
__MISSING__,49949,0.490822
InternalMedicine,14635,0.143810
Emergency/Trauma,7565,0.074337
Family/GeneralPractice,7440,0.073109
Cardiology,5352,0.052591
Surgery-General,3099,0.030452
Nephrology,1613,0.015850
Orthopedics,1400,0.013757
Orthopedics-Reconstructive,1233,0.012116



=== diag_1 | nunique=717 | top15 coverage=0.443 ===


,count,rate
diag_1,,
428,6862,0.067429
414,6581,0.064668
786,4016,0.039463
410,3614,0.035513
486,3508,0.034471
427,2766,0.027180
491,2275,0.022355
715,2151,0.021137
682,2042,0.020066



=== diag_2 | nunique=749 | top15 coverage=0.511 ===


,count,rate
diag_2,,
276,6752,0.066348
428,6662,0.065464
250,6071,0.059656
427,5036,0.049486
401,3736,0.036712
496,3305,0.032476
599,3288,0.032309
403,2823,0.027740
414,2650,0.026040



=== diag_3 | nunique=790 | top15 coverage=0.527 ===


,count,rate
diag_3,,
250,11555,0.113545
401,8289,0.081452
276,5175,0.050852
428,4577,0.044976
427,3955,0.038864
414,3664,0.036004
496,2605,0.025598
403,2357,0.023161
585,1992,0.019574



=== payer_code | nunique=18 | top15 coverage=0.999 ===


,count,rate
payer_code,,
__MISSING__,40256,0.395574
MC,32439,0.318761
HM,6274,0.061651
SP,5007,0.049201
BC,4655,0.045742
MD,3532,0.034707
CP,2533,0.024890
UN,2448,0.024055
CM,1937,0.019034


In [ ]:
# Dummy baseline (single split, illustrative)
excluded = {"readmitted", "readmitted_30d", "readmitted_3class"}
feature_cols = [c for c in DF.columns if c not in excluded]

dummy_res = s4.run_dummy_most_frequent_baseline(
    DF,
    feature_cols=feature_cols,
    target_col="readmitted_30d",
    test_size=0.2,
    random_state=42,
)
dummy_res.as_dict()


{'confusion_terms': {'tn': 18083, 'fp': 0, 'fn': 2271, 'tp': 0},
 'metrics': {'accuracy': 0.8884248796305394,
  'precision': 0.0,
  'recall': 0.0,
  'f1': 0.0}}

In [2]:
import section5_preprocessing_pipeline as s5

cfg = s5.DiabetesPreprocessConfig(
    onehot_sparse=True,
    rare_min_count=50,
)

groups = s5.infer_feature_groups(DF, cfg)
groups


NameError: name 'DF' is not defined

In [12]:
pre = s5.build_preprocessor(DF, cfg)

X = DF.drop(columns=[c for c in ["readmitted", "readmitted_30d", "readmitted_3class"] if c in DF.columns])
y = DF["readmitted_30d"].astype(int)

pre.fit(X, y)
Xt = pre.transform(X)

type(Xt), Xt.shape


C:\Users\Cris-SX\AppData\Roaming\Python\Python313\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: [0]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
C:\Users\Cris-SX\AppData\Roaming\Python\Python313\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: [0]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


(scipy.sparse._csr.csr_matrix, (101766, 893))

In [27]:
import section6_baseline_system as s6

baseline_table = s6.run_dummy_baselines(
    DF,
    target_col="readmitted_30d",
    strategies=("most_frequent", "stratified"),
    n_splits=10,
    cv_random_state=42,
    dummy_random_state=42,
    preprocess_config=cfg,
)
baseline_table


C:\Users\Cris-SX\AppData\Roaming\Python\Python313\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: [0]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
C:\Users\Cris-SX\AppData\Roaming\Python\Python313\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: [0]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
C:\Users\Cris-SX\AppData\Roaming\Python\Python313\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: [0]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
C:\Users\Cris-SX\AppData\Roaming\Python\Python313\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: [0]. At least one non-missing value is needed for imputation with strategy='median

,strategy,n_splits,roc_auc_mean,roc_auc_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,accuracy_mean,accuracy_std
0,most_frequent,10.0,0.500000,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.888401,0.000043
1,stratified,10.0,0.502107,0.004082,0.115465,0.00749,0.111825,0.007248,0.113616,0.007367,0.805279,0.001621


In [ ]:
import section8_core_modeling as s8

out8 = s8.run_section8_core_models(
    DF,
    target_col="readmitted_30d",
    preprocess_config=cfg,
    n_splits=10,
    cv_random_state=42,
    model_random_state=42,
    cv_splits=splits,   
    return_oof=True,
)

out8.split_fingerprint, out8.summary_table


C:\Users\Cris-SX\AppData\Roaming\Python\Python313\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: [0]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
C:\Users\Cris-SX\AppData\Roaming\Python\Python313\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: [0]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
C:\Users\Cris-SX\AppData\Roaming\Python\Python313\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: [0]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
C:\Users\Cris-SX\AppData\Roaming\Python\Python313\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: [0]. At least one non-missing value is needed for imputation with strategy='median

('f5968b2fab085e7f54fd828ac2d60e09701e9a727701c3d6a99a4e5b64150ef5',
                  model  roc_auc_mean  roc_auc_std  precision_mean  \
 0  logistic_regression      0.672770     0.007291        0.474961   
 1        decision_tree      0.530097     0.004659        0.165010   
 
    precision_std  recall_mean  recall_std   f1_mean    f1_std  accuracy_mean  \
 0       0.073138     0.018138    0.003939  0.034916  0.007437       0.888194   
 1       0.008206     0.165273    0.009973  0.165085  0.008508       0.813494   
 
    accuracy_std  balanced_accuracy_mean  balanced_accuracy_std  
 0      0.000570                0.507814               0.001961  
 1      0.003644                0.530097               0.004659  )

In [ ]:
import section9_expanded_modeling as s9

out9 = s9.run_section9_expanded_suite(
    DF,
    target_col="readmitted_30d",
    preprocess_config_base=cfg,
    n_splits=10,
    cv_random_state=42,
    model_random_state=42,
    cv_splits=splits,       
    svd_components=100,
    fast_mode=False,
    return_oof=False,
)

out9.summary_table


C:\Users\Cris-SX\AppData\Roaming\Python\Python313\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: [0]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
C:\Users\Cris-SX\AppData\Roaming\Python\Python313\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: [0]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
C:\Users\Cris-SX\AppData\Roaming\Python\Python313\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: [0]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
C:\Users\Cris-SX\AppData\Roaming\Python\Python313\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: [0]. At least one non-missing value is needed for imputation with strategy='median

,model,track,roc_auc_mean,roc_auc_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,accuracy_mean,accuracy_std,balanced_accuracy_mean,balanced_accuracy_std
0,decision_tree,tree,0.529926,0.004516,0.164672,0.007852,0.165008,0.009853,0.164787,0.008308,0.813395,0.003452,0.529926,0.004516
1,naive_bayes,nb_nonneg,0.582438,0.005839,0.169275,0.005490,0.400635,0.013403,0.237973,0.007455,0.713647,0.005343,0.576801,0.007037
2,svm_linear,scaled,0.671673,0.007332,0.557210,0.109483,0.010742,0.003315,0.021060,0.006431,0.888666,0.000414,0.504846,0.001641
3,mlp_svd,svd_dense,0.648488,0.007132,0.467381,0.100954,0.008805,0.004274,0.017233,0.008257,0.888273,0.000440,0.503778,0.001957
4,random_forest,tree,0.659262,0.010657,0.603870,0.131649,0.007220,0.002303,0.014264,0.004522,0.888686,0.000370,0.503317,0.001192
5,gradient_boosting_svd,svd_dense,0.648273,0.006763,0.465183,0.145415,0.009422,0.003645,0.018456,0.007102,0.888273,0.000625,0.504047,0.001875


In [30]:
import section10_imbalance_handling as s10

table10, map10 = s10.run_section10_imbalance_experiments(
    DF,
    target_col="readmitted_30d",
    preprocess_config=cfg,
    cv_splits=splits,     # MUST reuse
    models=("logistic_regression", "decision_tree", "random_forest"),
    methods=("none", "class_weight_balanced", "random_over", "random_under", "threshold_f1"),
    sampling_ratio=1.0,
    random_state=42,
    fast_mode=False,
)

table10


C:\Users\Cris-SX\AppData\Roaming\Python\Python313\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: [0]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
C:\Users\Cris-SX\AppData\Roaming\Python\Python313\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: [0]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
C:\Users\Cris-SX\AppData\Roaming\Python\Python313\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: [0]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
C:\Users\Cris-SX\AppData\Roaming\Python\Python313\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: [0]. At least one non-missing value is needed for imputation with strategy='median

,model,method,sampling_ratio,n_splits,split_fingerprint,threshold_mean,threshold_std,roc_auc_mean,precision_mean,recall_mean,f1_mean,accuracy_mean,balanced_accuracy_mean,roc_auc_std,precision_std,recall_std,f1_std,accuracy_std,balanced_accuracy_std
0,decision_tree,class_weight_balanced,1.0,10,f5968b2fab085e7f54fd828ac2d60e09701e9a727701c3...,NaN,NaN,0.535578,0.161150,0.205511,0.180632,0.791974,0.535578,0.007474,0.010133,0.014192,0.011726,0.003172,0.007474
1,decision_tree,none,1.0,10,f5968b2fab085e7f54fd828ac2d60e09701e9a727701c3...,NaN,NaN,0.530097,0.165010,0.165273,0.165085,0.813494,0.530097,0.004659,0.008206,0.009973,0.008508,0.003644,0.004659
2,decision_tree,random_over,1.0,10,f5968b2fab085e7f54fd828ac2d60e09701e9a727701c3...,NaN,NaN,0.537203,0.165225,0.203751,0.182459,0.796229,0.537203,0.005017,0.007334,0.009104,0.007901,0.003126,0.005017
3,decision_tree,random_under,1.0,10,f5968b2fab085e7f54fd828ac2d60e09701e9a727701c3...,NaN,NaN,0.554555,0.135522,0.549089,0.217385,0.558801,0.554555,0.004952,0.002140,0.011179,0.003514,0.004002,0.004952
4,decision_tree,threshold_f1,1.0,10,f5968b2fab085e7f54fd828ac2d60e09701e9a727701c3...,1.000000,0.000000,0.530097,0.165010,0.165273,0.165085,0.813494,0.530097,0.004659,0.008206,0.009973,0.008508,0.003644,0.004659
5,logistic_regression,class_weight_balanced,1.0,10,f5968b2fab085e7f54fd828ac2d60e09701e9a727701c3...,NaN,NaN,0.671702,0.177972,0.582988,0.272691,0.652939,0.622357,0.006854,0.002806,0.009601,0.004163,0.003576,0.004955
6,logistic_regression,none,1.0,10,f5968b2fab085e7f54fd828ac2d60e09701e9a727701c3...,NaN,NaN,0.672770,0.474961,0.018138,0.034916,0.888194,0.507814,0.007291,0.073138,0.003939,0.007437,0.000570,0.001961
7,logistic_regression,random_over,1.0,10,f5968b2fab085e7f54fd828ac2d60e09701e9a727701c3...,NaN,NaN,0.669734,0.177256,0.582373,0.271781,0.651721,0.621402,0.007078,0.002614,0.010537,0.003985,0.003661,0.004880
8,logistic_regression,random_under,1.0,10,f5968b2fab085e7f54fd828ac2d60e09701e9a727701c3...,NaN,NaN,0.667868,0.172285,0.596286,0.267325,0.635202,0.618188,0.007568,0.003911,0.012418,0.005811,0.005086,0.007201
9,logistic_regression,threshold_f1,1.0,10,f5968b2fab085e7f54fd828ac2d60e09701e9a727701c3...,0.146323,0.004901,0.672770,0.210194,0.411557,0.277893,0.761531,0.608526,0.007291,0.008635,0.031411,0.010848,0.011347,0.010251


In [31]:
import section12_feature_selection as s12

out12 = s12.run_section12_feature_selection(
    DF,
    target_col="readmitted_30d",
    preprocess_config_base=cfg,
    n_splits=10,
    cv_random_state=42,
    model_random_state=42,
    cv_splits=splits,
    build_interpretation=False,
)

out12.summary_table


C:\Users\Cris-SX\AppData\Roaming\Python\Python313\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: [0]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
C:\Users\Cris-SX\AppData\Roaming\Python\Python313\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
C:\Users\Cris-SX\AppData\Roaming\Python\Python313\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: [0]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
C:\Users\Cris-SX\AppData\Roaming\Python\Python313\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: [0]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
C:\Users\Cris-SX\AppData\Roaming\Pyt

,method,variant,split_fingerprint,roc_auc_mean,roc_auc_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,accuracy_mean,accuracy_std,balanced_accuracy_mean,balanced_accuracy_std
0,embedded_l1,all_features,f5968b2fab085e7f54fd828ac2d60e09701e9a727701c3...,0.672770,0.007291,0.474961,0.073138,0.018138,0.003939,0.034916,0.007437,0.888194,0.000570,0.507814,0.001961
1,embedded_l1,selected,f5968b2fab085e7f54fd828ac2d60e09701e9a727701c3...,0.672770,0.007291,0.474961,0.073138,0.018138,0.003939,0.034916,0.007437,0.888194,0.000570,0.507814,0.001961
2,filter_chi2,all_features,f5968b2fab085e7f54fd828ac2d60e09701e9a727701c3...,0.640067,0.006713,0.496052,0.090457,0.010566,0.003804,0.020675,0.007360,0.888440,0.000420,0.504642,0.001877
3,filter_chi2,selected,f5968b2fab085e7f54fd828ac2d60e09701e9a727701c3...,0.634841,0.006722,0.501588,0.093763,0.010743,0.004020,0.021014,0.007777,0.888470,0.000435,0.504735,0.001975


In [ ]:
import section13_dimensionality_reduction as s13

out13 = s13.run_section13_dimensionality_reduction(
    DF,
    target_col="readmitted_30d",
    preprocess_config=cfg,
    svd_components=(50, 100, 200),
    n_splits=10,
    cv_random_state=42,
    model_random_state=42,
    cv_splits=splits,   
)

out13.summary_table


C:\Users\Cris-SX\AppData\Roaming\Python\Python313\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: [0]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
C:\Users\Cris-SX\AppData\Roaming\Python\Python313\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: [0]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
C:\Users\Cris-SX\AppData\Roaming\Python\Python313\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: [0]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
C:\Users\Cris-SX\AppData\Roaming\Python\Python313\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: [0]. At least one non-missing value is needed for imputation with strategy='median

In [ ]:
import section14_hyperparameter_tuning as s14

manual_table = s14.manual_sweep_logreg_C(
    DF,
    target_col="readmitted_30d",
    preprocess_config=cfg,
    cv_splits=splits,   
    C_values=(0.001, 0.01, 0.1, 1.0, 3.0, 10.0),
    class_weight=None,
    random_state=42,
)
manual_table


C:\Users\Cris-SX\AppData\Roaming\Python\Python313\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: [0]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
C:\Users\Cris-SX\AppData\Roaming\Python\Python313\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: [0]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
C:\Users\Cris-SX\AppData\Roaming\Python\Python313\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: [0]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
C:\Users\Cris-SX\AppData\Roaming\Python\Python313\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: [0]. At least one non-missing value is needed for imputation with strategy='median

KeyboardInterrupt: 

In [ ]:
gs = s14.gridsearch_logreg(
    DF,
    target_col="readmitted_30d",
    preprocess_config=cfg,
    inner_folds=5,
    random_state=42,
    n_jobs=-1,
    verbose=1,
)
print("Best params:", gs.best_params_)
print("Best inner-CV ROC-AUC:", gs.best_score_)


Fitting 5 folds for each of 20 candidates, totalling 100 fits


KeyboardInterrupt: 

In [ ]:
nested_lr = s14.nested_cv_tuning(
    DF,
    model_name="logreg",
    target_col="readmitted_30d",
    preprocess_config=cfg,
    outer_splits=splits,
    inner_folds=3,
    search="random",
    n_iter=20,
    random_state=42,
    n_jobs=-1,
    verbose=0,
)

nested_lr.fold_metrics, nested_lr.mean_scores, nested_lr.std_scores, nested_lr.best_params_counts


KeyboardInterrupt: 

In [ ]:
import os, sys, importlib.util
import pandas as pd

import section5_preprocessing_pipeline as s5
import section7_validation_protocol as s7
import section9_expanded_modeling as s9

def import_module_from_path(module_name: str, file_path: str):
    spec = importlib.util.spec_from_file_location(module_name, file_path)
    if spec is None or spec.loader is None:
        raise ImportError(f"Could not load spec for {module_name} from {file_path}")
    mod = importlib.util.module_from_spec(spec)
    sys.modules.pop(module_name, None)
    sys.modules[module_name] = mod
    spec.loader.exec_module(mod)
    return mod

PROJECT_DIR = os.getcwd()
project_code = import_module_from_path("project_code", os.path.join(PROJECT_DIR, "code.py"))

CSV_PATH = os.path.join(PROJECT_DIR, "database", "diabetic_data.csv")
df_raw = pd.read_csv(CSV_PATH, low_memory=False)

df = project_code.engineer_targets(
    df_raw,
    source_col="readmitted",
    binary_col="readmitted_30d",
    multiclass_col="readmitted_3class",
    add_multiclass=True,
    drop_source=False,
    unknown_policy="error",
)

cfg = s5.DiabetesPreprocessConfig(
    onehot_sparse=True,
    rare_min_count=50,
)

y = df["readmitted_30d"].astype(int).to_numpy()
splits = s7.make_stratified_cv_splits(y, n_splits=10, shuffle=True, random_state=42)
print("CV fingerprint:", s7.cv_splits_fingerprint(splits))

out9 = s9.run_section9_expanded_suite(
    df,
    target_col="readmitted_30d",
    preprocess_config_base=cfg,
    n_splits=10,
    cv_random_state=42,
    model_random_state=42,
    cv_splits=splits,
    svd_components=100,
    fast_mode=False,
    return_oof=False,
)

out9.summary_table


CV fingerprint: f5968b2fab085e7f54fd828ac2d60e09701e9a727701c3d6a99a4e5b64150ef5


C:\Users\Cris-SX\AppData\Roaming\Python\Python313\site-packages\sklearn\impute\_base.py:653: UserWarning: Skipping features without any observed values: [0]. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


KeyboardInterrupt: 

In [ ]:
import numpy as np
from scipy.stats import wilcoxon

def wilcoxon_report(a, b, name_a, name_b, metric="roc_auc", alpha=0.05, alternative="two-sided"):
    a = np.asarray(a, dtype=float).ravel()
    b = np.asarray(b, dtype=float).ravel()
    if a.shape != b.shape:
        raise ValueError(f"Paired vectors must have same shape. Got {a.shape} vs {b.shape}")

    diffs = a - b
    nonzero = diffs[diffs != 0]

    if nonzero.size == 0:
        stat, p = 0.0, 1.0
    else:
        stat, p = wilcoxon(diffs, alternative=alternative)

    reject = (p < alpha)
    return {
        "model_a": name_a,
        "model_b": name_b,
        "metric": metric,
        "alpha": alpha,
        "alternative": alternative,
        "statistic": float(stat),
        "p_value": float(p),
        "n_folds": int(len(diffs)),
        "n_nonzero": int(len(nonzero)),
        "mean_diff": float(np.mean(diffs)),
        "median_diff": float(np.median(diffs)),
        "reject_H0": bool(reject),
    }

tab = out9.summary_table.sort_values("roc_auc_mean", ascending=False).reset_index(drop=True)
m1, m2 = tab.loc[0, "model"], tab.loc[1, "model"]

rep_auc = wilcoxon_report(
    out9.results_by_model[m1].fold_scores["roc_auc"],
    out9.results_by_model[m2].fold_scores["roc_auc"],
    m1, m2, metric="roc_auc", alpha=0.05
)

rep_f1 = wilcoxon_report(
    out9.results_by_model[m1].fold_scores["f1"],
    out9.results_by_model[m2].fold_scores["f1"],
    m1, m2, metric="f1", alpha=0.05
)

rep_auc, rep_f1


NameError: name 'out9' is not defined

In [ ]:
import pandas as pd
import numpy as np

leader9 = out9.summary_table.copy()
leader9["source"] = "section9"


leader10 = table10.copy()
leader10["source"] = "section10"
leader10 = leader10.rename(columns={"model":"model_name"})
leader10["model"] = leader10["model_name"] + "__" + leader10["method"].astype(str)

keep_cols = [c for c in leader9.columns if c in leader10.columns] + [c for c in leader10.columns if c in leader9.columns]
keep_cols = sorted(set(keep_cols))

def align_cols(df, cols):
    out = df.copy()
    for c in cols:
        if c not in out.columns:
            out[c] = np.nan
    return out[cols]

cols_union = sorted(set(leader9.columns).union(set(leader10.columns)))
dec_table = pd.concat([align_cols(leader9, cols_union), align_cols(leader10, cols_union)], axis=0, ignore_index=True)

for c in ["roc_auc_mean","f1_mean","recall_mean"]:
    if c not in dec_table.columns:
        dec_table[c] = np.nan

dec_ranked = dec_table.sort_values(
    by=["roc_auc_mean","f1_mean","recall_mean"],
    ascending=[False, False, False],
).reset_index(drop=True)

dec_ranked.head(15)


NameError: name 'out9' is not defined

In [ ]:
import pandas as pd
import numpy as np

leader9 = out9.summary_table.copy()
leader9["source"] = "section9"

leader10 = table10.copy()
leader10["source"] = "section10"
leader10 = leader10.rename(columns={"model":"model_name"})
leader10["model"] = leader10["model_name"] + "__" + leader10["method"].astype(str)

keep_cols = [c for c in leader9.columns if c in leader10.columns] + [c for c in leader10.columns if c in leader9.columns]
keep_cols = sorted(set(keep_cols))

def align_cols(df, cols):
    out = df.copy()
    for c in cols:
        if c not in out.columns:
            out[c] = np.nan
    return out[cols]

cols_union = sorted(set(leader9.columns).union(set(leader10.columns)))
dec_table = pd.concat([align_cols(leader9, cols_union), align_cols(leader10, cols_union)], axis=0, ignore_index=True)

for c in ["roc_auc_mean","f1_mean","recall_mean"]:
    if c not in dec_table.columns:
        dec_table[c] = np.nan

dec_ranked = dec_table.sort_values(
    by=["roc_auc_mean","f1_mean","recall_mean"],
    ascending=[False, False, False],
).reset_index(drop=True)

dec_ranked.head(15)


In [ ]:
best = dec_ranked.loc[0, "model"]
runner = dec_ranked.loc[1, "model"]
best, runner


In [ ]:
from scipy.stats import wilcoxon
import numpy as np

def get_fold_scores(model_key, metric):
    if "__" in model_key:
        base, method = model_key.split("__", 1)
        return map10[(base, method)].fold_scores[metric]
    return out9.results_by_model[model_key].fold_scores[metric]

def wilcoxon_pair(mA, mB, metric, alpha=0.05):
    a = np.asarray(get_fold_scores(mA, metric), dtype=float)
    b = np.asarray(get_fold_scores(mB, metric), dtype=float)
    stat, p = wilcoxon(a - b, alternative="two-sided")
    return {"metric": metric, "statistic": float(stat), "p_value": float(p), "reject_H0(p<alpha)": bool(p < alpha)}

wilcoxon_pair(best, runner, "roc_auc"), wilcoxon_pair(best, runner, "f1"), wilcoxon_pair(best, runner, "recall")
